# Yoruba ASR on Kaggle: XLS-R + hybrid data augmentation + domain adaptation (V1–V4)

**Project:** Enhancing ASR Accuracy for Low-Resource Nigerian Tonal Languages Through Hybrid Data Augmentation and Domain Adaptation: A Case Study of Yoruba.

This notebook runs the **whole training and evaluation workflow** of the project, **one stage per run**. The `yoruba_asr` package source is embedded below, so you only attach your **data**. You do not need a code dataset.

### Kaggle settings (right-hand panel → *Session options*)
1. **Accelerator:** GPU T4 (×1 or ×2; only one GPU is used) or GPU P100.
2. **Internet:** **On**. This needs a phone-verified account. It is used to download XLS-R, MMS-TTS and pip packages.
3. **Persistence:** *Files only* (optional, for interactive sessions).

### Datasets to attach (*Add Input*)
| Dataset | Required? | How |
|---|---|---|
| Common Voice Scripted Speech **27.0 Yoruba** (`.tar.gz` from the Mozilla Data Collective) | **yes** | upload it as a **PRIVATE** Kaggle dataset; never make it public |
| Google Yoruba corpus **OpenSLR 86** (`yo_ng_female.zip`, `yo_ng_male.zip`) | recommended (V3/V4 stage B1 + CPT) | private dataset |
| **ÌròyìnSpeech** (audio + a `manifest.csv` with `path,sentence,speaker`) | recommended | private dataset; put "iroyin" in the folder name |
| MUSAN subset (noise/music/speech) | optional (V2/V4 noise) | private dataset named `musan` |
| **Output of the previous run of this notebook** | from run 2 on | *Add Input → Your Work → this notebook → previous version* |

### How to use
1. Edit **only the PARAMETERS cell**: set `STAGE` (see the session plan printed below).
2. Click **Save Version → Save & Run All (Commit)**.
3. When the run finishes, open its **Output**, then attach that output as an input to the next version and set the next `STAGE`. The last cell tells you what to run next.

Full click-by-click guide: `docs/KAGGLE_GUIDE.md` in the project repository.


In [ ]:
# ============================ PARAMETERS (the only cell you edit) ============================
STAGE   = "prepare"   # one of: "smoke", "prepare", "V1", "augment", "tune_mask", "V2",
                      #         "cpt", "V3", "V4", "evaluate", "ablation", "analyse", "all_light"
                      # stages can be chained with "+", e.g. "augment+tune_mask"
RESUME  = True        # resume from checkpoints found in /kaggle/working or attached previous outputs
ABLATION_ONLY = None  # e.g. "no_tts" to run one ablation per session
PUSH_TO_HUB = False   # needs Kaggle Secret HF_TOKEN
HUB_REPO_PREFIX = "your-hf-username/yoruba-xlsr"   # used only if PUSH_TO_HUB
CPT_MAX_HOURS = 11.0
EXTRA_OVERRIDES = {}  # e.g. {"train.max_steps": 3000}
DRY_RUN = False       # True: print the plan for STAGE without training
PRIMARY_DATASET = "auto"   # "auto": Common Voice if attached and complete, otherwise OpenSLR 86
                           # "slr86": train/test on OpenSLR 86 only (no upload needed)
                           # "common_voice": always use Common Voice
DOWNLOAD_OPENSLR86 = True  # download OpenSLR 86 automatically (~0.9 GB, needs Internet On)


## Environment check

In [ ]:
# ---- Environment check -------------------------------------------------------------------
import os, sys, shutil, platform, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # use ONE GPU only (before importing torch)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    CODE_DIR, WORK_ROOT, INPUT_ROOT, TMP_DIR = ("/kaggle/working/yoruba_asr_code", "/kaggle/working",
                                                "/kaggle/input", "/tmp/yoruba_asr")
else:  # local acceptance test (python kaggle/build_notebook.py --test / nbconvert)
    _root = os.environ.get("YASR_LOCAL_TEST_ROOT", os.path.abspath("_nb_local"))
    CODE_DIR, WORK_ROOT, TMP_DIR = (f"{_root}/code", f"{_root}/working", f"{_root}/tmp")
    INPUT_ROOT = os.environ.get("YASR_INPUT_ROOT", f"{_root}/input")
CODE_DIR = CODE_DIR.replace("\\", "/")
for d in (CODE_DIR, WORK_ROOT, TMP_DIR):
    os.makedirs(d, exist_ok=True)

import torch
print("Python      :", platform.python_version())
print("torch       :", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU         :", p.name, f"{p.total_memory / 1024**3:.1f} GB")
else:
    print("GPU         : none (CPU only). For real training set Accelerator = GPU T4/P100.")
import importlib.metadata as _md  # do not import transformers yet: the next cell may pin its version
try:
    print("transformers:", _md.version("transformers"), "(before pinning)")
except _md.PackageNotFoundError:
    print("transformers: not installed yet")
for label, path in (("disk /kaggle/working", WORK_ROOT), ("disk /tmp", TMP_DIR)):
    print(f"{label:12s}: {shutil.disk_usage(path).free / 1024**3:.1f} GB free")
print("\nReminder: Kaggle's weekly GPU quota is about 30 h (see Settings → Quota). Each session is limited "
      "to 12 h; this notebook stops training at ~11 h and saves a checkpoint.")
if ON_KAGGLE and not torch.cuda.is_available() and STAGE not in ("smoke", "prepare", "evaluate", "analyse"):
    print("\n*** WARNING: no GPU detected. Turn on Accelerator = GPU in the session options. ***")


## Install pinned packages

In [ ]:
# ---- Install pinned extra packages (skips packages already present in the Kaggle image) -------
REQUIREMENTS = """# Extra installs on Kaggle (torch, librosa, soundfile, pandas, matplotlib... come with the image).
# The Hugging Face stack is pinned as a set to the versions the code was tested with:
# transformers 4.57 requires huggingface-hub < 1.0, so datasets/accelerate are pinned to compatible versions.
transformers==4.57.0
huggingface-hub==0.36.2
datasets==4.1.1
accelerate==1.10.1
jiwer==4.0.0
pyroomacoustics==0.10.1"""
import importlib.metadata as _md, re as _re

def _needs(req: str) -> bool:
    name, _, ver = req.partition("==")
    try:
        have = _md.version(name.strip())
    except _md.PackageNotFoundError:
        return True
    return bool(ver) and have != ver.strip()

_reqs = [r.split("#")[0].strip() for r in REQUIREMENTS.splitlines() if r.split("#")[0].strip()]
_todo = [r for r in _reqs if _needs(r)]
if ON_KAGGLE and _todo:
    print("pip install:", " ".join(_todo))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_todo], check=True)
else:
    print("nothing to install" if not _todo else f"local test: skipping pip install of {_todo}")


## Package source
The cells below write the `yoruba_asr` package (data pipeline, augmentation, domain adaptation, training, evaluation, inference), its configs and `pyproject.toml`. Collapse them, there is nothing to edit here.

In [ ]:
# ---- Package source: create folders, then the %%writefile cells below write every file -------
for _d in ['configs', 'src/yoruba_asr', 'src/yoruba_asr/adapt', 'src/yoruba_asr/augment', 'src/yoruba_asr/data', 'src/yoruba_asr/evaluate', 'src/yoruba_asr/train']:
    os.makedirs(f"{CODE_DIR}/{_d}", exist_ok=True)
print("writing the yoruba_asr package to", CODE_DIR)


In [ ]:
%%writefile {CODE_DIR}/README.md
# yoruba_asr

Package source embedded by the Kaggle notebook. See the project repository for documentation.


In [ ]:
%%writefile {CODE_DIR}/pyproject.toml
[build-system]
requires = ["setuptools>=64", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "yoruba-asr"
version = "1.0.0"
description = "Yoruba ASR with XLS-R: hybrid data augmentation and domain adaptation (V1-V4 factorial study)"
readme = "README.md"
requires-python = ">=3.10"
license = { text = "MIT" }
authors = [{ name = "Bolaji Oluseye" }]
dependencies = [
    "torch",
    "transformers>=4.45",
    "datasets>=2.19",
    "accelerate>=0.30",
    "jiwer>=3.0",
    "librosa>=0.10",
    "soundfile>=0.12",
    "pandas",
    "numpy",
    "scipy",
    "matplotlib",
    "pyyaml",
    "tqdm",
    "typer>=0.9",
]

[project.optional-dependencies]
augment = ["pyroomacoustics>=0.7"]
app = ["gradio>=4.0"]
dev = ["pytest>=7", "nbformat>=5", "nbconvert>=7", "ipykernel"]

[project.scripts]
yoruba-asr = "yoruba_asr.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-q"
filterwarnings = ["ignore::DeprecationWarning", "ignore::FutureWarning", "ignore::UserWarning"]


In [ ]:
%%writefile {CODE_DIR}/requirements-kaggle.txt
# Extra installs on Kaggle (torch, librosa, soundfile, pandas, matplotlib... come with the image).
# The Hugging Face stack is pinned as a set to the versions the code was tested with:
# transformers 4.57 requires huggingface-hub < 1.0, so datasets/accelerate are pinned to compatible versions.
transformers==4.57.0
huggingface-hub==0.36.2
datasets==4.1.1
accelerate==1.10.1
jiwer==4.0.0
pyroomacoustics==0.10.1


In [ ]:
%%writefile {CODE_DIR}/configs/ablation.yaml
# Leave-one-out ablation of V2 (Section 15.5). Every run is compared with v2_ref (same 1,500 steps).
variant: "V2"
training_set: "cv_aug"
init: "pretrained"
stage_b1: false
mask_time_prob: "tuned"
ablation:
  max_steps: 1500
  runs:
    v2_ref: {}
    no_tempo: { remove_transform: "tempo" }
    no_noise: { remove_transform: "noise" }
    no_reverb: { remove_transform: "reverb" }
    no_tts: { use_tts: false }
    no_tuned_mask: { mask_time_prob: 0.05 }
    plus_speed: { add_transform: "speed" }   # pitch-changing speed perturbation


In [ ]:
%%writefile {CODE_DIR}/configs/base.yaml
# Shared settings for every stage and every variant (V1-V4).
# Merge order: base.yaml -> variant/stage config -> --set key=value overrides.
seed: 42
run_name: "main"

paths:
  input_root: "/kaggle/input"            # searched recursively for datasets
  work_root: "/kaggle/working"           # persisted outputs (Kaggle "Output")
  work_dir: "${paths.work_root}"
  tmp_dir: "/tmp/yoruba_asr"             # extraction/cache (not persisted)
  results_dir: "${paths.work_dir}/results"
  models_dir: "${paths.work_dir}/models"
  cache_dir: "${paths.work_dir}/cache"
  shared_dir: "${paths.work_dir}/shared"
  manifests_dir: "${paths.work_dir}/manifests"
  checkpoints_dir: "${paths.work_dir}/checkpoints"
  augmented_dir: "${paths.work_dir}/augmented"
  extract_archives: true

data:
  common_voice:
    release: "27.0"
    locate_by: "train.tsv"               # find folder containing this file
    dir: null                            # optional explicit folder
  # Main (train/dev/test) corpus: "auto" = Common Voice when attached and complete, otherwise OpenSLR 86;
  # "common_voice" or "slr86" to force one. With "slr86", OpenSLR 86 is split by speaker (80/10/10).
  primary: "auto"
  slr86: { enabled: true, dir: null, force_include: false, download: true,
           split: { dev_share: 0.1, test_share: 0.1 } }
  iroyin: { enabled: true, manifest: null, dir: null, force_include: false }   # manifest CSV if not auto-detected
  bibletts: { enabled: false, dir: null }
  musan: { dir: null }                   # optional noise corpus for augmentation
  rirs: { dir: null }                    # optional real room impulse responses
  min_seconds: 0.5
  max_seconds: 15
  sampling_rate: 16000
  trim_top_db: 30
  drop_digits: true                      # utterances with digits are dropped and logged
  tone_gate_max_toneless_share: 0.20     # OOD corpus quality gate (Section 8.5)
  limit: { train: null, dev: null, test: null, other: null, ood: null, unlabelled: null }

text:
  keep_tones: true                       # training targets keep tone marks
  normal_form: "NFC"

model:
  name: "facebook/wav2vec2-xls-r-300m"
  attention_dropout: 0.0
  hidden_dropout: 0.0
  feat_proj_dropout: 0.0
  layerdrop: 0.0
  mask_time_prob: 0.05
  mask_time_length: 10
  mask_feature_prob: 0.0
  ctc_loss_reduction: "mean"
  ctc_zero_infinity: true
  freeze_feature_encoder: true
  gradient_checkpointing: true

train:
  per_device_train_batch_size: 8
  gradient_accumulation_steps: 4
  per_device_eval_batch_size: 8
  learning_rate: 3.0e-4
  adam_beta1: 0.9
  adam_beta2: 0.999
  adam_epsilon: 1.0e-8
  weight_decay: 0.0
  warmup_steps: 150
  max_steps: 3000
  eval_steps: 200
  save_steps: 200
  logging_steps: 50
  save_total_limit: 2
  fp16: true                             # only used when a CUDA GPU is present
  group_by_length: true
  metric_for_best_model: "wer"
  time_budget_hours: 11.0                # stop & checkpoint before Kaggle's 12 h limit
  dataloader_num_workers: 2
  keep_checkpoints: false                # delete checkpoints once the best model is saved

augment:
  copies_per_clip: 2
  transforms: ["tempo", "noise", "reverb", "gain"]
  transform_prob: 0.5
  tempo_factors: [0.9, 1.1]
  speed_factors: [0.9, 1.1]              # ablation only (changes pitch)
  snr_db: [10, 30]
  rt60: [0.2, 0.8]
  gain_db: [-6, 6]
  rir_bank_size: 64
  pitch_check_clips: 20
  pitch_check_max_rel_diff: 0.03

tts:
  enabled: true
  model: "facebook/mms-tts-yor"          # CC BY-NC 4.0 - research use only
  max_per_real_clip: 1.0
  listening_sample: 50

mask_tuning:
  candidates: [0.05, 0.075, 0.10]
  max_steps: 1000
  skip: false
  default_if_skipped: 0.075

adapt:
  b1_max_steps: 2000
  cleanup_b1_after_v4: true

eval:
  bootstrap_resamples: 1000
  ci: 0.95
  batch_size: 8
  pairs: [["V2", "V1"], ["V3", "V1"], ["V4", "V1"], ["V4", "V2"], ["V4", "V3"]]
  n_examples: 20


In [ ]:
%%writefile {CODE_DIR}/configs/cpt.yaml
# Continued self-supervised pretraining (wav2vec 2.0 objective) - Stage A of domain adaptation.
cpt:
  max_steps: 12000              # upper bound; the run also stops at max_total_hours
  max_total_hours: 20.0         # total GPU budget across sessions (two Kaggle sessions)
  per_device_batch_size: 4
  gradient_accumulation_steps: 8   # effective 32 crops
  learning_rate: 5.0e-5
  warmup_steps: 500
  weight_decay: 0.01
  adam_beta1: 0.9
  adam_beta2: 0.98
  adam_epsilon: 1.0e-6
  mask_time_prob: 0.65
  mask_time_length: 10
  diversity_loss_weight: 0.1
  max_gumbel_temperature: 2.0
  min_gumbel_temperature: 0.5
  gumbel_temperature_decay: 0.999995
  max_grad_norm: 1.0
  max_crop_seconds: 15
  checkpoint_every: 500
  log_every: 50
  fp16: true
  freeze_feature_encoder: true
  gradient_checkpointing: true
  perplexity_floor: 2.0
  perplexity_patience: 500
  time_budget_hours: 11.0
  num_workers: 2


In [ ]:
%%writefile {CODE_DIR}/configs/smoke.yaml
# Tiny settings for CPU tests and the Kaggle smoke run. Layer this on top of any other config.
run_name: "smoke"
paths:
  work_dir: "${paths.work_root}/smoke"
data:
  limit: { train: 32, dev: 16, test: 16, other: 16, ood: 16, unlabelled: 32 }
  slr86: { download: false }
model:
  name: "tiny-random"          # tiny randomly-initialised Wav2Vec2 (no download)
train:
  max_steps: 10
  eval_steps: 5
  save_steps: 5
  logging_steps: 5
  warmup_steps: 2
  learning_rate: 1.0e-3
  per_device_train_batch_size: 4
  gradient_accumulation_steps: 1
  per_device_eval_batch_size: 4
  fp16: false
  dataloader_num_workers: 0
  time_budget_hours: 1.0
augment:
  pitch_check_clips: 4
  rir_bank_size: 4
tts:
  model: "dummy"               # offline stand-in synthesiser; the real run uses MMS-TTS
  listening_sample: 4
mask_tuning:
  max_steps: 4
adapt:
  b1_max_steps: 6
  cleanup_b1_after_v4: false
cpt:
  max_steps: 10
  max_total_hours: 1.0
  per_device_batch_size: 2
  gradient_accumulation_steps: 1
  warmup_steps: 2
  learning_rate: 1.0e-3
  checkpoint_every: 5
  log_every: 2
  fp16: false
  num_workers: 0
  time_budget_hours: 1.0
ablation:
  max_steps: 4
eval:
  bootstrap_resamples: 200
  batch_size: 4
  n_examples: 3


In [ ]:
%%writefile {CODE_DIR}/configs/v1_baseline.yaml
# V1 - source-only baseline: no augmentation, no domain adaptation.
variant: "V1"
training_set: "cv"          # cv = Common Voice train only
init: "pretrained"          # start from facebook/wav2vec2-xls-r-300m
stage_b1: false
mask_time_prob: 0.05


In [ ]:
%%writefile {CODE_DIR}/configs/v2_augmentation.yaml
# V2 - augmentation only: CV train + 2 tone-preserving augmented copies + MMS-TTS clips.
variant: "V2"
training_set: "cv_aug"      # CV train + augmented + TTS
init: "pretrained"
stage_b1: false
mask_time_prob: "tuned"     # read from results/V2/mask_tuning.json


In [ ]:
%%writefile {CODE_DIR}/configs/v3_domain_adaptation.yaml
# V3 - domain adaptation only: CPT encoder -> B1 (OOD labelled) -> B2 (CV train).
variant: "V3"
training_set: "cv"
init: "cpt"                 # start from models/cpt_encoder
stage_b1: true
mask_time_prob: 0.05


In [ ]:
%%writefile {CODE_DIR}/configs/v4_hybrid.yaml
# V4 - hybrid: CPT encoder -> B1 (OOD labelled, shared with V3) -> B2 on the V2 training set.
variant: "V4"
training_set: "cv_aug"
init: "cpt"
stage_b1: true
mask_time_prob: "tuned"


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/__init__.py
"""Yoruba ASR: XLS-R fine-tuning with hybrid data augmentation and domain adaptation.

Final-year project: "Enhancing ASR Accuracy for Low-Resource Nigerian Tonal Languages
Through Hybrid Data Augmentation and Domain Adaptation: A Case Study of Yoruba".
"""

__version__ = "1.0.0"
VARIANTS = ("V1", "V2", "V3", "V4")


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/__main__.py
from .cli import main

main()


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/adapt/__init__.py
"""Domain adaptation: continued self-supervised pretraining (CPT) and staged fine-tuning."""


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/adapt/cpt.py
"""Continued self-supervised pretraining of XLS-R on unlabelled Yoruba audio (wav2vec 2.0 objective).

Modelled on Hugging Face's ``run_wav2vec2_pretraining_no_trainer.py``: contrastive loss + diversity
loss, Gumbel temperature decay, AdamW with linear warm-up/decay, FP16 via ``accelerate``.
Checkpoints (model, optimiser, scheduler, scaler, RNG, step, temperature) every N steps; exact resume.
"""
from __future__ import annotations

import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from .. import modeling as _modeling  # noqa: F401  (installs the gradient-checkpointing warning filter)
from ..audio import SR, safe_load_trimmed
from ..utils import RunInfo, get_logger, load_json, now_iso, save_json, seed_everything

log = get_logger(__name__)
QUANTIZER_KEYS = ("quantizer.", "project_q", "project_hid")


class CPTError(RuntimeError):
    pass


class UnlabelledDataset(torch.utils.data.Dataset):
    """Unlabelled audio, random-cropped to ``max_crop`` seconds (crop seeded by seed/epoch/index)."""

    def __init__(self, manifest: pd.DataFrame, feature_extractor, max_crop_s: float, seed: int, top_db: float = 30):
        self.paths = manifest["path"].astype(str).tolist()
        self.durations = manifest["duration"].astype(float).tolist()
        self.fe = feature_extractor
        self.max_len = int(max_crop_s * SR)
        self.seed = seed
        self.top_db = top_db
        self.epoch = 0

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int) -> dict:
        x = safe_load_trimmed(self.paths[i], SR, self.top_db)
        if x is None or len(x) < SR // 2:
            x = np.zeros(SR, dtype=np.float32) if x is None else np.pad(x, (0, SR // 2 - len(x)))
        if len(x) > self.max_len:
            rng = np.random.default_rng([self.seed, self.epoch, i])
            start = int(rng.integers(0, len(x) - self.max_len + 1))
            x = x[start: start + self.max_len]
        iv = self.fe(x, sampling_rate=SR).input_values[0].astype(np.float32)
        return {"input_values": iv}


def epoch_batches(durations: list[float], batch_size: int, seed: int, epoch: int) -> list[list[int]]:
    """Shuffle, then sort within chunks of 50 batches by duration (less padding), then shuffle batches."""
    rng = np.random.default_rng([seed, epoch, 4242])
    perm = rng.permutation(len(durations))
    chunk = batch_size * 50
    batches = []
    for s in range(0, len(perm), chunk):
        part = sorted(perm[s: s + chunk], key=lambda k: -durations[k])
        batches += [list(map(int, part[j: j + batch_size])) for j in range(0, len(part), batch_size)]
    order = rng.permutation(len(batches))
    return [batches[k] for k in order]


class PretrainingCollator:
    def __init__(self, model, feature_extractor, mask_time_prob: float, mask_time_length: int):
        self.model, self.fe = model, feature_extractor
        self.mask_time_prob, self.mask_time_length = mask_time_prob, mask_time_length

    def __call__(self, features: list[dict]) -> dict:
        from transformers.models.wav2vec2.modeling_wav2vec2 import _compute_mask_indices, _sample_negative_indices

        batch = self.fe.pad(features, padding=True, return_tensors="pt")
        bsz, n_samples = batch["input_values"].shape
        seq_len = int(self.model._get_feat_extract_output_lengths(n_samples))
        sub_mask = self.model._get_feature_vector_attention_mask(seq_len, batch["attention_mask"]) \
            if batch.get("attention_mask") is not None else None
        mask = _compute_mask_indices((bsz, seq_len), self.mask_time_prob, self.mask_time_length,
                                     attention_mask=sub_mask, min_masks=2)
        neg = _sample_negative_indices((bsz, seq_len), self.model.config.num_negatives, mask_time_indices=mask)
        batch["mask_time_indices"] = torch.tensor(mask, dtype=torch.bool)
        batch["sampled_negative_indices"] = torch.tensor(neg, dtype=torch.long)
        if sub_mask is not None:
            batch["sub_attention_mask"] = sub_mask
        return batch


def load_pretraining_model(name: str, c):
    """Load ``Wav2Vec2ForPreTraining`` and refuse to continue if the quantiser was not in the checkpoint."""
    from transformers import Wav2Vec2ForPreTraining

    model, info = Wav2Vec2ForPreTraining.from_pretrained(
        name, output_loading_info=True, mask_time_prob=float(c.mask_time_prob),
        mask_time_length=int(c.mask_time_length), diversity_loss_weight=float(c.diversity_loss_weight),
        mask_feature_prob=0.0, layerdrop=0.0, hidden_dropout=0.0, attention_dropout=0.0, feat_proj_dropout=0.0)
    missing = [k for k in info.get("missing_keys", []) if any(q in k for q in QUANTIZER_KEYS)]
    if missing:
        raise CPTError(f"Quantiser/projection weights are newly initialised for {name}: {missing}. "
                       "CPT must not run with a random quantiser - see docs/OPEN_QUESTIONS.md.")
    return model, info


def _lr_lambda(warmup: int, total: int):
    def f(step: int) -> float:
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def run_cpt(cfg, unlabelled: pd.DataFrame, model_name: str, out_dir: str | Path, ckpt_dir: str | Path,
            results_dir: str | Path, resume: bool = True, time_budget_hours: float | None = None) -> dict:
    """Run (or resume) CPT. Exports the encoder to ``out_dir`` whenever it stops."""
    from accelerate import Accelerator

    from ..vocab import make_feature_extractor

    c = cfg.cpt
    out_dir, ckpt_dir, results_dir = Path(out_dir), Path(ckpt_dir), Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    state_path = ckpt_dir / "cpt_state.json"
    final_info = load_json(out_dir / "cpt_complete.json")
    if resume and final_info and final_info.get("completed"):
        log.info("CPT already completed: %s", out_dir)
        return final_info

    seed_everything(int(cfg.seed))
    cuda = torch.cuda.is_available()
    accelerator = Accelerator(mixed_precision="fp16" if (cuda and c.fp16) else "no", cpu=not cuda)
    fe = make_feature_extractor()
    model, load_info = load_pretraining_model(model_name, c)
    if c.freeze_feature_encoder:
        model.freeze_feature_encoder()
    if c.gradient_checkpointing:
        model.gradient_checkpointing_enable()
    model.train()

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=float(c.learning_rate), betas=(float(c.adam_beta1), float(c.adam_beta2)),
                                  eps=float(c.adam_epsilon), weight_decay=float(c.weight_decay))
    max_steps = int(c.max_steps)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda(int(c.warmup_steps), max_steps))
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)
    raw_model = accelerator.unwrap_model(model)

    st = {"completed_steps": 0, "epoch": 0, "batch_in_epoch": 0, "total_hours": 0.0, "low_ppl_streak": 0,
          "gumbel_temperature": float(c.max_gumbel_temperature), "aborted": None}
    if resume and state_path.exists():
        accelerator.load_state(str(ckpt_dir / "state"))
        st.update(load_json(state_path))
        log.info("CPT: resumed at step %d (epoch %d, batch %d)", st["completed_steps"], st["epoch"], st["batch_in_epoch"])
    elif not resume and ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)
    raw_model.set_gumbel_temperature(st["gumbel_temperature"])

    dataset = UnlabelledDataset(unlabelled, fe, float(c.max_crop_seconds), int(cfg.seed))
    collator = PretrainingCollator(raw_model, fe, float(c.mask_time_prob), int(c.mask_time_length))
    bs, ga = int(c.per_device_batch_size), int(c.gradient_accumulation_steps)
    budget_s = float(time_budget_hours or c.time_budget_hours) * 3600
    log_path = results_dir / "cpt_log.csv"
    log_rows = []
    if resume and log_path.exists():
        try:
            log_rows = pd.read_csv(log_path).to_dict("records")
        except pd.errors.EmptyDataError:  # stopped before the first log step
            log_rows = []
    run = RunInfo(results_dir / "cpt_run_info.json", stage="cpt")
    t0, steps_at_start = time.time(), st["completed_steps"]
    hours_at_start = st["total_hours"]
    acc = {"loss": 0.0, "contrastive": 0.0, "diversity": 0.0, "ppl": 0.0, "n": 0}
    stop_reason = None

    def save_checkpoint():
        st["total_hours"] = hours_at_start + (time.time() - t0) / 3600
        accelerator.save_state(str(ckpt_dir / "state"))
        save_json(st, state_path)
        pd.DataFrame(log_rows).to_csv(log_path, index=False)

    while stop_reason is None:
        dataset.epoch = st["epoch"]
        batches = epoch_batches(dataset.durations, bs, int(cfg.seed), st["epoch"])
        loader = torch.utils.data.DataLoader(
            dataset, batch_sampler=batches[st["batch_in_epoch"]:], collate_fn=collator,
            num_workers=int(c.get("num_workers", 0)))
        micro = 0
        for batch in loader:
            st["batch_in_epoch"] += 1
            batch.pop("sub_attention_mask", None)
            batch = {k: v.to(accelerator.device) for k, v in batch.items()}
            num_losses = batch["mask_time_indices"].sum().clamp(min=1)
            with accelerator.autocast():
                out = model(**batch)
            loss = out.loss / num_losses / ga
            accelerator.backward(loss)
            acc["loss"] += float(out.loss.detach() / num_losses)
            acc["contrastive"] += float(out.contrastive_loss.detach() / num_losses)
            acc["diversity"] += float(out.diversity_loss.detach() / num_losses)  # HF scales it by #masked
            acc["ppl"] += float(out.codevector_perplexity.detach())
            acc["n"] += 1
            micro += 1
            if micro % ga != 0:
                continue
            grad_norm = accelerator.clip_grad_norm_(params, float(c.max_grad_norm))
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            st["completed_steps"] += 1
            st["gumbel_temperature"] = max(float(c.max_gumbel_temperature) *
                                           float(c.gumbel_temperature_decay) ** st["completed_steps"],
                                           float(c.min_gumbel_temperature))
            raw_model.set_gumbel_temperature(st["gumbel_temperature"])
            step = st["completed_steps"]
            mean_loss = acc["loss"] / max(acc["n"], 1)
            mean_ppl = acc["ppl"] / max(acc["n"], 1)

            # Stability checks: NaN loss or collapsed codebook usage.
            if not math.isfinite(mean_loss):
                st["aborted"] = f"NaN/inf loss at step {step}"
            st["low_ppl_streak"] = st["low_ppl_streak"] + 1 if mean_ppl < float(c.perplexity_floor) else 0
            if st["low_ppl_streak"] >= int(c.perplexity_patience):
                st["aborted"] = f"codevector perplexity < {c.perplexity_floor} for {c.perplexity_patience} steps"

            if step % int(c.log_every) == 0 or st["aborted"]:
                row = {"step": step, "loss": mean_loss, "contrastive_loss": acc["contrastive"] / max(acc["n"], 1),
                       "diversity_loss": acc["diversity"] / max(acc["n"], 1), "codevector_perplexity": mean_ppl,
                       "gumbel_temperature": st["gumbel_temperature"],
                       "grad_norm": float(grad_norm) if grad_norm is not None else None,
                       "lr": scheduler.get_last_lr()[0], "elapsed_h": (time.time() - t0) / 3600}
                log_rows.append(row)
                log.info("CPT step %d | loss %.4f | ppl %.1f | temp %.3f | gn %s", step, mean_loss, mean_ppl,
                         st["gumbel_temperature"], row["grad_norm"])
            acc = {k: 0.0 if k != "n" else 0 for k in acc}
            total_h = hours_at_start + (time.time() - t0) / 3600
            if st["aborted"]:
                stop_reason = "aborted"
            elif step >= max_steps:
                stop_reason = "max_steps"
            elif total_h >= float(c.max_total_hours):
                stop_reason = "max_total_hours"
            elif time.time() - t0 > budget_s:
                stop_reason = "session_time_budget"
            if stop_reason or step % int(c.checkpoint_every) == 0:
                save_checkpoint()
            if stop_reason:
                break
        else:
            st["epoch"] += 1
            st["batch_in_epoch"] = 0
            if micro % ga != 0:
                optimizer.zero_grad(set_to_none=True)  # drop a partial accumulation at epoch end
        if len(dataset) == 0:
            raise CPTError("Unlabelled pool is empty")

    completed = stop_reason in ("max_steps", "max_total_hours")
    if st["aborted"]:
        log.error("CPT aborted: %s (checkpoint saved)", st["aborted"])
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_model.save_pretrained(str(out_dir))
    fe.save_pretrained(str(out_dir))
    _plot_log(log_rows, results_dir / "cpt_loss.png")
    info = run.finish(steps=st["completed_steps"] - steps_at_start, global_step=st["completed_steps"],
                      stop_reason=stop_reason)
    summary = {"completed": completed, "stop_reason": stop_reason, "aborted": st["aborted"],
               "completed_steps": st["completed_steps"], "max_steps": max_steps,
               "total_hours": round(hours_at_start + (time.time() - t0) / 3600, 3),
               "unlabelled_clips": len(dataset), "unlabelled_hours": round(float(unlabelled["duration"].sum()) / 3600, 2),
               "gpu_hours_total": info["total_gpu_hours"], "finished_at": now_iso(),
               "missing_keys_on_load": load_info.get("missing_keys", [])}
    save_json(summary, out_dir / "cpt_complete.json")
    save_json(summary, results_dir / "cpt_status.json")
    if completed and not cfg.train.get("keep_checkpoints", False):
        shutil.rmtree(ckpt_dir, ignore_errors=True)
    return summary


def _plot_log(rows: list[dict], path: Path) -> None:
    if not rows:
        return
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    df = pd.DataFrame(rows)
    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    ax1.plot(df["step"], df["loss"], "k-", label="total loss")
    ax1.plot(df["step"], df["contrastive_loss"], "k--", label="contrastive")
    ax1.set_xlabel("update step"); ax1.set_ylabel("loss per masked frame")
    ax2 = ax1.twinx()
    ax2.plot(df["step"], df["codevector_perplexity"], color="0.5", linestyle=":", label="codevector perplexity")
    ax2.set_ylabel("perplexity")
    lines = ax1.get_lines() + ax2.get_lines()
    ax1.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="upper right")
    ax1.set_title("Continued pretraining (CPT)")
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.close(fig)


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/adapt/staged.py
"""Staged fine-tuning (Stage B): B1 on out-of-domain labelled speech, then B2 on Common Voice.

B1 is identical for V3 and V4 (same CPT encoder, same OOD data, same settings), so it is trained
once into ``models/B1`` and reused.
"""
from __future__ import annotations

from pathlib import Path

from ..train.finetune import is_complete, run_finetune
from ..utils import get_logger, load_json

log = get_logger(__name__)


def run_stage_b1(cfg, *, ood_train, cv_dev, processor, cpt_dir: Path, models_dir: Path, results_dir: Path,
                 resume: bool, previous_outputs) -> dict:
    """Fine-tune the CPT encoder on OOD labelled data; select the best checkpoint on CV dev."""
    b1_dir = models_dir / "B1"
    if ood_train is None or len(ood_train) == 0:
        log.warning("No OOD labelled corpus available: stage B1 skipped (V3 = CPT + CV fine-tuning).")
        return {"stage_b1": "skipped", "init_for_b2": str(cpt_dir)}
    if resume and is_complete(b1_dir):
        return {"stage_b1": "complete", "init_for_b2": str(b1_dir), **(load_json(b1_dir / "training_complete.json") or {})}
    summary = run_finetune(
        cfg, tag="B1", train_ds=ood_train, eval_ds=cv_dev, processor=processor, init_from=str(cpt_dir),
        mask_time_prob=float(cfg.model.mask_time_prob), max_steps=int(cfg.adapt.b1_max_steps), model_dir=b1_dir,
        results_dir=results_dir / "B1", resume=resume, previous_outputs=previous_outputs,
        extra_info={"stage": "B1", "data": "OOD labelled (ÌròyìnSpeech + SLR86)"})
    if summary["status"] != "complete":
        return {"stage_b1": "incomplete", **summary}
    return {"stage_b1": "complete", "init_for_b2": str(b1_dir), **summary}


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/audio.py
"""Audio loading, resampling, silence trimming and duration filtering."""
from __future__ import annotations

from pathlib import Path

import numpy as np

from .utils import get_logger

log = get_logger(__name__)
SR = 16000


class AudioLoadError(RuntimeError):
    pass


def _to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 2:
        # soundfile returns (frames, channels)
        x = x.mean(axis=1)
    return x


def resample(x: np.ndarray, orig_sr: int, sr: int = SR) -> np.ndarray:
    if orig_sr == sr:
        return x.astype(np.float32, copy=False)
    import librosa

    return librosa.resample(x.astype(np.float32), orig_sr=orig_sr, target_sr=sr).astype(np.float32)


def load_audio(path: str | Path, sr: int = SR) -> np.ndarray:
    """Decode MP3/WAV/FLAC/OGG/M4A, convert to mono float32, resample to ``sr``.

    The signal is peak-normalised only if it would clip (|x| > 1).
    Raises :class:`AudioLoadError` for corrupt or empty files.
    """
    path = str(path)
    x = None
    orig_sr = None
    err: Exception | None = None
    try:
        import soundfile as sf

        x, orig_sr = sf.read(path, dtype="float32", always_2d=False)
    except Exception as exc:  # unsupported format -> fall back to librosa/audioread
        err = exc
        try:
            import librosa

            x, orig_sr = librosa.load(path, sr=None, mono=True)
        except Exception as exc2:
            raise AudioLoadError(f"cannot decode {path}: {exc2} (soundfile: {err})") from exc2
    x = _to_mono(np.asarray(x, dtype=np.float32))
    if x.size == 0:
        raise AudioLoadError(f"empty audio: {path}")
    if not np.all(np.isfinite(x)):
        raise AudioLoadError(f"non-finite samples: {path}")
    x = resample(x, int(orig_sr), sr)
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    if peak > 1.0:
        x = x / peak * 0.999
    return x.astype(np.float32)


def trim_silence(x: np.ndarray, top_db: float = 30) -> np.ndarray:
    """Trim leading/trailing silence (librosa ``effects.trim``)."""
    import librosa

    if x.size == 0:
        return x
    y, _ = librosa.effects.trim(x, top_db=top_db)
    return y if y.size else x


def duration(x: np.ndarray, sr: int = SR) -> float:
    return float(len(x)) / sr


def probe(path: str | Path) -> tuple[float, int] | None:
    """(duration seconds, native sample rate) from the file header, or None if unreadable."""
    try:
        import soundfile as sf

        info = sf.info(str(path))
        if info.frames <= 0:
            return None
        return info.frames / info.samplerate, int(info.samplerate)
    except Exception:
        try:
            import librosa

            return float(librosa.get_duration(path=str(path))), int(librosa.get_samplerate(str(path)))
        except Exception:
            return None


def duration_reason(dur: float, min_s: float, max_s: float) -> str | None:
    """None if the clip is kept, else the reason it is dropped."""
    if dur < min_s:
        return "too_short"
    if dur > max_s:
        return "too_long"
    return None


def load_trimmed(path: str | Path, sr: int = SR, top_db: float = 30) -> np.ndarray:
    return trim_silence(load_audio(path, sr), top_db=top_db)


def safe_load_trimmed(path: str | Path, sr: int = SR, top_db: float = 30) -> np.ndarray | None:
    """Like :func:`load_trimmed` but logs and returns None for corrupt files."""
    try:
        return load_trimmed(path, sr, top_db)
    except Exception as exc:
        log.warning("skipping corrupt audio %s: %s", path, exc)
        return None


def save_flac(path: str | Path, x: np.ndarray, sr: int = SR) -> None:
    """Write 16-bit FLAC (clip-safe)."""
    import soundfile as sf

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    if peak > 0.999:
        x = x / peak * 0.999
    sf.write(str(path), x.astype(np.float32), sr, subtype="PCM_16", format="FLAC")


def split_on_silence(x: np.ndarray, sr: int = SR, max_seconds: float = 15.0, top_db: float = 35) -> list[np.ndarray]:
    """Split long audio into chunks of at most ``max_seconds``, cutting at silences."""
    import librosa

    max_len = int(max_seconds * sr)
    if len(x) <= max_len:
        return [x]
    intervals = librosa.effects.split(x, top_db=top_db)
    if len(intervals) == 0:
        intervals = np.array([[0, len(x)]])
    chunks: list[np.ndarray] = []
    start, end = None, None
    for s, e in intervals:
        if start is None:
            start, end = s, e
        elif e - start <= max_len:
            end = e
        else:
            chunks.append(x[start:end])
            start, end = s, e
    if start is not None:
        chunks.append(x[start:end])
    out: list[np.ndarray] = []
    for c in chunks:  # hard-split any chunk that is still too long
        for i in range(0, len(c), max_len):
            piece = c[i: i + max_len]
            if len(piece) > int(0.2 * sr):
                out.append(piece)
    return out or [x[:max_len]]


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/augment/__init__.py
"""Tone-preserving data augmentation: audio transforms, MMS-TTS synthesis, offline set builder."""


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/augment/build_augmented.py
"""Offline generation of the augmented training set (2 copies per CV train clip, FLAC + manifest)."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from ..audio import SR, duration_reason, safe_load_trimmed, save_flac
from ..utils import get_logger, save_json
from .transforms import NoiseBank, RIRBank, apply_chain, choose_chain, median_f0, tempo

log = get_logger(__name__)


def clip_rng(seed: int, clip_index: int, copy_index: int) -> np.random.Generator:
    """Generator seeded by (seed, clip, copy): the augmented set is identical on every run."""
    return np.random.default_rng([int(seed), int(clip_index), int(copy_index)])


def build_augmented_set(train: pd.DataFrame, out_dir: str | Path, cfg_aug, seed: int, transforms: list[str],
                        min_s: float, max_s: float, top_db: float, musan_dir=None, rir_dir=None,
                        name: str = "main") -> tuple[pd.DataFrame, dict]:
    """Create ``copies_per_clip`` augmented copies of every row of ``train`` (needs ``path, text_tone``)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest_augmented.csv"
    noise_bank = NoiseBank.from_dir(musan_dir)
    rir_bank = RIRBank.build(int(cfg_aug["rir_bank_size"]), seed, tuple(cfg_aug["rt60"]), rir_dir)
    copies = int(cfg_aug["copies_per_clip"])
    rows, dropped = [], {"corrupt": 0, "too_short": 0, "too_long": 0}
    records = train.reset_index(drop=True).to_dict("records")
    for ci, r in enumerate(tqdm(records, desc=f"augment[{name}]", mininterval=10)):
        audio = None
        for k in range(copies):
            path = out_dir / f"aug_{ci:06d}_{k}.flac"
            rng = clip_rng(seed, ci, k)
            chain = choose_chain(rng, list(transforms), float(cfg_aug["transform_prob"]))
            if path.exists():
                # Deterministic: re-derive the description without recomputing audio.
                import soundfile as sf

                info = sf.info(str(path))
                rows.append({**_row(r, path, info.frames / info.samplerate), "chain": "+".join(chain)})
                continue
            if audio is None:
                audio = safe_load_trimmed(r["path"], SR, top_db)
                if audio is None:
                    dropped["corrupt"] += copies
                    break
            y, applied = apply_chain(audio, chain, rng, cfg_aug, noise_bank, rir_bank)
            dur = len(y) / SR
            reason = duration_reason(dur, min_s, max_s)
            if reason:
                dropped[reason] += 1
                continue
            save_flac(path, y)
            rows.append({**_row(r, path, dur), "chain": "+".join(chain), "chain_detail": ";".join(applied)})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False, encoding="utf-8")
    stats = {"name": name, "source_clips": len(records), "copies_per_clip": copies, "written": len(manifest),
             "dropped": dropped, "transforms": list(transforms),
             "hours": round(float(manifest["duration"].sum()) / 3600, 3) if len(manifest) else 0.0,
             "noise_source": "musan" if noise_bank.files else "generated coloured noise",
             "rir_source": rir_bank.labels[0].split(":")[0] if rir_bank.labels else None}
    save_json(stats, out_dir / "augment_stats.json")
    log.info("augmented[%s]: %d clips written, dropped %s", name, len(manifest), dropped)
    return manifest, stats


def _row(r: dict, path: Path, dur: float) -> dict:
    return {"path": str(path), "sentence": r.get("sentence", r["text_tone"]), "text_tone": r["text_tone"],
            "text_notone": r["text_notone"], "speaker": r.get("speaker", "unknown"), "corpus": "cv_aug",
            "split": "train", "duration": dur, "source_path": r["path"], "synthetic": False}


def pitch_check(train: pd.DataFrame, cfg_aug, seed: int, top_db: float, n: int = 20) -> dict:
    """Tempo perturbation must keep pitch: median F0 change < 3% (median over clips)."""
    rng = np.random.default_rng([seed, 31337])
    idx = rng.choice(len(train), size=min(n, len(train)), replace=False)
    results = []
    for i in idx:
        r = train.iloc[int(i)]
        x = safe_load_trimmed(r["path"], SR, top_db)
        if x is None:
            continue
        f = float(rng.choice(cfg_aug["tempo_factors"]))
        f0_orig, f0_aug = median_f0(x), median_f0(tempo(x, f))
        rel = abs(f0_aug - f0_orig) / f0_orig if np.isfinite(f0_orig) and f0_orig > 0 and np.isfinite(f0_aug) else None
        results.append({"path": str(r["path"]), "tempo_factor": f, "median_f0_original": f0_orig,
                        "median_f0_augmented": f0_aug, "relative_diff": rel})
    rels = [x["relative_diff"] for x in results if x["relative_diff"] is not None]
    limit = float(cfg_aug["pitch_check_max_rel_diff"])
    median_rel = float(np.median(rels)) if rels else float("nan")
    out = {"clips": results, "n_measured": len(rels), "median_relative_diff": median_rel,
           "share_within_limit": float(np.mean([x < limit for x in rels])) if rels else None,
           "limit": limit, "passed": bool(rels) and median_rel < limit}
    return out


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/augment/transforms.py
"""Audio transforms. Every transform takes a ``numpy.random.Generator`` so results are reproducible.

| transform | pitch      | use           |
|-----------|------------|---------------|
| tempo     | preserved  | main          |
| noise     | preserved  | main          |
| reverb    | preserved  | main          |
| gain      | preserved  | main          |
| speed     | SHIFTED    | ablation only |
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
from scipy.signal import fftconvolve

from ..utils import get_logger

log = get_logger(__name__)
SR = 16000
MAIN_TRANSFORMS = ("tempo", "noise", "reverb", "gain")


# ---------------------------------------------------------------- tempo / speed
def tempo(x: np.ndarray, factor: float) -> np.ndarray:
    """Time-stretch by ``factor`` (>1 = faster) with pitch preserved (phase vocoder)."""
    import librosa

    return librosa.effects.time_stretch(x.astype(np.float32), rate=float(factor)).astype(np.float32)


def speed(x: np.ndarray, factor: float, sr: int = SR) -> np.ndarray:
    """Speed perturbation by resampling: changes duration AND pitch (ablation only)."""
    import librosa

    return librosa.resample(x.astype(np.float32), orig_sr=sr, target_sr=int(round(sr / float(factor)))).astype(np.float32)


# ---------------------------------------------------------------- noise
def colored_noise(n: int, rng: np.random.Generator, beta: float = 1.0) -> np.ndarray:
    """Noise with a 1/f^beta power spectrum (0 = white, 1 = pink, 2 = brown)."""
    white = rng.standard_normal(n)
    spec = np.fft.rfft(white)
    f = np.fft.rfftfreq(n)
    f[0] = f[1] if len(f) > 1 else 1.0
    spec = spec / (f ** (beta / 2.0))
    y = np.fft.irfft(spec, n=n)
    return (y / (np.std(y) + 1e-9)).astype(np.float32)


def mix_at_snr(x: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    ps = float(np.mean(x ** 2)) + 1e-12
    pn = float(np.mean(noise ** 2)) + 1e-12
    scale = np.sqrt(ps / (pn * 10 ** (snr_db / 10.0)))
    return (x + scale * noise).astype(np.float32)


class NoiseBank:
    """Noise source: MUSAN files when available, otherwise generated coloured noise."""

    def __init__(self, files: list[str] | None = None, max_files: int = 2000):
        self.files = sorted(files or [])[:max_files]
        self._cache: dict[str, np.ndarray] = {}

    @classmethod
    def from_dir(cls, musan_dir: str | Path | None) -> "NoiseBank":
        if not musan_dir:
            return cls([])
        root = Path(musan_dir)
        files = [str(p) for p in root.rglob("*") if p.suffix.lower() in (".wav", ".flac")]
        log.info("noise bank: %d files from %s", len(files), root)
        return cls(files)

    def sample(self, n: int, rng: np.random.Generator) -> tuple[np.ndarray, str]:
        if self.files:
            path = self.files[int(rng.integers(len(self.files)))]
            if path not in self._cache:
                from ..audio import load_audio

                try:
                    self._cache[path] = load_audio(path, SR)
                except Exception:
                    self._cache[path] = np.zeros(0, np.float32)
                if len(self._cache) > 64:
                    self._cache.pop(next(iter(self._cache)))
            src = self._cache[path]
            if len(src) > SR // 4:
                if len(src) < n:
                    src = np.tile(src, int(np.ceil(n / len(src))))
                start = int(rng.integers(0, len(src) - n + 1))
                return src[start: start + n].astype(np.float32), f"musan:{Path(path).name}"
        beta = float(rng.choice([0.0, 1.0, 2.0]))
        return colored_noise(n, rng, beta), f"colored(beta={beta:g})"


def add_noise(x: np.ndarray, rng: np.random.Generator, bank: NoiseBank, snr_range=(10, 30)) -> tuple[np.ndarray, str]:
    snr = float(rng.uniform(*snr_range))
    noise, name = bank.sample(len(x), rng)
    return mix_at_snr(x, noise, snr), f"noise({name},snr={snr:.1f})"


# ---------------------------------------------------------------- reverberation
def synthetic_rir(rt60: float, rng: np.random.Generator, sr: int = SR) -> np.ndarray:
    """Exponentially decaying noise RIR (fallback when pyroomacoustics is unavailable)."""
    n = int(rt60 * 1.2 * sr)
    t = np.arange(n) / sr
    rir = rng.standard_normal(n) * np.exp(-6.9 * t / rt60)
    rir[0] = 1.0
    return (rir / np.max(np.abs(rir))).astype(np.float32)


def simulate_rir(rng: np.random.Generator, rt60_range=(0.2, 0.8), sr: int = SR) -> tuple[np.ndarray, float]:
    """Random shoebox room RIR with pyroomacoustics (image-source method)."""
    rt60 = float(rng.uniform(*rt60_range))
    try:
        import pyroomacoustics as pra
    except ImportError:
        return synthetic_rir(rt60, rng, sr), rt60
    for _ in range(10):
        dims = rng.uniform([3.0, 3.0, 2.5], [10.0, 8.0, 4.0])
        try:
            e_abs, max_order = pra.inverse_sabine(rt60, dims)
            room = pra.ShoeBox(dims, fs=sr, materials=pra.Material(e_abs), max_order=min(int(max_order), 17))
            src = rng.uniform([0.5, 0.5, 1.0], dims - 0.5)
            mic = rng.uniform([0.5, 0.5, 1.0], dims - 0.5)
            room.add_source(src)
            room.add_microphone(mic.reshape(3, 1))
            room.compute_rir()
            rir = np.asarray(room.rir[0][0], dtype=np.float32)
            if rir.size and np.all(np.isfinite(rir)) and np.max(np.abs(rir)) > 0:
                return rir / np.max(np.abs(rir)), rt60
        except Exception:
            continue
    return synthetic_rir(rt60, rng, sr), rt60


class RIRBank:
    """Fixed, seeded bank of RIRs (simulated, or real OpenSLR RIRs when provided)."""

    def __init__(self, rirs: list[np.ndarray], labels: list[str]):
        self.rirs, self.labels = rirs, labels

    @classmethod
    def build(cls, size: int, seed: int, rt60_range=(0.2, 0.8), rir_dir: str | Path | None = None) -> "RIRBank":
        rirs, labels = [], []
        if rir_dir:
            from ..audio import load_audio

            files = sorted(p for p in Path(rir_dir).rglob("*") if p.suffix.lower() in (".wav", ".flac"))
            files = [p for p in files if "noise" not in p.name.lower()][: max(size, 1) * 4]
            for p in files[:size]:
                try:
                    r = load_audio(p, SR)
                    if r.size:
                        rirs.append(r / (np.max(np.abs(r)) + 1e-9)); labels.append(f"real:{p.name}")
                except Exception:
                    continue
        rng = np.random.default_rng([seed, 7919])
        while len(rirs) < size:
            r, rt60 = simulate_rir(rng, rt60_range)
            rirs.append(r); labels.append(f"sim(rt60={rt60:.2f})")
        return cls(rirs, labels)


def reverb(x: np.ndarray, rir: np.ndarray) -> np.ndarray:
    """Convolve with an RIR, align the direct path and keep the original length and RMS."""
    d = int(np.argmax(np.abs(rir)))
    y = fftconvolve(x, rir)[d: d + len(x)]
    if len(y) < len(x):
        y = np.pad(y, (0, len(x) - len(y)))
    rms_x, rms_y = np.sqrt(np.mean(x ** 2)) + 1e-9, np.sqrt(np.mean(y ** 2)) + 1e-9
    return (y * rms_x / rms_y).astype(np.float32)


# ---------------------------------------------------------------- gain
def gain(x: np.ndarray, db: float) -> np.ndarray:
    """Apply gain in dB, then scale down only if the result would clip."""
    y = x * (10 ** (db / 20.0))
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    if peak > 0.99:
        y = y / peak * 0.99
    return y.astype(np.float32)


# ---------------------------------------------------------------- chains
def choose_chain(rng: np.random.Generator, transforms: list[str], p: float = 0.5) -> list[str]:
    """Each transform is included with probability ``p``; at least one is always applied."""
    chain = [t for t in transforms if rng.random() < p]
    if not chain:
        chain = [transforms[int(rng.integers(len(transforms)))]]
    order = ["speed", "tempo", "reverb", "noise", "gain"]
    return sorted(chain, key=order.index)


def apply_chain(x: np.ndarray, chain: list[str], rng: np.random.Generator, cfg_aug, noise_bank: NoiseBank,
                rir_bank: RIRBank) -> tuple[np.ndarray, list[str]]:
    applied = []
    y = x
    for t in chain:
        if t == "tempo":
            f = float(rng.choice(cfg_aug["tempo_factors"]))
            y = tempo(y, f); applied.append(f"tempo({f})")
        elif t == "speed":
            f = float(rng.choice(cfg_aug["speed_factors"]))
            y = speed(y, f); applied.append(f"speed({f})")
        elif t == "reverb":
            k = int(rng.integers(len(rir_bank.rirs)))
            y = reverb(y, rir_bank.rirs[k]); applied.append(f"reverb({rir_bank.labels[k]})")
        elif t == "noise":
            y, desc = add_noise(y, rng, noise_bank, tuple(cfg_aug["snr_db"])); applied.append(desc)
        elif t == "gain":
            db = float(rng.uniform(*cfg_aug["gain_db"]))
            y = gain(y, db); applied.append(f"gain({db:+.1f}dB)")
        else:
            raise ValueError(f"unknown transform {t}")
    return y, applied


def median_f0(x: np.ndarray, sr: int = SR) -> float:
    """Median F0 (Hz) over voiced frames using librosa.pyin."""
    import librosa

    f0, voiced, _ = librosa.pyin(x.astype(np.float32), fmin=60, fmax=500, sr=sr, frame_length=1024)
    f0 = f0[np.isfinite(f0)]
    return float(np.median(f0)) if f0.size else float("nan")


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/augment/tts.py
"""Synthetic Yoruba speech with MMS-TTS (``facebook/mms-tts-yor``, CC BY-NC 4.0, research use only).

Before synthesising, a coverage check verifies that the TTS tokenizer can represent the tone marks
and sub-dots of the normalised text. If tone marks cannot be represented, TTS is skipped
(``enabled_effective = false``): toneless audio must never be labelled as toned text.
"""
from __future__ import annotations

import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from ..audio import SR, duration_reason, resample, save_flac, trim_silence
from ..text import ACUTE, GRAVE, MACRON, SUBDOT, TONE_MARKS, nfc, strip_tones, tone_sequence
from ..utils import get_logger, save_json

log = get_logger(__name__)


class DummyTTS:
    """Offline stand-in for tests/smoke runs: renders the tone sequence as short sine syllables."""

    name = "dummy"
    sampling_rate = SR

    def __init__(self):
        self.vocab = None  # represents every character

    def synthesize(self, text: str, seed: int) -> np.ndarray:
        rng = np.random.default_rng(seed)
        f0 = {"H": 240.0, "M": 200.0, "L": 160.0}
        parts = []
        for lab in tone_sequence(text) or ["M"]:
            t = np.arange(int(0.12 * SR)) / SR
            parts.append(0.3 * np.sin(2 * np.pi * f0[lab] * t) * np.hanning(len(t)))
            parts.append(np.zeros(int(0.02 * SR)))
        y = np.concatenate([np.zeros(int(0.1 * SR))] + parts + [np.zeros(int(0.1 * SR))])
        return (y + 0.003 * rng.standard_normal(len(y))).astype(np.float32)


class MMSTTS:
    def __init__(self, model_name: str = "facebook/mms-tts-yor", device: str | None = None):
        import torch
        from transformers import AutoTokenizer, VitsModel

        self.name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = VitsModel.from_pretrained(model_name).to(self.device).eval()
        self.sampling_rate = int(self.model.config.sampling_rate)
        self.vocab = set(self.tokenizer.get_vocab().keys())
        self.normal_form = "NFC"

    def synthesize(self, text: str, seed: int) -> np.ndarray:
        import torch

        torch.manual_seed(seed)
        text = unicodedata.normalize(self.normal_form, text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            wav = self.model(**inputs).waveform[0].float().cpu().numpy()
        return wav.astype(np.float32)


def load_tts(model_name: str):
    if model_name == "dummy":
        return DummyTTS()
    return MMSTTS(model_name)


def coverage_report(sentences: list[str], vocab: set[str] | None) -> dict:
    """Which characters (NFC and NFD forms) the TTS tokenizer cannot represent."""
    text = " ".join(sentences)
    if vocab is None:
        return {"normal_form": "NFC", "missing_nfc": {}, "missing_nfd": {}, "tone_marks_representable": True,
                "subdots_representable": True, "enabled_effective": True}

    def missing(form: str) -> dict[str, int]:
        out: dict[str, int] = {}
        for ch in unicodedata.normalize(form, text):
            if ch != " " and ch not in vocab and ch.lower() not in vocab:
                out[ch] = out.get(ch, 0) + 1
        return out

    def tones_ok(form: str, miss: dict[str, int]) -> tuple[bool, bool]:
        # High (acute) and low (grave) tones carry the contrast in standard orthography and must be
        # representable. The mid-tone macron is rare (mid tone is normally unmarked); sentences that
        # contain an unrepresentable macron are skipped one by one instead of disabling TTS.
        decomposed = {ch: unicodedata.normalize("NFD", ch) for ch in miss}
        toned = [ch for ch, d in decomposed.items() if (GRAVE in d or ACUTE in d)]
        dotted = [ch for ch, d in decomposed.items() if SUBDOT in d]
        return not toned, not dotted

    m_nfc, m_nfd = missing("NFC"), missing("NFD")
    t_nfc, d_nfc = tones_ok("NFC", m_nfc)
    t_nfd, d_nfd = tones_ok("NFD", m_nfd)
    share = {f: float(np.mean([sentence_representable(s, vocab, f) for s in sentences])) if sentences else 0.0
             for f in ("NFC", "NFD")}
    if t_nfc and d_nfc:
        form, t_ok, d_ok = "NFC", True, True
    elif t_nfd and d_nfd:
        form, t_ok, d_ok = "NFD", True, True
    else:
        # A few rare characters (e.g. ḿ, an m with a tone mark) may be missing. Tones still count as
        # representable if the core toned vowels can be written in this form; the sentences that contain
        # an unrepresentable character are then skipped one by one (never stripped).
        form = max(("NFC", "NFD"), key=lambda f: share[f])
        core = [unicodedata.normalize(form, v + m) for v in "aeiou" for m in (ACUTE, GRAVE)]
        core_ok = all(all(ch in vocab or ch.lower() in vocab for ch in c) for c in core)
        dot_ok = all(ch in vocab for ch in unicodedata.normalize(form, "ẹọṣ"))
        t_ok, d_ok = core_ok and share[form] >= 0.5, dot_ok
    named = lambda d: {f"{k} ({unicodedata.name(k, hex(ord(k)))})": v for k, v in d.items()}  # noqa: E731
    return {
        "normal_form": form, "missing_nfc": named(m_nfc), "missing_nfd": named(m_nfd),
        "tone_marks_representable": t_ok, "subdots_representable": d_ok,
        "enabled_effective": bool(t_ok and d_ok),
        "share_of_sentences_representable": round(share[form], 4),
        "macron_representable": not any(MACRON in unicodedata.normalize("NFD", ch)
                                        for ch in (m_nfc if form == "NFC" else m_nfd)),
        "policy": "sentences containing any unrepresentable character are skipped (never silently stripped)",
        "combining_marks_in_vocab": {unicodedata.name(c): c in vocab for c in (GRAVE, ACUTE, MACRON, SUBDOT)},
    }


def sentence_representable(sentence: str, vocab: set[str] | None, form: str) -> bool:
    if vocab is None:
        return True
    return all(ch == " " or ch in vocab or ch.lower() in vocab for ch in unicodedata.normalize(form, sentence))


def synthesize_set(sentences: pd.DataFrame, out_dir: str | Path, results_dir: str | Path, model_name: str,
                   seed: int, max_items: int, min_s: float, max_s: float, top_db: float,
                   listening_sample: int = 50) -> tuple[pd.DataFrame, dict]:
    """Synthesise up to ``max_items`` sentences (column ``text_tone``); returns (manifest, coverage)."""
    out_dir, results_dir = Path(out_dir), Path(results_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        tts = load_tts(model_name)
    except Exception as exc:
        cov = {"enabled_effective": False, "error": f"could not load {model_name}: {exc}"}
        save_json(cov, results_dir / "tts_coverage.json")
        log.warning("TTS disabled: %s", cov["error"])
        return pd.DataFrame(), cov

    texts = sentences["text_tone"].tolist()
    cov = coverage_report(texts, tts.vocab)
    cov["model"] = model_name
    save_json(cov, results_dir / "tts_coverage.json")
    if not cov["enabled_effective"]:
        log.warning("=" * 80 + "\nTTS SKIPPED: the TTS tokenizer cannot represent Yoruba tone marks/sub-dots "
                    "(see results/V2/tts_coverage.json). V2/V4 will train without synthetic speech.\n" + "=" * 80)
        return pd.DataFrame(), cov
    if hasattr(tts, "normal_form"):
        tts.normal_form = cov["normal_form"]

    rows, skipped = [], {"unrepresentable": 0, "too_short": 0, "too_long": 0, "error": 0}
    for i, r in enumerate(sentences.to_dict("records")):
        if len(rows) >= max_items:
            break
        text = r["text_tone"]
        if not sentence_representable(text, tts.vocab, cov["normal_form"]):
            skipped["unrepresentable"] += 1
            continue
        path = out_dir / f"tts_{i:06d}.flac"
        try:
            if path.exists():
                import soundfile as sf

                y, _ = sf.read(str(path), dtype="float32")
            else:
                y = tts.synthesize(text, seed=seed * 100003 + i)
                y = trim_silence(resample(y, tts.sampling_rate, SR), top_db)
            dur = len(y) / SR
            reason = duration_reason(dur, min_s, max_s)
            if reason:
                skipped[reason] += 1
                continue
            if not path.exists():
                save_flac(path, y)
        except Exception as exc:
            log.warning("TTS failed for %r: %s", text[:40], exc)
            skipped["error"] += 1
            continue
        rows.append({"path": str(path), "sentence": r.get("sentence", text), "text_tone": text,
                     "text_notone": strip_tones(text), "speaker": "tts", "corpus": "tts", "split": "train",
                     "duration": dur, "synthetic": True, "chain": "tts"})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(out_dir / "manifest_tts.csv", index=False, encoding="utf-8")
    cov["synthesised"] = len(rows)
    cov["skipped"] = skipped
    save_json(cov, results_dir / "tts_coverage.json")

    # Listening sample for manual quality control.
    sample_dir = results_dir / "tts_listening_sample"
    if len(manifest):
        sample_dir.mkdir(parents=True, exist_ok=True)
        pick = manifest.sample(n=min(listening_sample, len(manifest)), random_state=seed)
        for p in pick["path"]:
            shutil.copy2(p, sample_dir / Path(p).name)
        pick.assign(file=[Path(p).name for p in pick["path"]])[["file", "text_tone"]].to_csv(
            sample_dir / "transcripts.csv", index=False, encoding="utf-8")
    log.info("TTS: %d clips synthesised (%s)", len(rows), skipped)
    return manifest, cov


def nfc_all(xs):
    return [nfc(x) for x in xs]


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/cli.py
"""Command-line interface: ``yoruba-asr <command>`` (or ``python -m yoruba_asr.cli <command>``)."""
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import List, Optional

import typer

app = typer.Typer(add_completion=False, help="Yoruba ASR (XLS-R) - data, training, evaluation and inference.",
                  no_args_is_help=True, pretty_exceptions_enable=False)

ConfigOpt = typer.Option(None, "--config", "-c", help="YAML config(s); base.yaml is always merged first.")
SetOpt = typer.Option(None, "--set", help="Override a config value, e.g. --set train.max_steps=10")
ResumeOpt = typer.Option(True, "--resume/--no-resume", help="Resume from checkpoints / skip finished work.")
DryOpt = typer.Option(False, "--dry-run", help="Print the plan without running anything.")


def _cfg(config: Optional[List[Path]], sets: Optional[List[str]], default: str = "base.yaml"):
    from .config import CONFIGS_DIR, load_config

    files = list(config) if config else [CONFIGS_DIR / default]
    return load_config(files, sets)


def _done(title: str, payload) -> None:
    typer.echo(f"\n== {title} finished ==")
    if isinstance(payload, dict):
        short = {k: v for k, v in payload.items() if not isinstance(v, (dict, list)) or k in ("tts",)}
        typer.echo(json.dumps(short, indent=2, ensure_ascii=False, default=str)[:3000])


def _dry(stage: str) -> None:
    from .pipeline import run_stage

    run_stage(stage, dry_run=True)


@app.command()
def audit(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, dry_run: bool = DryOpt):
    """Corpus audit (clip counts, hours, speakers, tone coverage)."""
    if dry_run:
        return _dry("audit")
    from .pipeline import stage_audit

    stage_audit(_cfg(config, set))
    typer.echo("== audit finished: results/corpus_audit.md ==")


@app.command()
def prepare(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
            dry_run: bool = DryOpt):
    """Load, normalise, filter, leakage-check, build vocabulary and cache Common Voice."""
    if dry_run:
        return _dry("prepare")
    from .pipeline import stage_prepare

    _done("prepare", stage_prepare(_cfg(config, set), resume=resume))


@app.command()
def augment(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
            dry_run: bool = DryOpt):
    """Build augmented copies + TTS clips for V2/V4."""
    if dry_run:
        return _dry("augment")
    from .pipeline import stage_augment

    _done("augment", stage_augment(_cfg(config, set, "v2_augmentation.yaml"), resume=resume))


@app.command("tune-mask")
def tune_mask(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
              skip_mask_tuning: bool = typer.Option(False, "--skip-mask-tuning", help="Use 0.075 without tuning."),
              dry_run: bool = DryOpt):
    """Tune mask_time_prob for V2/V4 on the dev set."""
    if dry_run:
        return _dry("tune_mask")
    from .pipeline import stage_tune_mask

    cfg = _cfg(config, set, "v2_augmentation.yaml")
    if skip_mask_tuning:
        cfg.mask_tuning.skip = True
    _done("tune-mask", stage_tune_mask(cfg, resume=resume))


@app.command()
def cpt(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
        dry_run: bool = DryOpt):
    """Continued self-supervised pretraining (resumable)."""
    if dry_run:
        return _dry("cpt")
    from .pipeline import stage_cpt

    _done("cpt", stage_cpt(_cfg(config, set, "cpt.yaml"), resume=resume))


@app.command()
def train(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
          dry_run: bool = DryOpt):
    """Train one variant (the config's `variant`: V1/V2/V3/V4) and evaluate it on the test set."""
    cfg = _cfg(config, set, "v1_baseline.yaml")
    if dry_run:
        return _dry(cfg.variant)
    from .pipeline import stage_train

    _done(f"train {cfg.variant}", stage_train(cfg, resume=resume))


@app.command()
def evaluate(variant: Optional[List[str]] = typer.Option(None, "--variant", help="Variant(s) to evaluate"),
             config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt,
             force: bool = typer.Option(False, "--force", help="Re-decode even if metrics exist."),
             dry_run: bool = DryOpt):
    """Evaluate trained variants on the CV test set (+ significance + effects)."""
    if dry_run:
        return _dry("evaluate")
    from .pipeline import stage_evaluate

    stage_evaluate(_cfg(config, set), variants=list(variant) if variant else None, force=force)
    typer.echo("== evaluate finished: results/summary.csv ==")


@app.command()
def significance(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt):
    """Blockwise (speaker) bootstrap significance tests."""
    from .pipeline import stage_significance

    stage_significance(_cfg(config, set))
    typer.echo("== significance finished: results/significance.csv ==")


@app.command()
def effects(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt):
    """Factorial main effects and interaction."""
    from .pipeline import stage_effects

    stage_effects(_cfg(config, set))
    typer.echo("== effects finished: results/factorial_effects.csv ==")


@app.command()
def ablation(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
             only: Optional[str] = typer.Option(None, "--only", help="Run a single ablation, e.g. no_tts"),
             dry_run: bool = DryOpt):
    """Leave-one-out ablation runs of V2."""
    if dry_run:
        return _dry("ablation")
    from .pipeline import stage_ablation

    stage_ablation(_cfg(config, set, "ablation.yaml"), resume=resume, only=only)
    typer.echo("== ablation finished: results/ablation.csv ==")


@app.command()
def analyse(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt):
    """Error analysis, figures and thesis tables."""
    from .pipeline import stage_analyse

    _done("analyse", stage_analyse(_cfg(config, set)))


@app.command("export-tables")
def export_tables(config: Optional[List[Path]] = ConfigOpt, set: Optional[List[str]] = SetOpt):
    """Write results/thesis_tables.{md,csv,tex}."""
    from .pipeline import stage_export_tables

    _done("export-tables", stage_export_tables(_cfg(config, set)))


@app.command()
def transcribe(audio: Path = typer.Argument(..., help="Audio file (wav/mp3/flac/ogg/m4a)"),
               model: str = typer.Option("models/V4", "--model", help="Model folder or Hugging Face repo id"),
               no_tone: bool = typer.Option(False, "--no-tone", help="Print the transcript without tone marks")):
    """Transcribe an audio file with a trained model."""
    from .inference import Transcriber

    t = Transcriber.from_dir(model) if Path(model).exists() else Transcriber.from_hub(model)
    r = t.transcribe_file(audio)
    typer.echo(r["text_notone"] if no_tone else r["text"])
    typer.echo(f"[{r['model']}, {r['seconds']} s]", err=True)


@app.command()
def stage(name: str = typer.Argument(..., help="Stage name as used in the Kaggle notebook"),
          smoke: bool = typer.Option(False, "--smoke", help="Layer configs/smoke.yaml on top"),
          set: Optional[List[str]] = SetOpt, resume: bool = ResumeOpt,
          only: Optional[str] = typer.Option(None, "--only"), dry_run: bool = DryOpt):
    """Run a notebook stage (prepare, V1, augment+tune_mask, cpt, ..., all_light)."""
    from .config import parse_overrides
    from .pipeline import next_step_message, run_stage

    run_stage(name, parse_overrides(set), smoke=smoke, resume=resume, only=only, dry_run=dry_run)
    typer.echo(next_step_message(name))


@app.command("smoke")
def smoke_cmd(set: Optional[List[str]] = SetOpt, dry_run: bool = DryOpt,
              fixture: Path = typer.Option(None, "--fixture", help="Fixture data folder (default: tests/fixtures/data)"),
              out: Path = typer.Option(None, "--out", help="Work root (default: ./_smoke)")):
    """End-to-end smoke pipeline on the tiny synthetic fixture with a tiny random model (CPU, minutes)."""
    from .config import REPO_ROOT, parse_overrides
    from .pipeline import run_stage

    overrides = parse_overrides(set)
    if not Path("/kaggle/input").exists() and "paths.input_root" not in overrides:
        fixture = fixture or REPO_ROOT / "tests" / "fixtures" / "data"
        if not (fixture / "common_voice").exists():
            import importlib.util

            spec = importlib.util.spec_from_file_location("mk", REPO_ROOT / "scripts" / "make_tiny_fixture.py")
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)  # type: ignore[union-attr]
            mod.make_fixture(fixture)
        root = (out or Path("_smoke")).resolve()
        overrides = {"paths.input_root": str(fixture), "paths.work_root": str(root),
                     "paths.tmp_dir": str(root / "tmp"), **overrides}
    run_stage("smoke", overrides, smoke=True, resume=False, dry_run=dry_run)
    typer.echo("== smoke pipeline finished ==")


def main() -> None:
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    app()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/config.py
"""Configuration loading, merging and dataset path discovery.

Configs are YAML files merged in this order:
    configs/base.yaml  ->  variant/stage config(s)  ->  command-line overrides (--set key=value)

Values may reference other values with ``${dotted.key}`` (e.g. ``${paths.work_dir}/results``).
The merged result is a :class:`Config`: a nested dict that also allows attribute access
(``cfg.train.max_steps``).
"""
from __future__ import annotations

import copy
import os
import re
import shutil
import tarfile
import unicodedata
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable

import yaml

PACKAGE_ROOT = Path(__file__).resolve().parent
# Repository root (…/yoruba-asr) when running from a source checkout.
REPO_ROOT = PACKAGE_ROOT.parent.parent
CONFIGS_DIR = REPO_ROOT / "configs"

_INTERP = re.compile(r"\$\{([^}]+)\}")
OUTPUT_MARKER = "YORUBA_ASR_OUTPUT.txt"  # not a dotfile, so it survives Kaggle's output saving
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg", ".m4a", ".opus")


# --------------------------------------------------------------------------------------
# Config object
# --------------------------------------------------------------------------------------
class Config(dict):
    """Nested dictionary with attribute access and dotted-key helpers."""

    def __getattr__(self, key: str) -> Any:
        try:
            return self[key]
        except KeyError as exc:
            raise AttributeError(key) from exc

    def __setattr__(self, key: str, value: Any) -> None:
        self[key] = value

    def __deepcopy__(self, memo):
        return Config(copy.deepcopy(dict(self), memo))

    def get_path(self, dotted: str, default: Any = None) -> Any:
        node: Any = self
        for part in dotted.split("."):
            if not isinstance(node, dict) or part not in node:
                return default
            node = node[part]
        return node

    def set_path(self, dotted: str, value: Any) -> None:
        node = self
        parts = dotted.split(".")
        for part in parts[:-1]:
            if part not in node or not isinstance(node[part], dict):
                node[part] = Config()
            node = node[part]
        node[parts[-1]] = _wrap(value)

    def to_dict(self) -> dict:
        return _unwrap(self)


def _wrap(obj: Any) -> Any:
    if isinstance(obj, dict) and not isinstance(obj, Config):
        return Config({k: _wrap(v) for k, v in obj.items()})
    if isinstance(obj, Config):
        for k, v in list(obj.items()):
            obj[k] = _wrap(v)
        return obj
    if isinstance(obj, list):
        return [_wrap(v) for v in obj]
    return obj


def _unwrap(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: _unwrap(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_unwrap(v) for v in obj]
    return obj


def deep_merge(base: dict, other: dict) -> dict:
    """Recursively merge ``other`` into a copy of ``base`` (``other`` wins)."""
    out = copy.deepcopy(base)
    for key, value in (other or {}).items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = deep_merge(out[key], value)
        else:
            out[key] = copy.deepcopy(value)
    return out


def _resolve_interpolations(cfg: Config) -> None:
    """Replace ``${a.b}`` references until nothing changes (max 10 passes)."""

    def resolve(value: Any) -> Any:
        if isinstance(value, str):
            def repl(m: re.Match) -> str:
                ref = cfg.get_path(m.group(1).strip())
                if ref is None:
                    raise KeyError(f"Config interpolation: unknown key '{m.group(1)}'")
                return str(ref)
            return _INTERP.sub(repl, value)
        if isinstance(value, dict):
            for k in value:
                value[k] = resolve(value[k])
            return value
        if isinstance(value, list):
            return [resolve(v) for v in value]
        return value

    for _ in range(10):
        before = repr(cfg)
        resolve(cfg)
        if repr(cfg) == before:
            break


def parse_overrides(items: Iterable[str] | dict | None) -> dict:
    """Turn ``["train.max_steps=10", ...]`` (or a dict) into ``{"train.max_steps": 10}``."""
    if not items:
        return {}
    if isinstance(items, dict):
        return dict(items)
    out = {}
    for item in items:
        if "=" not in item:
            raise ValueError(f"Override must look like key=value, got: {item!r}")
        key, raw = item.split("=", 1)
        out[key.strip()] = yaml.safe_load(raw)
    return out


def load_config(
    config_files: str | Path | list[str | Path] | None = None,
    overrides: Iterable[str] | dict | None = None,
    include_base: bool = True,
) -> Config:
    """Load ``base.yaml`` + the given config files + overrides into a :class:`Config`."""
    if config_files is None:
        config_files = []
    if isinstance(config_files, (str, Path)):
        config_files = [config_files]
    files = [Path(f) for f in config_files]

    merged: dict = {}
    if include_base:
        base = None
        for f in files:
            if f.name == "base.yaml":
                base = f
                break
        if base is None:
            candidates = [f.parent / "base.yaml" for f in files] + [CONFIGS_DIR / "base.yaml"]
            base = next((c for c in candidates if c.exists()), None)
        if base is not None:
            merged = yaml.safe_load(base.read_text(encoding="utf-8")) or {}
        files = [f for f in files if f.name != "base.yaml" or f != base]

    for f in files:
        data = yaml.safe_load(Path(f).read_text(encoding="utf-8")) or {}
        merged = deep_merge(merged, data)

    cfg = _wrap(merged)
    for key, value in parse_overrides(overrides).items():
        cfg.set_path(key, value)
    _resolve_interpolations(cfg)
    return cfg


def config_path(name: str) -> Path:
    """Path of a bundled config (``"v1_baseline"`` -> ``configs/v1_baseline.yaml``)."""
    p = Path(name)
    if p.suffix == ".yaml" and p.exists():
        return p
    return CONFIGS_DIR / (name if name.endswith(".yaml") else f"{name}.yaml")


# --------------------------------------------------------------------------------------
# Typed views
# --------------------------------------------------------------------------------------
@dataclass
class Paths:
    input_root: Path
    work_dir: Path
    tmp_dir: Path
    results_dir: Path
    models_dir: Path
    cache_dir: Path
    shared_dir: Path
    manifests_dir: Path
    checkpoints_dir: Path
    augmented_dir: Path

    @classmethod
    def from_config(cls, cfg: Config, create: bool = True) -> "Paths":
        p = cfg.paths
        obj = cls(**{k: Path(p[k]) for k in cls.__dataclass_fields__})
        if create:
            for name in cls.__dataclass_fields__:
                if name != "input_root":
                    getattr(obj, name).mkdir(parents=True, exist_ok=True)
            marker = obj.work_dir / OUTPUT_MARKER
            if not marker.exists():
                marker.write_text("yoruba_asr work directory\n", encoding="utf-8")
        return obj


@dataclass
class DataPaths:
    """Dataset folders found by :func:`discover_paths`."""

    common_voice: Path | None = None
    slr86: list[Path] = field(default_factory=list)
    iroyin: Path | None = None
    iroyin_manifest: Path | None = None
    bibletts: Path | None = None
    musan: Path | None = None
    rirs: Path | None = None
    previous_outputs: list[Path] = field(default_factory=list)
    extracted: list[Path] = field(default_factory=list)
    notes: list[str] = field(default_factory=list)

    def summary(self) -> dict:
        return {
            "common_voice": str(self.common_voice) if self.common_voice else None,
            "slr86": [str(p) for p in self.slr86],
            "iroyin": str(self.iroyin) if self.iroyin else None,
            "iroyin_manifest": str(self.iroyin_manifest) if self.iroyin_manifest else None,
            "bibletts": str(self.bibletts) if self.bibletts else None,
            "musan": str(self.musan) if self.musan else None,
            "rirs": str(self.rirs) if self.rirs else None,
            "previous_outputs": [str(p) for p in self.previous_outputs],
            "extracted_archives": [str(p) for p in self.extracted],
            "notes": self.notes,
        }


# --------------------------------------------------------------------------------------
# Path discovery
# --------------------------------------------------------------------------------------
def _plain(name: str) -> str:
    """Lower-case, accent-free version of a folder name (ÌròyìnSpeech -> iroyinspeech)."""
    s = unicodedata.normalize("NFD", name)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.lower()


def _is_archive(name: str) -> bool:
    n = name.lower()
    return n.endswith((".tar.gz", ".tgz", ".tar", ".zip", ".tar.bz2", ".tar.xz"))


def _archive_stem(name: str) -> str:
    for ext in (".tar.gz", ".tar.bz2", ".tar.xz", ".tgz", ".tar", ".zip"):
        if name.lower().endswith(ext):
            return name[: -len(ext)]
    return name


def extract_archive(archive: Path, dest_root: Path) -> Path:
    """Extract ``archive`` into ``dest_root/<stem>`` once (a ``.done`` marker skips re-extraction)."""
    dest = dest_root / _archive_stem(archive.name)
    done = dest / ".done"
    if done.exists():
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    if archive.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                target = (dest / member).resolve()
                if not str(target).startswith(str(dest.resolve())):
                    raise RuntimeError(f"Unsafe path in archive {archive}: {member}")
            zf.extractall(dest)
    else:
        with tarfile.open(archive) as tf:
            try:
                tf.extractall(dest, filter="data")  # Python >= 3.12 (and patched 3.10/3.11)
            except TypeError:
                for member in tf.getmembers():
                    target = (dest / member.name).resolve()
                    if not str(target).startswith(str(dest.resolve())):
                        raise RuntimeError(f"Unsafe path in archive {archive}: {member.name}")
                tf.extractall(dest)
    done.write_text("ok", encoding="utf-8")
    return dest


def _walk(root: Path, skip: set[Path]):
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        d = Path(dirpath)
        if d in skip:
            dirnames[:] = []
            continue
        if OUTPUT_MARKER in filenames or (d / "shared" / "vocab.json").exists():
            # A previous notebook output: never treat it as raw data.
            yield d, "previous_output", filenames
            dirnames[:] = []
            continue
        # Do not descend into huge audio folders.
        dirnames[:] = [x for x in dirnames if x not in ("clips", "__pycache__", ".git")]
        yield d, "dir", filenames


def _first_slr86_id(line_index: Path) -> str:
    try:
        with open(line_index, encoding="utf-8") as fh:
            first = fh.readline()
    except Exception:
        return ""
    return re.split(r"[\t ]", first.strip(), maxsplit=1)[0]


def discover_paths(cfg: Config, verbose: bool = True) -> DataPaths:
    """Find dataset folders under ``paths.input_root`` by marker files.

    * Common Voice: a folder containing ``train.tsv`` + ``dev.tsv`` + ``test.tsv``.
    * SLR86: folders containing ``line_index.tsv`` (ids like ``yof_…``/``yom_…``).
    * ÌròyìnSpeech: a folder whose name contains "iroyin" (or ``data.iroyin.manifest``).
    * BibleTTS / MUSAN / RIRs: folders named ``*bibletts*``, ``musan``, ``*rir*``.
    * Previous outputs: folders containing the ``.yoruba_asr_output`` marker.
    Archives (``.tar.gz``/``.zip``…) are extracted to ``tmp_dir/extracted`` and searched too.
    Kaggle dataset slugs are never hard-coded.
    """
    found = DataPaths()
    input_root = Path(cfg.paths.input_root)
    tmp_dir = Path(cfg.paths.tmp_dir)
    work_dir = Path(cfg.paths.work_dir).resolve() if cfg.paths.get("work_dir") else None
    roots = [input_root] if input_root.exists() else []
    if not roots:
        found.notes.append(f"input_root {input_root} does not exist")

    skip: set[Path] = set()
    if work_dir is not None:
        skip.add(work_dir)
    cv_candidates: list[Path] = []

    def scan(root: Path, archives: list[Path]):
        for d, kind, files in _walk(root, skip):
            if kind == "previous_output":
                if work_dir is None or d.resolve() != work_dir:
                    found.previous_outputs.append(d)
                continue
            fset = set(files)
            name = _plain(d.name)
            if {"train.tsv", "dev.tsv", "test.tsv"} <= fset:
                cv_candidates.append(d)
            if "line_index.tsv" in fset and _first_slr86_id(d / "line_index.tsv").startswith("yo"):
                found.slr86.append(d)
            if "iroyin" in name and found.iroyin is None:
                found.iroyin = d
            if "bibletts" in name and found.bibletts is None:
                found.bibletts = d
            if name == "musan" and found.musan is None:
                found.musan = d
            if ("rir" in name and "rirs_noises" not in name) and found.rirs is None:
                found.rirs = d
            if name == "rirs_noises" and found.rirs is None:
                found.rirs = d
            for f in files:
                if _is_archive(f):
                    archives.append(d / f)

    archives: list[Path] = []
    for root in roots:
        scan(root, archives)

    if cfg.paths.get("extract_archives", True) and archives:
        extract_root = tmp_dir / "extracted"
        for arc in archives:
            try:
                dest = extract_archive(arc, extract_root)
                found.extracted.append(dest)
                if verbose:
                    print(f"[discover] extracted {arc.name} -> {dest}")
            except Exception as exc:  # corrupted/unsupported archive
                found.notes.append(f"could not extract {arc}: {exc}")
        for dest in found.extracted:
            scan(dest, [])

    # Explicit paths from the config win over auto-detection.
    d = cfg.data
    if d.common_voice.get("dir"):
        cv_candidates.insert(0, Path(d.common_voice.dir))
    if d.slr86.get("dir"):
        found.slr86 = [p.parent for p in Path(d.slr86.dir).rglob("line_index.tsv")] or found.slr86
    if d.iroyin.get("dir"):
        found.iroyin = Path(d.iroyin.dir)
    if d.iroyin.get("manifest"):
        found.iroyin_manifest = Path(d.iroyin.manifest)
    if d.bibletts.get("dir"):
        found.bibletts = Path(d.bibletts.dir)
    if d.get("musan", {}).get("dir"):
        found.musan = Path(d.musan.dir)
    if d.get("rirs", {}).get("dir"):
        found.rirs = Path(d.rirs.dir)

    if cv_candidates:
        # Prefer the folder with the most clips (handles duplicates: archive + extracted copy).
        def n_clips(p: Path) -> int:
            c = p / "clips"
            return sum(1 for _ in c.iterdir()) if c.exists() else 0

        cv_candidates = list(dict.fromkeys(cv_candidates))
        found.common_voice = max(cv_candidates, key=n_clips)
        if len(cv_candidates) > 1:
            found.notes.append(f"several Common Voice folders found; using {found.common_voice}")

    found.slr86 = sorted(set(found.slr86))
    if not d.slr86.get("enabled", True):
        found.slr86 = []
    elif not found.slr86 and d.slr86.get("download", False):
        # Not attached: download it (no Kaggle upload needed; Internet must be on).
        from .data.download import download_openslr86

        try:
            found.slr86 = download_openslr86(tmp_dir / "downloads" / "openslr86")
            found.notes.append("OpenSLR 86 downloaded automatically")
        except Exception as exc:
            found.notes.append(f"OpenSLR 86 download failed: {exc}")
    if not d.iroyin.get("enabled", True):
        found.iroyin, found.iroyin_manifest = None, None
    if not d.bibletts.get("enabled", False):
        found.bibletts = None

    if verbose:
        print("[discover] " + ", ".join(
            f"{k}={v}" for k, v in found.summary().items() if v and k not in ("notes", "extracted_archives")
        ))
        for n in found.notes:
            print(f"[discover] note: {n}")
    return found


def copy_previous_outputs(found: DataPaths, work_dir: Path,
                          subdirs: tuple[str, ...] = ("shared", "results", "manifests", "cache", "models",
                                                      "checkpoints", "augmented")) -> list[str]:
    """Copy outputs of earlier notebook versions into ``work_dir`` (chains Kaggle sessions).

    Files that already exist with the same size are skipped, so this is cheap to repeat.
    """
    copied: list[str] = []
    for prev in found.previous_outputs:
        for sub in subdirs:
            src = prev / sub
            if not src.exists():
                continue
            for path in src.rglob("*"):
                if path.is_dir():
                    continue
                rel = path.relative_to(prev)
                dst = work_dir / rel
                if dst.exists() and dst.stat().st_size == path.stat().st_size:
                    continue
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst)
            copied.append(f"{prev}/{sub}")
    marker = work_dir / OUTPUT_MARKER
    if not marker.exists():
        work_dir.mkdir(parents=True, exist_ok=True)
        marker.write_text("yoruba_asr work directory\n", encoding="utf-8")
    return copied


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/__init__.py
"""Dataset loaders. Every loader returns a DataFrame with at least:
``path, sentence, speaker, corpus, split, gender, age``."""

STANDARD_COLUMNS = ["path", "sentence", "speaker", "corpus", "split", "gender", "age"]


def ensure_columns(df):
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = "" if col not in ("speaker",) else "unknown"
    df["speaker"] = df["speaker"].fillna("unknown").astype(str).replace("", "unknown")
    for col in ("gender", "age"):
        df[col] = df[col].fillna("").astype(str)
    return df


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/audit.py
"""Corpus audit (Research Question 1): clip counts, hours, speakers, durations, tone coverage."""
from __future__ import annotations

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from ..text import char_inventory, describe_char, has_subdot, has_tone_mark
from ..utils import save_json
from .common_voice import REFERENCE_COUNTS


def _stats(group: pd.DataFrame) -> dict:
    kept = group[group["drop_reason"].isna()] if "drop_reason" in group else group
    dur = kept["duration"].dropna().astype(float) if "duration" in kept else pd.Series(dtype=float)
    texts = kept["sentence"].astype(str).tolist() if "sentence" in kept else []
    reasons = Counter(group["drop_reason"].dropna()) if "drop_reason" in group else Counter()
    sr = Counter(int(x) for x in kept["sample_rate"].dropna()) if "sample_rate" in kept else Counter()
    return {
        "clips_found": int(len(group)),
        "clips_kept": int(len(kept)),
        "hours": round(float(dur.sum()) / 3600, 3) if len(dur) else 0.0,
        "speakers": int(kept["speaker"].nunique()) if "speaker" in kept else 0,
        "sample_rates": {str(k): v for k, v in sorted(sr.items())},
        "duration_mean_s": round(float(dur.mean()), 2) if len(dur) else None,
        "duration_min_s": round(float(dur.min()), 2) if len(dur) else None,
        "duration_max_s": round(float(dur.max()), 2) if len(dur) else None,
        "filtered": dict(reasons),
        "share_with_tone_mark": round(float(np.mean([has_tone_mark(t) for t in texts])), 4) if texts else None,
        "share_with_subdot": round(float(np.mean([has_subdot(t) for t in texts])), 4) if texts else None,
        "share_toneless": round(float(np.mean([not has_tone_mark(t) for t in texts])), 4) if texts else None,
    }


def build_audit(rows: pd.DataFrame, extra: dict | None = None) -> dict:
    """``rows`` holds every loaded item with ``corpus, split, sentence, speaker, duration,
    sample_rate, drop_reason`` (``drop_reason`` is NaN when the item is kept)."""
    audit: dict = {"corpora": {}, "extra": extra or {}}
    for (corpus, split), g in rows.groupby(["corpus", "split"], sort=True):
        audit["corpora"].setdefault(corpus, {})[split] = _stats(g)
    for corpus, g in rows.groupby("corpus"):
        kept = g[g["drop_reason"].isna()] if "drop_reason" in g else g
        inv = char_inventory(kept["sentence"].astype(str)) if "sentence" in kept else {}
        audit["corpora"][corpus]["_character_inventory"] = {
            ch: {"count": n, "name": describe_char(ch)} for ch, n in inv.items()}
    cv = audit["corpora"].get("common_voice", {})
    if not cv:
        return audit
    audit["common_voice_reference_check"] = {
        split: {"expected": exp, "found": cv.get(split, {}).get("clips_found"),
                "matches": cv.get(split, {}).get("clips_found") == exp}
        for split, exp in REFERENCE_COUNTS.items() if split in cv or split in ("train", "dev", "test")
    }
    return audit


def audit_markdown(audit: dict) -> str:
    lines = ["# Corpus audit", "", "| corpus | split | found | kept | hours | speakers | mean dur (s) | "
             "min | max | tone-marked | sub-dotted | filtered |", "|" + "---|" * 12]
    for corpus, splits in audit["corpora"].items():
        for split, s in splits.items():
            if split.startswith("_"):
                continue
            filt = ", ".join(f"{k}: {v}" for k, v in s["filtered"].items()) or "-"
            fmt = lambda x: "-" if x is None else (f"{x:.1%}" if isinstance(x, float) and x <= 1 else str(x))  # noqa: E731
            lines.append(f"| {corpus} | {split} | {s['clips_found']} | {s['clips_kept']} | {s['hours']:.2f} | "
                         f"{s['speakers']} | {s['duration_mean_s']} | {s['duration_min_s']} | {s['duration_max_s']} | "
                         f"{fmt(s['share_with_tone_mark'])} | {fmt(s['share_with_subdot'])} | {filt} |")
    lines += ["", "## Common Voice 27.0 reference counts", "", "| split | expected | found | match |",
              "|---|---|---|---|"]
    for split, r in audit.get("common_voice_reference_check", {}).items():
        lines.append(f"| {split} | {r['expected']} | {r['found']} | {'yes' if r['matches'] else '**NO**'} |")
    extra = audit.get("extra", {})
    if extra:
        lines += ["", "## Notes", ""]
        for k, v in extra.items():
            lines.append(f"- **{k}**: {v}")
    for corpus, splits in audit["corpora"].items():
        inv = splits.get("_character_inventory", {})
        if inv:
            lines += ["", f"## Character inventory - {corpus}", "",
                      " ".join(f"`{c}`({v['count']})" for c, v in inv.items() if c != " ")]
    return "\n".join(lines) + "\n"


def write_audit(audit: dict, results_dir: str | Path) -> None:
    results_dir = Path(results_dir)
    save_json(audit, results_dir / "corpus_audit.json")
    (results_dir / "corpus_audit.md").write_text(audit_markdown(audit), encoding="utf-8")


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/common_voice.py
"""Common Voice Scripted Speech 27.0 - Yoruba (Mozilla Data Collective release)."""
from __future__ import annotations

import csv
from pathlib import Path

import pandas as pd

from . import ensure_columns

# Official reference statistics for release 27.0 (2026-09-11), used only for the audit comparison.
REFERENCE_COUNTS = {"train": 1468, "dev": 983, "test": 1096, "other": 1152, "invalidated": 269}
SPLIT_FILES = {"train": "train.tsv", "dev": "dev.tsv", "test": "test.tsv", "other": "other.tsv",
               "validated": "validated.tsv", "invalidated": "invalidated.tsv"}


def _gender(g: str) -> str:
    g = (g or "").lower()
    if g.startswith("female"):
        return "female"
    if g.startswith("male"):
        return "male"
    return g


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False,
                       encoding="utf-8")


def load_split(cv_dir: str | Path, split: str) -> pd.DataFrame:
    """Load one Common Voice split. ``split`` is train/dev/test/other/validated/invalidated."""
    cv_dir = Path(cv_dir)
    tsv = cv_dir / SPLIT_FILES[split]
    if not tsv.exists():
        return ensure_columns(pd.DataFrame(columns=["path", "sentence"]))
    df = read_tsv(tsv)
    clips = cv_dir / "clips"
    out = pd.DataFrame({
        "path": [str(clips / p) for p in df["path"]],
        "sentence": df.get("sentence", ""),
        "speaker": df.get("client_id", "unknown"),
        "corpus": "common_voice",
        "split": split,
        "gender": [_gender(g) for g in df.get("gender", [""] * len(df))],
        "age": df.get("age", ""),
    })
    out = out[out["sentence"].astype(str).str.strip() != ""]
    return ensure_columns(out.reset_index(drop=True))


def index_audio_files(roots, exts=(".mp3", ".wav", ".flac", ".ogg")) -> dict[str, str]:
    """File name -> full path for every audio file under ``roots`` (first occurrence wins)."""
    import os

    index: dict[str, str] = {}
    for root in roots:
        if not root or not Path(root).exists():
            continue
        for dirpath, _dirs, files in os.walk(root, followlinks=True):
            for f in files:
                if f.lower().endswith(exts) and f not in index:
                    index[f] = os.path.join(dirpath, f)
    return index


def resolve_clip_paths(df: pd.DataFrame, search_roots) -> tuple[pd.DataFrame, dict]:
    """Point ``path`` at the real clip when it is not in ``<cv>/clips/`` (Kaggle may unpack uploads into
    other folders). Returns the updated frame and a report {found_in_place, remapped, missing}."""
    exists = df["path"].map(lambda p: Path(p).exists())
    report = {"found_in_place": int(exists.sum()), "remapped": 0, "missing": 0}
    if exists.all():
        return df, report
    index = index_audio_files(search_roots)
    df = df.copy()
    names = df.loc[~exists, "path"].map(lambda p: Path(p).name)
    hits = names.map(index.get)
    df.loc[hits.index[hits.notna()], "path"] = hits[hits.notna()]
    report["remapped"] = int(hits.notna().sum())
    report["missing"] = int(hits.isna().sum())
    report["audio_files_found_under_input"] = len(index)
    return df, report


def dataset_tree_report(root, max_lines: int = 40) -> str:
    """Short listing of folders under ``root`` with file counts (for error messages)."""
    import os

    lines = []
    for dirpath, _dirs, files in os.walk(root, followlinks=True):
        if not files:
            continue
        audio = sum(f.lower().endswith((".mp3", ".wav", ".flac")) for f in files)
        other = [f for f in files if not f.lower().endswith((".mp3", ".wav", ".flac"))][:6]
        lines.append(f"  {dirpath}  | files: {len(files)} | audio: {audio} | e.g. {other}")
        if len(lines) >= max_lines:
            lines.append("  ...")
            break
    return "\n".join(lines) or f"  (nothing found under {root})"


def load_common_voice(cv_dir: str | Path, splits=("train", "dev", "test", "other")) -> pd.DataFrame:
    return pd.concat([load_split(cv_dir, s) for s in splits], ignore_index=True)


def load_sentence_pool(cv_dir: str | Path, search_roots=None) -> pd.DataFrame:
    """Text-only sentences for TTS: ``validated_sentences.tsv`` from the CV folder, or found anywhere
    under ``search_roots`` (Kaggle may unpack files into other folders). Falls back to the sentences of
    ``validated.tsv``. The ``pool_source`` attribute of the result records which file was used."""
    import os

    candidates = [Path(cv_dir) / "validated_sentences.tsv"]
    for root in search_roots or []:
        if root and Path(root).exists():
            for dirpath, _d, files in os.walk(root, followlinks=True):
                if "validated_sentences.tsv" in files:
                    candidates.append(Path(dirpath) / "validated_sentences.tsv")
    for p in candidates:
        if p.exists():
            df = read_tsv(p)
            if "sentence" in df.columns and len(df):
                keep = [c for c in ("sentence", "is_used", "clips_count", "source") if c in df.columns]
                out = df[keep].copy()
                out.attrs["pool_source"] = str(p)
                return out
    fallback = Path(cv_dir) / "validated.tsv"
    if fallback.exists():
        df = read_tsv(fallback)
        if "sentence" in df.columns:
            out = df[["sentence"]].drop_duplicates().copy()
            out.attrs["pool_source"] = f"{fallback} (fallback: validated_sentences.tsv not found)"
            return out
    out = pd.DataFrame(columns=["sentence", "is_used", "clips_count"])
    out.attrs["pool_source"] = "none found"
    return out


def clip_durations(cv_dir: str | Path) -> dict[str, float]:
    """Durations (s) from ``clip_durations.tsv`` when present (file name -> seconds)."""
    p = Path(cv_dir) / "clip_durations.tsv"
    if not p.exists():
        return {}
    df = read_tsv(p)
    col = [c for c in df.columns if c.startswith("duration")]
    if not col:
        return {}
    return {r["clip"]: float(r[col[0]]) / 1000.0 for _, r in df.iterrows()}


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/download.py
"""Automatic download of the Google Yoruba speech corpus (OpenSLR 86, CC BY-SA 4.0).

No Kaggle upload is needed: the notebook downloads ~0.9 GB from OpenSLR mirrors (Internet must be On).
Files (verified on the OpenSLR 86 page, 2026-10-20): yo_ng_female.zip (462 MB), yo_ng_male.zip (445 MB),
each a flat archive with line_index.tsv + 48 kHz mono WAVs.
"""
from __future__ import annotations

import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

from ..utils import get_logger

log = get_logger(__name__)
MIRRORS = ["https://openslr.elda.org/resources/86", "https://openslr.trmal.net/resources/86",
           "https://openslr.magicdatatech.com/resources/86", "https://www.openslr.org/resources/86"]
ARCHIVES = ["yo_ng_female.zip", "yo_ng_male.zip"]


def _fetch(name: str, dest: Path, retries: int = 3) -> None:
    tmp = dest.with_suffix(dest.suffix + ".part")
    last: Exception | None = None
    for mirror in MIRRORS:
        for attempt in range(retries):
            try:
                log.info("downloading %s/%s", mirror, name)
                with urllib.request.urlopen(f"{mirror}/{name}", timeout=60) as r, open(tmp, "wb") as fh:
                    shutil.copyfileobj(r, fh, length=1 << 20)
                tmp.replace(dest)
                return
            except Exception as exc:  # try again, then the next mirror
                last = exc
                time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Could not download {name} from any OpenSLR mirror ({last}). "
                       "Check that Internet is On in the Kaggle notebook settings.")


def download_openslr86(dest_root: str | Path) -> list[Path]:
    """Download + extract both archives into ``dest_root/yo_ng_{female,male}/`` (once). Returns the folders."""
    dest_root = Path(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)
    folders = []
    for name in ARCHIVES:
        folder = dest_root / name.replace(".zip", "")
        if (folder / ".done").exists():
            folders.append(folder)
            continue
        archive = dest_root / name
        if not archive.exists():
            _fetch(name, archive)
        folder.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                if not str((folder / member).resolve()).startswith(str(folder.resolve())):
                    raise RuntimeError(f"unsafe path in {name}: {member}")
            zf.extractall(folder)
        archive.unlink()  # keep disk usage low
        (folder / ".done").write_text("ok", encoding="utf-8")
        folders.append(folder)
        log.info("OpenSLR 86: %s ready (%d files)", folder.name, sum(1 for _ in folder.iterdir()))
    return folders


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/generic.py
"""Generic manifest loader: a CSV/TSV with an audio-path column and a transcript column.

Accepted column names (case-insensitive):
  * audio:      path, audio, audio_path, file, filename, wav, audio_filepath
  * transcript: sentence, text, transcript, transcription, normalized_text
  * speaker:    speaker, speaker_id, client_id, spk (optional)
Relative paths are resolved against the manifest's folder (then searched recursively by name).
"""
from __future__ import annotations

import csv
import json
from pathlib import Path

import pandas as pd

from . import ensure_columns

AUDIO_COLS = ("path", "audio", "audio_path", "file", "filename", "wav", "audio_filepath", "file_name")
TEXT_COLS = ("sentence", "text", "transcript", "transcription", "normalized_text")
SPEAKER_COLS = ("speaker", "speaker_id", "client_id", "spk", "user_id")


class ManifestError(ValueError):
    pass


MANIFEST_HELP = (
    "Create a manifest CSV with the header 'path,sentence,speaker' (speaker optional), one row per "
    "utterance, where 'path' is the audio file path (absolute or relative to the CSV) and 'sentence' "
    "is the Yoruba transcript with tone marks. Upload it with the audio and set "
    "data.<corpus>.manifest to its path (or place it in the corpus folder)."
)


def _read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in (".jsonl", ".json"):
        rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        return pd.DataFrame(rows).astype(str)
    sep = "\t" if path.suffix.lower() == ".tsv" else ","
    try:
        return pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False, encoding="utf-8")
    except Exception:
        return pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False, encoding="utf-8",
                           quoting=csv.QUOTE_NONE)


def _pick(columns, options) -> str | None:
    lower = {c.lower().strip(): c for c in columns}
    for o in options:
        if o in lower:
            return lower[o]
    return None


def looks_like_manifest(path: Path) -> bool:
    try:
        df = _read_table(path).head(2)
    except Exception:
        return False
    return _pick(df.columns, AUDIO_COLS) is not None and _pick(df.columns, TEXT_COLS) is not None


def load_manifest(manifest: str | Path, corpus: str, split: str = "ood_train",
                  audio_root: str | Path | None = None) -> pd.DataFrame:
    manifest = Path(manifest)
    if not manifest.exists():
        raise ManifestError(f"Manifest not found: {manifest}. {MANIFEST_HELP}")
    df = _read_table(manifest)
    a_col, t_col = _pick(df.columns, AUDIO_COLS), _pick(df.columns, TEXT_COLS)
    if a_col is None or t_col is None:
        raise ManifestError(f"{manifest} has columns {list(df.columns)}; need an audio column {AUDIO_COLS} "
                            f"and a transcript column {TEXT_COLS}. {MANIFEST_HELP}")
    s_col = _pick(df.columns, SPEAKER_COLS)
    root = Path(audio_root) if audio_root else manifest.parent
    by_name: dict[str, Path] | None = None

    def resolve(p: str) -> str:
        nonlocal by_name
        cand = Path(p)
        if cand.is_absolute() and cand.exists():
            return str(cand)
        for base in (root, manifest.parent):
            if (base / p).exists():
                return str(base / p)
        if by_name is None:
            by_name = {}
            for f in root.rglob("*"):
                if f.suffix.lower() in (".wav", ".flac", ".mp3", ".ogg", ".m4a", ".opus"):
                    by_name.setdefault(f.name, f)
        hit = by_name.get(Path(p).name)
        return str(hit) if hit else str(root / p)

    out = pd.DataFrame({
        "path": [resolve(p) for p in df[a_col]],
        "sentence": df[t_col],
        "speaker": df[s_col] if s_col else "unknown",
        "corpus": corpus,
        "split": split,
    })
    out = out[out["sentence"].astype(str).str.strip() != ""]
    return ensure_columns(out.reset_index(drop=True))


def find_manifest(folder: str | Path) -> Path | None:
    """Auto-detect a manifest table inside ``folder`` (largest matching CSV/TSV/JSONL)."""
    folder = Path(folder)
    cands = [p for ext in ("*.csv", "*.tsv", "*.jsonl") for p in folder.rglob(ext) if looks_like_manifest(p)]
    if not cands:
        return None
    return max(cands, key=lambda p: p.stat().st_size)


def list_audio(folder: str | Path) -> list[str]:
    folder = Path(folder)
    return sorted(str(p) for p in folder.rglob("*")
                  if p.suffix.lower() in (".wav", ".flac", ".mp3", ".ogg", ".m4a", ".opus"))


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/iroyin.py
"""ÌròyìnSpeech (Ògúnrẹ̀mí et al., 2024) loader, built on the generic manifest loader.

The exact release layout is not fixed in the project specification, so the loader:
1. uses ``data.iroyin.manifest`` when given;
2. otherwise auto-detects a CSV/TSV/JSONL in the ÌròyìnSpeech folder that has an audio-path column
   and a transcript column;
3. otherwise raises :class:`ManifestError` explaining how to create the manifest.
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

from .generic import MANIFEST_HELP, ManifestError, find_manifest, load_manifest


def load_iroyin(folder: str | Path | None = None, manifest: str | Path | None = None) -> pd.DataFrame:
    if manifest is None:
        if folder is None:
            raise ManifestError("ÌròyìnSpeech: no folder or manifest given. " + MANIFEST_HELP)
        manifest = find_manifest(folder)
        if manifest is None:
            raise ManifestError(
                f"ÌròyìnSpeech: no manifest with audio/transcript columns found under {folder}. " + MANIFEST_HELP)
    return load_manifest(manifest, corpus="iroyin", split="ood_train", audio_root=folder)


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/leakage.py
"""Leakage control: keep Common Voice dev/test out of every training and unlabelled pool."""
from __future__ import annotations

from pathlib import Path

import pandas as pd

from ..text import normalize_text, strip_tones


class LeakageError(AssertionError):
    pass


def leakage_key(sentence: str) -> str:
    """Normalised, tone-stripped form used to compare sentences across corpora."""
    return strip_tones(normalize_text(sentence, keep_tones=True))


def forbidden_sentences(cv_dev: pd.DataFrame, cv_test: pd.DataFrame) -> set[str]:
    return {leakage_key(s) for s in pd.concat([cv_dev["sentence"], cv_test["sentence"]])}


def remove_forbidden(df: pd.DataFrame, forbidden: set[str], column: str = "sentence") -> tuple[pd.DataFrame, int]:
    """Drop rows whose tone-stripped normalised sentence is in ``forbidden``."""
    if df.empty:
        return df, 0
    mask = df[column].map(leakage_key).isin(forbidden)
    return df[~mask].reset_index(drop=True), int(mask.sum())


def _norm_path(p: str) -> str:
    return str(Path(p).name).lower() if p else ""


def assert_no_eval_audio(pools: dict[str, pd.DataFrame], cv_dev: pd.DataFrame, cv_test: pd.DataFrame) -> dict:
    """Raise :class:`LeakageError` if a CV dev/test clip appears in a training/unlabelled pool."""
    eval_paths = set(map(str, cv_dev["path"])) | set(map(str, cv_test["path"]))
    eval_names = {_norm_path(p) for p in eval_paths}
    report = {}
    for name, df in pools.items():
        if df is None or df.empty:
            report[name] = 0
            continue
        paths = df["path"].astype(str)
        hits = paths.isin(eval_paths) | (df.get("corpus", "") == "common_voice") & paths.map(_norm_path).isin(eval_names)
        n = int(hits.sum())
        report[name] = n
        if n:
            raise LeakageError(f"{n} CV dev/test clips found in pool '{name}', e.g. {paths[hits].iloc[0]}")
    return report


def speaker_overlap(train: pd.DataFrame, dev: pd.DataFrame, test: pd.DataFrame) -> dict:
    tr, dv, te = set(train["speaker"]), set(dev["speaker"]), set(test["speaker"])
    tr.discard("unknown"); dv.discard("unknown"); te.discard("unknown")
    return {"train_dev": len(tr & dv), "train_test": len(tr & te), "dev_test": len(dv & te)}


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/data/slr86.py
"""Google crowdsourced Yoruba speech corpus (OpenSLR 86, CC BY-SA 4.0).

Each archive (``yo_ng_female`` / ``yo_ng_male``) has a ``line_index.tsv`` mapping a file id to a
transcript, plus ``<id>.wav`` files. Lines are ``<id>\\t<transcript>``; some releases use a
space instead of a tab, so both are accepted. The speaker is the id prefix (``yof_00295``).
"""
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from . import ensure_columns


_TAG = re.compile(r"\[[^\]]*\]")
# Recordings whose audio stops mid-sentence: the transcript does not match the audio, so they are dropped.
DROP_TAGS = {"[abrupt]"}


def clean_annotations(sentence: str) -> tuple[str, list[str]]:
    """Remove OpenSLR annotation tags ([breath], [snap], [external], [hesitation], [abrupt])."""
    tags = [t.lower() for t in _TAG.findall(sentence)]
    return " ".join(_TAG.sub(" ", sentence).split()), tags


def _index_audio(folder: Path) -> dict[str, Path]:
    return {p.stem: p for ext in ("*.wav", "*.flac") for p in folder.rglob(ext)}


def speaker_split(df: pd.DataFrame, dev_share: float = 0.1, test_share: float = 0.1, seed: int = 42,
                  key=None) -> dict[str, pd.DataFrame]:
    """Speaker-disjoint train/dev/test split (done per gender so both are represented in every split).

    Speakers are shuffled with ``seed`` and assigned to test, then dev, until each holds about its share of
    utterances. Training utterances whose sentence (``key``) also occurs in dev/test are removed, so the
    evaluation sentences are unseen as well.
    """
    import numpy as np

    rng = np.random.default_rng(seed)
    parts = {"train": [], "dev": [], "test": []}
    for _gender, g in df.groupby(df["gender"].fillna(""), sort=True):
        counts = g.groupby("speaker").size()
        speakers = list(counts.index[rng.permutation(len(counts))])
        total = len(g)
        assigned = {"test": [], "dev": []}
        for split, share in (("test", test_share), ("dev", dev_share)):
            n = 0
            while len(speakers) > 1 and n < share * total:  # always keep >= 1 speaker for training
                s = speakers.pop(0)
                assigned[split].append(s)
                n += counts[s]
        parts["test"].append(g[g["speaker"].isin(assigned["test"])])
        parts["dev"].append(g[g["speaker"].isin(assigned["dev"])])
        parts["train"].append(g[g["speaker"].isin(speakers)])
    out = {k: pd.concat(v, ignore_index=True) if v else df.iloc[:0] for k, v in parts.items()}
    if key is not None:
        held_out = set(out["dev"]["sentence"].map(key)) | set(out["test"]["sentence"].map(key))
        out["train"] = out["train"][~out["train"]["sentence"].map(key).isin(held_out)].reset_index(drop=True)
    for k in out:
        out[k] = out[k].assign(split=k).reset_index(drop=True)
    return out


def load_slr86(folders: list[str | Path]) -> pd.DataFrame:
    rows = []
    for folder in folders:
        folder = Path(folder)
        index = folder / "line_index.tsv"
        if not index.exists():
            continue
        audio = _index_audio(folder)
        gender = "female" if "female" in folder.name.lower() else ("male" if "male" in folder.name.lower() else "")
        with open(index, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("\t", 1) if "\t" in line else re.split(r"\s+", line, maxsplit=1)
                if len(parts) != 2:
                    continue
                fid, sentence = parts[0].strip().lstrip("﻿"), parts[1].strip()
                sentence, tags = clean_annotations(sentence)
                if not sentence or DROP_TAGS & set(tags):
                    continue
                path = audio.get(fid)
                if path is None:
                    continue
                m = re.match(r"^(yo[fm]_\d+)", fid)
                if gender == "" and fid.startswith(("yof", "yom")):
                    g = "female" if fid.startswith("yof") else "male"
                else:
                    g = gender
                rows.append({"path": str(path), "sentence": sentence,
                             "speaker": m.group(1) if m else "unknown", "corpus": "slr86",
                             "split": "ood_train", "gender": g, "age": ""})
    return ensure_columns(pd.DataFrame(rows, columns=["path", "sentence", "speaker", "corpus", "split",
                                                      "gender", "age"]))


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/__init__.py
"""Evaluation: metrics, decoding, significance testing, factorial effects, ablation, error analysis."""


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/ablation.py
"""Leave-one-out ablation of V2 (Section 15.5).

Each run differs from ``v2_ref`` in exactly one respect and uses the same (short) step budget.
Runs are resumable and can be selected one at a time (``--only no_tts``).
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

ABLATION_COLUMNS = ["run", "description", "transforms", "use_tts", "mask_time_prob", "max_steps", "train_clips",
                    "dev_wer", "dev_ter", "test_wer", "test_ter", "test_cer", "test_wer_no_tone", "status"]

DESCRIPTIONS = {
    "v2_ref": "full V2 recipe (reference)",
    "no_tempo": "without tempo perturbation",
    "no_noise": "without additive noise",
    "no_reverb": "without reverberation",
    "no_tts": "without TTS synthetic speech",
    "no_tuned_mask": "mask_time_prob = 0.05 instead of the tuned value",
    "plus_speed": "+ pitch-changing speed perturbation",
}


def resolve_run(name: str, spec: dict, base_transforms: list[str], tuned_mask: float) -> dict:
    spec = dict(spec or {})
    transforms = list(base_transforms)
    if spec.get("remove_transform"):
        transforms = [t for t in transforms if t != spec["remove_transform"]]
    if spec.get("add_transform") and spec["add_transform"] not in transforms:
        transforms.append(spec["add_transform"])
    return {
        "run": name, "description": DESCRIPTIONS.get(name, name), "transforms": transforms,
        "use_tts": bool(spec.get("use_tts", True)),
        "mask_time_prob": float(spec.get("mask_time_prob", tuned_mask)),
        # Runs whose transform set equals the main set reuse the main augmented copies.
        "aug_set": "main" if sorted(transforms) == sorted(base_transforms) else "abl_" + name,
    }


def update_table(path: Path, row: dict) -> pd.DataFrame:
    df = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=ABLATION_COLUMNS)
    df = df[df["run"] != row["run"]]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    order = {k: i for i, k in enumerate(DESCRIPTIONS)}
    df = df.sort_values("run", key=lambda s: s.map(lambda x: order.get(x, 99)))
    df.to_csv(path, index=False)
    return df


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/bootstrap.py
"""Blockwise (speaker) bootstrap significance testing (Liu & Peng, 2020).

Speakers are resampled with replacement; for each resample the corpus-level error rate of each
system is recomputed over all utterances of the sampled speakers, and the difference is taken.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .metrics import utterance_stats

METRIC_COLUMNS = {
    "wer": ("word_errors", "ref_words"),
    "cer": ("char_errors", "ref_chars"),
    "wer_no_tone": ("word_errors_notone", "ref_words"),
    "ter": ("tone_errors", "ref_tbus"),
}


def per_utterance(preds: pd.DataFrame) -> pd.DataFrame:
    stats = [utterance_stats(str(r), str(h) if isinstance(h, str) else "")
             for r, h in zip(preds["reference"], preds["hypothesis"])]
    out = pd.DataFrame(stats)
    out["speaker"] = preds["speaker"].astype(str).values
    out["path"] = preds["path"].astype(str).values
    return out


def _speaker_sums(u: pd.DataFrame, num: str, den: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    g = u.groupby("speaker")[[num, den]].sum()
    return g[num].to_numpy(float), g[den].to_numpy(float), list(g.index)


def system_ci(preds: pd.DataFrame, metric: str = "wer", n: int = 1000, ci: float = 0.95, seed: int = 42) -> dict:
    """Point estimate and percentile CI (percent) of one system's metric."""
    u = per_utterance(preds)
    num, den = METRIC_COLUMNS[metric]
    e, d, _ = _speaker_sums(u, num, den)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(e), size=(n, len(e)))
    rates = 100 * e[idx].sum(1) / np.maximum(d[idx].sum(1), 1e-9)
    lo, hi = np.percentile(rates, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return {"metric": metric, "point": 100 * e.sum() / max(d.sum(), 1e-9), "ci_low": float(lo), "ci_high": float(hi)}


def paired_bootstrap(preds_a: pd.DataFrame, preds_b: pd.DataFrame, metric: str = "wer", n: int = 1000,
                     ci: float = 0.95, seed: int = 42) -> dict:
    """Delta = metric(A) - metric(B) in percentage points with a speaker-block bootstrap CI."""
    a = per_utterance(preds_a).set_index("path")
    b = per_utterance(preds_b).set_index("path")
    common = a.index.intersection(b.index)
    if len(common) == 0:
        raise ValueError("The two prediction sets share no utterances")
    a, b = a.loc[common].reset_index(), b.loc[common].reset_index()
    num, den = METRIC_COLUMNS[metric]
    ea, d, speakers = _speaker_sums(a, num, den)
    b = b.assign(speaker=a["speaker"].values)
    eb, _, _ = _speaker_sums(b, num, den)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(speakers), size=(n, len(speakers)))
    den_s = np.maximum(d[idx].sum(1), 1e-9)
    deltas = 100 * ea[idx].sum(1) / den_s - 100 * eb[idx].sum(1) / den_s
    lo, hi = np.percentile(deltas, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    point_a, point_b = 100 * ea.sum() / max(d.sum(), 1e-9), 100 * eb.sum() / max(d.sum(), 1e-9)
    return {
        "metric": metric, "value_a": point_a, "value_b": point_b, "delta": point_a - point_b,
        "delta_mean": float(deltas.mean()), "ci_low": float(lo), "ci_high": float(hi),
        "significant": bool(lo > 0 or hi < 0), "n_speakers": len(speakers), "n_utts": int(len(common)),
        "resamples": n,
    }


def significance_table(predictions: dict[str, pd.DataFrame], pairs: list[tuple[str, str]],
                       n: int = 1000, ci: float = 0.95, seed: int = 42,
                       metrics: tuple[str, ...] = ("wer", "ter")) -> pd.DataFrame:
    rows = []
    for a, b in pairs:
        if a not in predictions or b not in predictions:
            continue
        for m in metrics:
            r = paired_bootstrap(predictions[a], predictions[b], m, n, ci, seed)
            rows.append({"system_a": a, "system_b": b, "comparison": f"{a}-{b}", **r})
    return pd.DataFrame(rows)


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/decode.py
"""Batched greedy CTC decoding of a test set and writing ``predictions.csv`` + ``metrics.json``."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ..text import nfc, strip_tones
from ..utils import get_logger, save_json
from .metrics import all_metrics

log = get_logger(__name__)


def load_model_and_processor(model_dir: str | Path, device: str | None = None):
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    processor = Wav2Vec2Processor.from_pretrained(str(model_dir))
    model = Wav2Vec2ForCTC.from_pretrained(str(model_dir)).to(device).eval()
    return model, processor, device


@torch.no_grad()
def greedy_decode_arrays(model, processor, arrays: list[np.ndarray], device: str, batch_size: int = 8,
                         normalized: bool = True) -> list[str]:
    """Decode a list of 16 kHz arrays. ``normalized=True`` means they are already feature-normalised."""
    order = np.argsort([-len(a) for a in arrays])
    out: list[str] = [""] * len(arrays)
    for start in range(0, len(arrays), batch_size):
        idx = order[start: start + batch_size]
        batch = [arrays[i] for i in idx]
        if normalized:
            feats = processor.feature_extractor.pad([{"input_values": a} for a in batch], padding=True,
                                                    return_tensors="pt")
        else:
            feats = processor(batch, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs = feats["input_values"].to(device)
        mask = feats.get("attention_mask")
        logits = model(inputs, attention_mask=mask.to(device) if mask is not None else None).logits
        ids = torch.argmax(logits, dim=-1).cpu().numpy()
        texts = processor.batch_decode(ids)
        for i, t in zip(idx, texts):
            out[i] = nfc(t).strip()
    return out


def decode_cached(model, processor, hf_dataset, device: str, batch_size: int = 8) -> pd.DataFrame:
    """Decode a cached HF split (input_values + metadata) -> predictions DataFrame."""
    from ..train.datasets import arrow_row

    arrays = [arrow_row(hf_dataset, i, ("input_values",))["input_values"] for i in range(len(hf_dataset))]
    hyps = greedy_decode_arrays(model, processor, arrays, device, batch_size, normalized=True)
    meta = hf_dataset.select_columns([c for c in ("path", "speaker", "gender", "age", "text_tone", "duration")
                                      if c in hf_dataset.column_names]).to_pandas()
    return pd.DataFrame({
        "path": meta["path"], "speaker": meta["speaker"],
        "gender": meta.get("gender", ""), "age": meta.get("age", ""), "duration": meta.get("duration", np.nan),
        "reference": meta["text_tone"], "hypothesis": hyps,
        "reference_notone": [strip_tones(r) for r in meta["text_tone"]],
        "hypothesis_notone": [strip_tones(h) for h in hyps],
    })


def evaluate_predictions(preds: pd.DataFrame, variant: str, out_dir: str | Path, extra: dict | None = None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    preds.to_csv(out_dir / "predictions.csv", index=False, encoding="utf-8")
    m = all_metrics(preds["reference"].tolist(), preds["hypothesis"].fillna("").tolist())
    metrics = {"variant": variant, "n_utts": int(len(preds)), **{k: (round(v, 4) if isinstance(v, float) else v)
                                                               for k, v in m.items()}, **(extra or {})}
    save_json(metrics, out_dir / "metrics.json")
    log.info("[%s] test: WER=%.2f CER=%.2f WER(no tone)=%.2f TER=%.2f (n=%d)", variant, m["wer"], m["cer"],
             m["wer_no_tone"], m["ter"], len(preds))
    return metrics


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/effects.py
"""2 x 2 factorial effects from the four variants.

* Augmentation main effect = mean[(V2 - V1), (V4 - V3)]
* Adaptation main effect   = mean[(V3 - V1), (V4 - V2)]
* Interaction              = (V4 - V3) - (V2 - V1)
Negative values mean the treatment lowers the error rate.
"""
from __future__ import annotations

import pandas as pd

DEFAULT_METRICS = ("wer", "cer", "wer_no_tone", "ter")


def factorial_effects(v1: float, v2: float, v3: float, v4: float) -> dict:
    return {
        "augmentation_main_effect": ((v2 - v1) + (v4 - v3)) / 2,
        "adaptation_main_effect": ((v3 - v1) + (v4 - v2)) / 2,
        "interaction": (v4 - v3) - (v2 - v1),
    }


def effects_table(metrics: dict[str, dict], keys=DEFAULT_METRICS) -> pd.DataFrame:
    rows = []
    if not all(v in metrics for v in ("V1", "V2", "V3", "V4")):
        missing = [v for v in ("V1", "V2", "V3", "V4") if v not in metrics]
        raise ValueError(f"Factorial effects need all four variants; missing {missing}")
    for k in keys:
        vals = [float(metrics[v][k]) for v in ("V1", "V2", "V3", "V4")]
        rows.append({"metric": k, "V1": vals[0], "V2": vals[1], "V3": vals[2], "V4": vals[3],
                     **factorial_effects(*vals)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/error_analysis.py
"""Error analysis: tonal error share, H/M/L confusion, breakdowns, qualitative examples and figures.

Figures are 300 dpi PNGs drawn in greyscale with hatching/markers so they stay readable in print.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from ..text import TONE_LABELS, has_subdot
from ..utils import get_logger, load_json, save_json
from .metrics import all_metrics, word_alignment

log = get_logger(__name__)
HATCHES = ["", "///", "...", "xxx", "\\\\\\", "++"]
GREYS = ["0.15", "0.45", "0.7", "0.9", "0.3", "0.6"]


def _plt():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
    return plt


def length_bucket(d: float) -> str:
    if not np.isfinite(d):
        return "unknown"
    if d < 3:
        return "<3 s"
    if d < 6:
        return "3-6 s"
    if d < 10:
        return "6-10 s"
    return ">10 s"


def breakdown(preds: pd.DataFrame, by: str) -> pd.DataFrame:
    rows = []
    for key, g in preds.groupby(by, dropna=False):
        if len(g) == 0:
            continue
        m = all_metrics(g["reference"].tolist(), g["hypothesis"].fillna("").tolist())
        rows.append({by: key if key != "" else "unknown", "n_utts": len(g), "wer": m["wer"], "ter": m["ter"],
                     "cer": m["cer"], "wer_no_tone": m["wer_no_tone"]})
    return pd.DataFrame(rows)


def subdot_word_error(preds: pd.DataFrame) -> pd.DataFrame:
    """Word error rate (sub+del over reference words) for words with vs without sub-dotted letters."""
    counts = {True: [0, 0], False: [0, 0]}
    for r, h in zip(preds["reference"], preds["hypothesis"].fillna("")):
        for op, rw, _hw in word_alignment(str(r), str(h)):
            if rw is None:
                continue
            k = has_subdot(rw)
            counts[k][1] += 1
            counts[k][0] += op != "equal"
    return pd.DataFrame([{"word_type": "with sub-dot" if k else "without sub-dot", "ref_words": v[1],
                          "word_error_rate": 100 * v[0] / v[1] if v[1] else np.nan} for k, v in counts.items()])


def highlight(ref: str, hyp: str) -> tuple[str, str]:
    """Markdown: substitutions **bold**, deletions ~~struck~~ (in REF), insertions _italic_ (in HYP)."""
    r_out, h_out = [], []
    for op, rw, hw in word_alignment(ref, hyp):
        if op == "equal":
            r_out.append(rw); h_out.append(hw)
        elif op == "substitute":
            r_out.append(f"**{rw}**"); h_out.append(f"**{hw}**")
        elif op == "delete":
            r_out.append(f"~~{rw}~~")
        else:
            h_out.append(f"_{hw}_")
    return " ".join(r_out), " ".join(h_out)


def examples_markdown(preds: pd.DataFrame, variant: str, n: int = 20, seed: int = 42) -> str:
    errs = preds[preds["reference"] != preds["hypothesis"].fillna("")]
    pick = errs.sample(n=min(n, len(errs)), random_state=seed) if len(errs) else errs
    lines = [f"# {variant}: qualitative examples", "",
             "Substitutions in **bold**, deletions ~~struck~~, insertions in _italics_.", ""]
    for i, (_, row) in enumerate(pick.iterrows(), 1):
        r, h = highlight(str(row["reference"]), str(row["hypothesis"]) if isinstance(row["hypothesis"], str) else "")
        lines += [f"{i}. REF: {r}", f"   HYP: {h}", ""]
    return "\n".join(lines)


def plot_confusion(conf, title: str, path: Path) -> None:
    plt = _plt()
    conf = np.asarray(conf, dtype=float)
    norm = conf / np.maximum(conf.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(3.4, 3))
    ax.imshow(norm, cmap="Greys", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{int(conf[i, j])}\n({norm[i, j]:.0%})", ha="center", va="center", fontsize=8,
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xticks(range(3), TONE_LABELS); ax.set_yticks(range(3), TONE_LABELS)
    ax.set_xlabel("hypothesis tone"); ax.set_ylabel("reference tone"); ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_variant_bars(summary: pd.DataFrame, cis: dict, path: Path) -> None:
    """WER/CER/TER for V1-V4 with bootstrap 95% CIs."""
    plt = _plt()
    metrics = [m for m in ("wer", "cer", "ter") if m in summary.columns]
    variants = summary["variant"].tolist()
    x = np.arange(len(metrics))
    w = 0.8 / max(len(variants), 1)
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for k, v in enumerate(variants):
        vals = [float(summary.loc[summary.variant == v, m].iloc[0]) for m in metrics]
        err = None
        if v in cis:
            lo = [vals[i] - cis[v].get(m, {}).get("ci_low", vals[i]) for i, m in enumerate(metrics)]
            hi = [cis[v].get(m, {}).get("ci_high", vals[i]) - vals[i] for i, m in enumerate(metrics)]
            err = [lo, hi]
        ax.bar(x + k * w - 0.4 + w / 2, vals, w, label=v, color=GREYS[k % 6], edgecolor="black",
               hatch=HATCHES[k % 6], yerr=err, capsize=2, error_kw={"elinewidth": 0.8})
    ax.set_xticks(x, [m.upper() for m in metrics]); ax.set_ylabel("error rate (%)")
    ax.legend(fontsize=8, ncol=4); ax.set_title("Test-set error rates (95% speaker-bootstrap CI)")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_interaction(summary: pd.DataFrame, path: Path, metric: str = "wer") -> None:
    plt = _plt()
    s = summary.set_index("variant")
    if not all(v in s.index for v in ("V1", "V2", "V3", "V4")):
        return
    fig, ax = plt.subplots(figsize=(4, 3.2))
    ax.plot([0, 1], [s.loc["V1", metric], s.loc["V2", metric]], "k-o", label="no domain adaptation")
    ax.plot([0, 1], [s.loc["V3", metric], s.loc["V4", metric]], "k--s", label="with domain adaptation")
    ax.set_xticks([0, 1], ["no augmentation", "augmentation"]); ax.set_ylabel(f"{metric.upper()} (%)")
    ax.set_title("Factorial interaction"); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_ablation(abl: pd.DataFrame, path: Path) -> None:
    plt = _plt()
    if abl.empty or "test_wer" not in abl:
        return
    abl = abl.sort_values("test_wer")
    fig, ax = plt.subplots(figsize=(5, 3.2))
    colors = ["0.2" if n == "v2_ref" else "0.7" for n in abl["run"]]
    ax.barh(abl["run"], abl["test_wer"], color=colors, edgecolor="black")
    ax.set_xlabel("test WER (%)"); ax.set_title("Ablation (1,500 steps each; dark = full V2)")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_dev_curves(results_dir: Path, path: Path) -> None:
    plt = _plt()
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    styles = ["k-o", "k--s", "k-.^", "k:d", "k-x"]
    found = False
    for k, tag in enumerate(["V1", "V2", "V3", "V4", "B1"]):
        p = results_dir / tag / "train_log.csv"
        if not p.exists():
            continue
        df = pd.read_csv(p)
        if "eval_wer" not in df:
            continue
        df = df.dropna(subset=["eval_wer"])
        ax.plot(df["step"], 100 * df["eval_wer"], styles[k], markersize=3, label=tag)
        found = True
    if found:
        ax.set_xlabel("update step"); ax.set_ylabel("dev WER (%)"); ax.legend(fontsize=8)
        ax.set_title("Fine-tuning dev-WER curves")
        fig.tight_layout(); fig.savefig(path, dpi=300)
    plt.close(fig)


def analyse_variant(preds: pd.DataFrame, variant: str, out_dir: Path, n_examples: int = 20, seed: int = 42) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    m = all_metrics(preds["reference"].tolist(), preds["hypothesis"].fillna("").tolist())
    preds = preds.copy()
    preds["length_bucket"] = preds.get("duration", pd.Series(np.nan, index=preds.index)).astype(float).map(length_bucket)
    for col in ("gender", "age"):
        preds[col] = preds.get(col, "").fillna("").astype(str).replace("", "unknown")
    tables = {by: breakdown(preds, by) for by in ("gender", "age", "length_bucket")}
    tables["subdot_words"] = subdot_word_error(preds)
    for name, t in tables.items():
        t.to_csv(out_dir / f"breakdown_{name}.csv", index=False)
    plot_confusion(m["tone_confusion"], f"{variant}: tone confusion", out_dir / "tone_confusion.png")
    (out_dir / "examples.md").write_text(examples_markdown(preds, variant, n_examples, seed), encoding="utf-8")
    res = {"variant": variant, "tonal_word_error_share": m["wer"] - m["wer_no_tone"], **m,
           "breakdowns": {k: v.to_dict("records") for k, v in tables.items()}}
    save_json(res, out_dir / "error_analysis.json")
    return res


def write_figures(results_dir: Path, summary: pd.DataFrame | None, cis: dict | None) -> list[str]:
    fig_dir = results_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    made = []
    if summary is not None and len(summary):
        plot_variant_bars(summary, cis or {}, fig_dir / "variants_wer_cer_ter.png"); made.append("variants_wer_cer_ter.png")
        plot_interaction(summary, fig_dir / "factorial_interaction.png")
        if (fig_dir / "factorial_interaction.png").exists():
            made.append("factorial_interaction.png")
    abl = results_dir / "ablation.csv"
    if abl.exists():
        plot_ablation(pd.read_csv(abl), fig_dir / "ablation.png"); made.append("ablation.png")
    plot_dev_curves(results_dir, fig_dir / "dev_wer_curves.png")
    if (fig_dir / "dev_wer_curves.png").exists():
        made.append("dev_wer_curves.png")
    cpt_png = results_dir / "V3" / "cpt_loss.png"
    if cpt_png.exists():
        import shutil

        shutil.copy2(cpt_png, fig_dir / "cpt_loss.png"); made.append("cpt_loss.png")
    return made


def load_cis(results_dir: Path) -> dict:
    return load_json(results_dir / "system_cis.json", default={}) or {}


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/evaluate/metrics.py
"""ASR metrics. All public functions return PERCENTAGES (0-100).

* ``wer``/``cer``: jiwer, on tone-preserved normalised text (corpus level).
* ``wer_no_tone``: WER after removing tone marks on both sides (sub-dots kept).
* ``tone_error_rate`` (TER): edit distance between tone-label sequences / reference TBUs x 100.
* ``ter_matched_words``: tone errors only inside word pairs whose tone-stripped forms are equal.
"""
from __future__ import annotations

from typing import Sequence

import numpy as np

from ..text import TONE_LABELS, strip_tones, tone_sequence, word_tone_pairs


def levenshtein(a: Sequence, b: Sequence) -> int:
    """Edit distance (substitutions + deletions + insertions) between two sequences."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, y in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y))
        prev = cur
    return prev[-1]


def _clean(refs: Sequence[str], hyps: Sequence[str]) -> tuple[list[str], list[str]]:
    if len(refs) != len(hyps):
        raise ValueError(f"{len(refs)} references but {len(hyps)} hypotheses")
    pairs = [(str(r).strip(), str(h).strip()) for r, h in zip(refs, hyps) if str(r).strip()]
    return [p[0] for p in pairs], [p[1] for p in pairs]


def wer(refs: Sequence[str], hyps: Sequence[str]) -> float:
    refs, hyps = _clean(refs, hyps)
    if not refs:
        return float("nan")
    try:
        import jiwer

        return 100.0 * float(jiwer.wer(refs, hyps))
    except Exception:  # jiwer edge cases (e.g. empty hypotheses in old versions)
        e = sum(levenshtein(r.split(), h.split()) for r, h in zip(refs, hyps))
        return 100.0 * e / sum(len(r.split()) for r in refs)


def cer(refs: Sequence[str], hyps: Sequence[str]) -> float:
    refs, hyps = _clean(refs, hyps)
    if not refs:
        return float("nan")
    try:
        import jiwer

        return 100.0 * float(jiwer.cer(refs, hyps))
    except Exception:
        e = sum(levenshtein(list(r), list(h)) for r, h in zip(refs, hyps))
        return 100.0 * e / sum(len(r) for r in refs)


def wer_no_tone(refs: Sequence[str], hyps: Sequence[str]) -> float:
    return wer([strip_tones(r) for r in refs], [strip_tones(h) for h in hyps])


def tone_edits(ref: str, hyp: str) -> tuple[int, int]:
    """(edits, reference TBUs) for one utterance."""
    r, h = tone_sequence(ref), tone_sequence(hyp)
    return levenshtein(r, h), len(r)


def tone_error_rate(refs: Sequence[str], hyps: Sequence[str]) -> float:
    refs, hyps = _clean(refs, hyps)
    edits = total = 0
    for r, h in zip(refs, hyps):
        e, n = tone_edits(r, h)
        edits += e
        total += n
    return 100.0 * edits / total if total else float("nan")


def word_alignment(ref: str, hyp: str) -> list[tuple[str, str | None, str | None]]:
    """Word alignment as ``(op, ref_word, hyp_word)`` with op in equal/substitute/delete/insert."""
    r, h = ref.split(), hyp.split()
    n, m = len(r), len(h)
    d = np.zeros((n + 1, m + 1), dtype=np.int32)
    d[:, 0] = np.arange(n + 1)
    d[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (r[i - 1] != h[j - 1]):
            ops.append(("equal" if r[i - 1] == h[j - 1] else "substitute", r[i - 1], h[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            ops.append(("delete", r[i - 1], None))
            i -= 1
        else:
            ops.append(("insert", None, h[j - 1]))
            j -= 1
    return ops[::-1]


def ter_matched_words(refs: Sequence[str], hyps: Sequence[str]) -> tuple[float, list[list[int]]]:
    """Pure tone error rate on aligned word pairs with equal tone-stripped forms, plus the 3x3
    H/M/L confusion matrix (rows = reference, columns = hypothesis)."""
    refs, hyps = _clean(refs, hyps)
    idx = {lab: i for i, lab in enumerate(TONE_LABELS)}
    conf = np.zeros((3, 3), dtype=int)
    errors = total = 0
    for r, h in zip(refs, hyps):
        for op, rw, hw in word_alignment(r, h):
            if op in ("equal", "substitute") and strip_tones(rw) == strip_tones(hw):
                for rl, hl in word_tone_pairs(rw, hw):
                    conf[idx[rl], idx[hl]] += 1
                    total += 1
                    errors += rl != hl
    rate = 100.0 * errors / total if total else float("nan")
    return rate, conf.tolist()


def all_metrics(refs: Sequence[str], hyps: Sequence[str]) -> dict:
    ter_mw, confusion = ter_matched_words(refs, hyps)
    return {
        "wer": wer(refs, hyps), "cer": cer(refs, hyps), "wer_no_tone": wer_no_tone(refs, hyps),
        "ter": tone_error_rate(refs, hyps), "ter_matched_words": ter_mw, "tone_confusion": confusion,
    }


def utterance_stats(ref: str, hyp: str) -> dict:
    """Per-utterance error counts used by the bootstrap and the error analysis."""
    rw, hw = ref.split(), hyp.split()
    e_ter, n_tbu = tone_edits(ref, hyp)
    return {
        "word_errors": levenshtein(rw, hw), "ref_words": len(rw),
        "char_errors": levenshtein(list(ref), list(hyp)), "ref_chars": len(ref),
        "word_errors_notone": levenshtein(strip_tones(ref).split(), strip_tones(hyp).split()),
        "tone_errors": e_ter, "ref_tbus": n_tbu,
    }


def make_compute_metrics(tokenizer):
    """``compute_metrics`` for the HF Trainer: dev ``wer``, ``cer``, ``wer_no_tone`` as FRACTIONS
    (as in the original notebook). Predictions are argmax ids (see ``preprocess_logits``)."""

    def compute_metrics(pred):
        ids = np.asarray(pred.predictions)
        if ids.ndim == 3:  # logits were not pre-reduced
            ids = ids.argmax(-1)
        ids = np.where(ids < 0, tokenizer.pad_token_id, ids)
        label_ids = np.where(pred.label_ids == -100, tokenizer.pad_token_id, pred.label_ids)
        hyp = tokenizer.batch_decode(ids)
        ref = tokenizer.batch_decode(label_ids, group_tokens=False)
        return {"wer": wer(ref, hyp) / 100, "cer": cer(ref, hyp) / 100, "wer_no_tone": wer_no_tone(ref, hyp) / 100}

    return compute_metrics


def preprocess_logits(logits, labels):
    """Reduce logits to argmax ids on the GPU (keeps evaluation memory small)."""
    import torch

    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.argmax(logits, dim=-1)


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/inference.py
"""Inference: load a fine-tuned model and transcribe audio files or arrays (greedy CTC, no LM)."""
from __future__ import annotations

import time
from pathlib import Path

import numpy as np

from .audio import SR, load_audio, resample, split_on_silence, trim_silence
from .text import nfc, strip_tones

_SPECIAL = ("[UNK]", "[PAD]", "<s>", "</s>")


def clean_output(text: str) -> str:
    """Remove special tokens a CTC model can emit and tidy spaces (display only)."""
    for tok in _SPECIAL:
        text = text.replace(tok, "")
    return " ".join(nfc(text).split())


class Transcriber:
    def __init__(self, model, processor, device: str, name: str = "model"):
        self.model, self.processor, self.device, self.name = model, processor, device, name

    @classmethod
    def from_dir(cls, path: str | Path, device: str | None = None) -> "Transcriber":
        return cls._load(str(path), device, name=Path(path).name)

    @classmethod
    def from_hub(cls, repo_id: str, device: str | None = None) -> "Transcriber":
        return cls._load(repo_id, device, name=repo_id)

    @classmethod
    def _load(cls, source: str, device: str | None, name: str) -> "Transcriber":
        import torch
        from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        processor = Wav2Vec2Processor.from_pretrained(source)
        model = Wav2Vec2ForCTC.from_pretrained(source).to(device).eval()
        return cls(model, processor, device, name)

    def _decode_chunks(self, chunks: list[np.ndarray]) -> list[str]:
        import torch

        texts = []
        for c in chunks:
            inputs = self.processor(c, sampling_rate=SR, return_tensors="pt", padding=True)
            mask = inputs.get("attention_mask")
            with torch.no_grad():
                logits = self.model(inputs.input_values.to(self.device),
                                    attention_mask=mask.to(self.device) if mask is not None else None).logits
            ids = torch.argmax(logits, dim=-1).cpu().numpy()
            texts.append(clean_output(self.processor.batch_decode(ids)[0]))
        return texts

    def transcribe_array(self, audio: np.ndarray, sr: int) -> dict:
        """Transcribe a waveform. Audio longer than 15 s is split on silence into <= 15 s chunks."""
        t0 = time.time()
        x = np.asarray(audio, dtype=np.float32)
        if x.ndim == 2:
            x = x.mean(axis=1) if x.shape[1] <= 8 else x.mean(axis=0)
        if np.issubdtype(np.asarray(audio).dtype, np.integer):
            x = x / float(np.iinfo(np.asarray(audio).dtype).max)
        x = resample(x, int(sr), SR)
        peak = float(np.max(np.abs(x))) if x.size else 0.0
        if peak > 1.0:
            x = x / peak
        x = trim_silence(x, 30)
        if len(x) < int(0.1 * SR):
            return {"text": "", "text_notone": "", "seconds": round(time.time() - t0, 3), "model": self.name}
        chunks = split_on_silence(x, SR, max_seconds=15.0)
        text = " ".join(t for t in self._decode_chunks(chunks) if t)
        return {"text": text, "text_notone": strip_tones(text), "seconds": round(time.time() - t0, 3),
                "model": self.name, "audio_seconds": round(len(x) / SR, 2), "chunks": len(chunks)}

    def transcribe_file(self, path: str | Path) -> dict:
        return self.transcribe_array(load_audio(path, SR), SR)

    def transcribe_batch(self, paths: list[str], batch_size: int = 8) -> list[dict]:
        """Batched transcription of short (<= 15 s) files; longer files fall back to chunking."""
        import torch

        out: list[dict | None] = [None] * len(paths)
        short: list[tuple[int, np.ndarray]] = []
        for i, p in enumerate(paths):
            x = trim_silence(load_audio(p, SR), 30)
            if len(x) > 15 * SR:
                out[i] = self.transcribe_array(x, SR)
            else:
                short.append((i, x))
        short.sort(key=lambda t: -len(t[1]))
        for s in range(0, len(short), batch_size):
            part = short[s: s + batch_size]
            t0 = time.time()
            inputs = self.processor([x for _, x in part], sampling_rate=SR, return_tensors="pt", padding=True)
            mask = inputs.get("attention_mask")
            with torch.no_grad():
                logits = self.model(inputs.input_values.to(self.device),
                                    attention_mask=mask.to(self.device) if mask is not None else None).logits
            texts = self.processor.batch_decode(torch.argmax(logits, dim=-1).cpu().numpy())
            dt = (time.time() - t0) / len(part)
            for (i, _), t in zip(part, texts):
                t = clean_output(t)
                out[i] = {"text": t, "text_notone": strip_tones(t), "seconds": round(dt, 3), "model": self.name}
        return out  # type: ignore[return-value]


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/modeling.py
"""Model construction: XLS-R CTC models and the tiny random model used for CPU smoke tests."""
from __future__ import annotations

import warnings
from pathlib import Path

from .config import Config
from .utils import get_logger

log = get_logger(__name__)
# The frozen CNN feature encoder is still wrapped by gradient checkpointing; torch then warns that its
# (frozen) inputs need no gradient. Verified harmless: all trainable layers receive gradients.
warnings.filterwarnings("ignore", message="None of the inputs have requires_grad=True")
TINY_NAME = "tiny-random"


def tiny_wav2vec2_config():
    """A very small Wav2Vec2 config (hidden 32, 2 layers, 3 conv layers, 80x downsampling)."""
    from transformers import Wav2Vec2Config

    return Wav2Vec2Config(
        vocab_size=8, hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=37,
        conv_dim=(32, 32, 32), conv_stride=(5, 4, 4), conv_kernel=(10, 4, 4), num_feat_extract_layers=3,
        conv_bias=True, feat_extract_norm="layer", do_stable_layer_norm=True,
        num_conv_pos_embeddings=16, num_conv_pos_embedding_groups=2,
        num_codevector_groups=2, num_codevectors_per_group=8, codevector_dim=16, proj_codevector_dim=16,
        num_negatives=10, diversity_loss_weight=0.1, mask_time_prob=0.05, mask_time_length=4,
        hidden_dropout=0.0, attention_dropout=0.0, activation_dropout=0.0, feat_proj_dropout=0.0,
        final_dropout=0.0, layerdrop=0.0,
    )


def ensure_tiny_init(work_dir: str | Path, seed: int = 42) -> Path:
    """Create (once) a tiny random ``Wav2Vec2ForPreTraining`` checkpoint that plays the role of XLS-R."""
    import torch
    from transformers import Wav2Vec2ForPreTraining

    path = Path(work_dir) / "tiny_init"
    if (path / "config.json").exists():
        return path
    torch.manual_seed(seed)
    model = Wav2Vec2ForPreTraining(tiny_wav2vec2_config())
    model.save_pretrained(path)
    return path


def resolve_model_name(cfg: Config) -> str:
    name = cfg.model.name
    if name == TINY_NAME:
        return str(ensure_tiny_init(cfg.paths.work_dir, cfg.seed))
    return name


def ctc_kwargs(cfg: Config, processor, mask_time_prob: float) -> dict:
    m = cfg.model
    tiny = cfg.model.name == TINY_NAME
    return dict(
        attention_dropout=m.attention_dropout, hidden_dropout=m.hidden_dropout,
        feat_proj_dropout=m.feat_proj_dropout, layerdrop=m.layerdrop,
        mask_time_prob=float(mask_time_prob), mask_time_length=4 if tiny else m.mask_time_length,
        mask_feature_prob=m.mask_feature_prob,
        ctc_loss_reduction=m.ctc_loss_reduction, ctc_zero_infinity=m.ctc_zero_infinity,
        pad_token_id=processor.tokenizer.pad_token_id, vocab_size=len(processor.tokenizer),
    )


def load_ctc_model(init_from: str, cfg: Config, processor, mask_time_prob: float):
    """``Wav2Vec2ForCTC.from_pretrained`` with the project's settings; logs newly initialised weights."""
    from transformers import Wav2Vec2ForCTC

    model, info = Wav2Vec2ForCTC.from_pretrained(init_from, output_loading_info=True,
                                                 **ctc_kwargs(cfg, processor, mask_time_prob))
    if info.get("missing_keys"):
        log.info("Newly initialised weights (expected for a new CTC head): %s", info["missing_keys"])
    model.config.mask_time_prob = float(mask_time_prob)
    if cfg.model.freeze_feature_encoder:
        model.freeze_feature_encoder()
    if cfg.model.gradient_checkpointing:
        model.gradient_checkpointing_enable()
    return model, info


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/pipeline.py
"""Stage orchestration: prepare -> V1 -> augment -> tune_mask -> V2 -> cpt -> V3 -> V4 -> evaluate -> analyse.

Every stage reads/writes under ``paths.work_dir`` and is safe to re-run (completed work is skipped
when ``resume=True``). The CLI and the Kaggle notebook both call :func:`run_stage`.
"""
from __future__ import annotations

import shutil
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd

from . import VARIANTS
from .config import (CONFIGS_DIR, Config, DataPaths, Paths, deep_merge, discover_paths, load_config)
from .data.audit import build_audit, write_audit
from .data.common_voice import clip_durations, load_sentence_pool, load_split
from .data.generic import ManifestError, list_audio
from .data.iroyin import load_iroyin
from .data.leakage import (assert_no_eval_audio, forbidden_sentences, leakage_key, remove_forbidden,
                           speaker_overlap)
from .data.slr86 import load_slr86
from .text import count_intraword, has_digits, has_tone_mark, normalize_text, strip_tones
from .utils import (RunInfo, get_logger, gpu_info, load_json, now_iso, save_json, seed_everything)

log = get_logger("yoruba_asr")

STAGE_CONFIGS = {"V1": "v1_baseline.yaml", "V2": "v2_augmentation.yaml", "V3": "v3_domain_adaptation.yaml",
                 "V4": "v4_hybrid.yaml", "cpt": "cpt.yaml", "ablation": "ablation.yaml",
                 "tune_mask": "v2_augmentation.yaml", "augment": "v2_augmentation.yaml"}

NEXT_STAGE = {
    "smoke": ("prepare", "Attach Common Voice (+ SLR86, ÌròyìnSpeech, MUSAN if you have them)."),
    "prepare": ("V1", "Attach the data datasets + the output of this run."),
    "V1": ("augment+tune_mask", "Attach the data datasets + the output of this run."),
    "augment": ("tune_mask", "Attach the data + the output of this run."),
    "augment+tune_mask": ("V2", "Attach the data + the output of this run."),
    "tune_mask": ("V2", "Attach the data + the output of this run."),
    "V2": ("cpt", "Attach the data + the output of this run. CPT needs two sessions."),
    "cpt": ("cpt (again until completed) then V3", "Attach the data + the output of this run."),
    "V3": ("V4", "Attach the data + the output of this run."),
    "V4": ("evaluate", "Attach the output of this run."),
    "evaluate": ("ablation", "Attach the data + the output of this run (one ablation per session if needed)."),
    "ablation": ("ablation (next run) or analyse", "Attach the output of this run."),
    "analyse": ("done", "Download /kaggle/working/results for the thesis and models/ for the app."),
    "all_light": ("prepare", "The light run is only a sanity check; now run the real stages."),
}


# --------------------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------------------
@dataclass
class Ctx:
    cfg: Config
    paths: Paths
    found: DataPaths | None

    @property
    def results(self) -> Path:
        return self.paths.results_dir

    def manifest(self, name: str) -> Path:
        return self.paths.manifests_dir / f"{name}.csv"


def make_ctx(cfg: Config, discover: bool = True) -> Ctx:
    seed_everything(int(cfg.seed))
    paths = Paths.from_config(cfg)
    found = discover_paths(cfg) if discover else None
    return Ctx(cfg, paths, found)


def stage_config(stage: str, overrides=None, smoke: bool = False, extra_files: list | None = None) -> Config:
    files = [CONFIGS_DIR / "base.yaml"]
    if stage in STAGE_CONFIGS:
        files.append(CONFIGS_DIR / STAGE_CONFIGS[stage])
    files += [Path(f) for f in (extra_files or [])]
    if smoke:
        files.append(CONFIGS_DIR / "smoke.yaml")
    return load_config(files, overrides)


def _ensure_block(cfg: Config, name: str, file: str) -> None:
    import yaml

    defaults = (yaml.safe_load((CONFIGS_DIR / file).read_text(encoding="utf-8")) or {}).get(name, {})
    cfg[name] = Config(deep_merge(defaults, dict(cfg.get(name) or {})))
    from .config import _wrap

    cfg[name] = _wrap(cfg[name])


def _limit(df: pd.DataFrame, n) -> pd.DataFrame:
    return df.head(int(n)).reset_index(drop=True) if n else df


def _read_manifest(path: Path, required: bool = True) -> pd.DataFrame:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{path} not found - run the 'prepare' stage first (or attach its output).")
        return pd.DataFrame()
    return pd.read_csv(path, dtype={"speaker": str, "gender": str, "age": str}, keep_default_na=False,
                       encoding="utf-8")


def add_text_columns(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = df.copy()
    keep = bool(cfg.text.keep_tones)
    df["text_tone"] = df["sentence"].astype(str).map(lambda s: normalize_text(s, keep_tones=keep))
    df["text_notone"] = df["text_tone"].map(strip_tones)
    reason = pd.Series([None] * len(df), index=df.index, dtype=object)
    if cfg.data.drop_digits:
        reason[df["sentence"].astype(str).map(has_digits)] = "digits"
    reason[(df["text_tone"].str.strip() == "") & reason.isna()] = "empty_text"
    df["drop_reason"] = reason
    return df


def resolve_mask(cfg: Config, ctx: Ctx) -> float:
    m = cfg.get("mask_time_prob", cfg.model.mask_time_prob)
    if m != "tuned":
        return float(m)
    info = load_json(ctx.results / "V2" / "mask_tuning.json")
    if info and info.get("chosen") is not None:
        return float(info["chosen"])
    if cfg.mask_tuning.get("skip"):
        return float(cfg.mask_tuning.default_if_skipped)
    raise FileNotFoundError("results/V2/mask_tuning.json not found: run the 'tune_mask' stage first "
                            "(or set mask_tuning.skip=true to use 0.075).")


def _processor(ctx: Ctx):
    from .vocab import load_shared_processor

    return load_shared_processor(ctx.paths.shared_dir, ctx.results)


def _cv_cached(ctx: Ctx, split: str):
    from .train.datasets import CachedASRDataset, load_cached_split

    p = ctx.paths.cache_dir / f"cv_{split}"
    if not p.exists():
        raise FileNotFoundError(f"{p} not found - run the 'prepare' stage first (or attach its output).")
    return CachedASRDataset(load_cached_split(p))


def _manifest_dataset(ctx: Ctx, df: pd.DataFrame, processor):
    from .train.datasets import ManifestASRDataset

    if df is None or len(df) == 0:
        return None
    return ManifestASRDataset(df, processor, top_db=float(ctx.cfg.data.trim_top_db))


def _previous(ctx: Ctx) -> list[Path]:
    return ctx.found.previous_outputs if ctx.found else []


# --------------------------------------------------------------------------------------
# loading corpora (shared by audit + prepare)
# --------------------------------------------------------------------------------------
class MissingAudioError(FileNotFoundError):
    pass


def check_cv_audio(cv: dict, cfg: Config, max_missing_share: float = 0.2) -> dict:
    """Make sure the clips listed in the TSVs exist; find them anywhere under the input if needed.

    Raises :class:`MissingAudioError` with a listing of what the dataset contains when more than
    ``max_missing_share`` of train/dev/test clips cannot be found.
    """
    from .data.common_voice import dataset_tree_report, resolve_clip_paths

    roots = [cfg.paths.input_root, str(Path(cfg.paths.tmp_dir) / "extracted")]
    report = {}
    for split in ("train", "dev", "test", "other"):
        if len(cv[split]) == 0:
            continue
        cv[split], report[split] = resolve_clip_paths(cv[split], roots)
        r = report[split]
        if r["remapped"]:
            log.info("cv_%s: %d clips were not in clips/ and were found elsewhere in the input", split, r["remapped"])
        if r["missing"]:
            log.warning("cv_%s: %d of %d clips are missing from the attached data", split, r["missing"], len(cv[split]))
    bad = {s: r for s, r in report.items() if s != "other" and r["missing"] > max_missing_share * len(cv[s])}
    if bad:
        found = next(iter(report.values())).get("audio_files_found_under_input", "?")
        raise MissingAudioError(
            "\n" + "=" * 90 + "\nCOMMON VOICE AUDIO IS MISSING. The TSV files were found, but most MP3 clips were not:\n"
            + "\n".join(f"  {s}: {r['missing']} of {len(cv[s])} clips missing" for s, r in bad.items())
            + f"\n  audio files found anywhere under {cfg.paths.input_root}: {found} (Common Voice 27.0 Yoruba has 4,970)\n"
            "\nWhat the attached input contains:\n" + dataset_tree_report(cfg.paths.input_root) +
            "\n\nFIX: delete the Kaggle dataset and upload it again as ONE file: the original archive\n"
            "  '1789491097449-cv-corpus-27.0-2026-09-11-yo.tar.gz' (about 175 MB). Do not upload the extracted\n"
            "  folder (the one on your PC is incomplete). Wait until Kaggle finishes processing, check that the\n"
            "  dataset page shows a 'clips' folder with 4,970 files, then attach it again (see docs/KAGGLE_GUIDE.md).\n"
            + "=" * 90)
    return report


def _load_cv_primary(cfg: Config, cv_dir: Path) -> dict:
    lim = cfg.data.limit
    cv = {s: add_text_columns(_limit(load_split(cv_dir, s), lim.get(s)), cfg) for s in ("train", "dev", "test")}
    cv["other"] = _limit(load_split(cv_dir, "other"), lim.get("other"))
    check_cv_audio(cv, cfg)
    return cv


def _load_slr86_primary(cfg: Config, folders: list[Path]) -> dict:
    """OpenSLR 86 as the main corpus: speaker-disjoint (and sentence-disjoint) train/dev/test."""
    from .data.slr86 import speaker_split

    df = load_slr86(folders)
    if df.empty:
        raise FileNotFoundError(f"OpenSLR 86 folders {folders} contain no usable recordings.")
    sp = cfg.data.slr86.get("split", {}) or {}
    parts = speaker_split(df, float(sp.get("dev_share", 0.1)), float(sp.get("test_share", 0.1)), int(cfg.seed),
                          key=leakage_key)
    lim = cfg.data.limit
    cv = {s: add_text_columns(_limit(parts[s], lim.get(s)), cfg) for s in ("train", "dev", "test")}
    cv["other"] = df.iloc[:0].copy()
    empty = [s for s in ("train", "dev", "test") if len(cv[s]) == 0]
    if empty:
        raise ValueError(f"OpenSLR 86 split produced empty {empty} sets ({df['speaker'].nunique()} speakers found). "
                         "Attach both yo_ng_female and yo_ng_male (or let the notebook download them).")
    log.info("OpenSLR 86 as primary corpus: train %d / dev %d / test %d utterances, speakers %d / %d / %d",
             *(len(cv[s]) for s in ("train", "dev", "test")), *(cv[s]["speaker"].nunique() for s in ("train", "dev", "test")))
    return cv


def choose_primary(cfg: Config, found: DataPaths) -> tuple[str, dict, str]:
    """Return (primary corpus name, train/dev/test/other frames, note)."""
    mode = str(cfg.data.get("primary", "auto"))
    have_cv, have_slr = found is not None and found.common_voice is not None, bool(found and found.slr86)
    if mode == "slr86" or (mode == "auto" and not have_cv):
        if not have_slr:
            raise FileNotFoundError(
                f"No training data found under {cfg.paths.input_root}: attach Common Voice 27.0 Yoruba, or turn "
                "Internet On so OpenSLR 86 can be downloaded automatically (data.slr86.download: true).")
        return "slr86", _load_slr86_primary(cfg, found.slr86), "OpenSLR 86 (Common Voice not used)"
    if not have_cv:
        raise FileNotFoundError(
            f"Common Voice Yoruba was not found under {cfg.paths.input_root}. Upload the Common Voice 27.0 Yoruba "
            "archive as a PRIVATE Kaggle dataset and attach it, or set PRIMARY_DATASET = 'slr86'.")
    try:
        return "common_voice", _load_cv_primary(cfg, found.common_voice), "Common Voice 27.0"
    except MissingAudioError as exc:
        if mode == "auto" and have_slr:
            log.warning("=" * 90 + "\nCommon Voice audio is missing, so OpenSLR 86 is used as the main corpus "
                        "instead (PRIMARY_DATASET='auto').\n" + str(exc).splitlines()[2] + "\n" + "=" * 90)
            return "slr86", _load_slr86_primary(cfg, found.slr86), "OpenSLR 86 (fallback: Common Voice audio missing)"
        raise


def load_corpora(ctx: Ctx) -> dict:
    cfg, found = ctx.cfg, ctx.found
    lim = cfg.data.limit
    primary, cv, primary_note = choose_primary(cfg, found)
    cv_dir = found.common_voice if primary == "common_voice" else None
    extra_counts = {s: len(load_split(cv_dir, s)) for s in ("validated", "invalidated")} if cv_dir else {}
    forbidden = forbidden_sentences(cv["dev"], cv["test"])

    ood, ood_notes, gate = [], {}, {}
    if found.slr86 and primary != "slr86":
        ood.append(load_slr86(found.slr86))
    if found.iroyin is not None or found.iroyin_manifest is not None:
        try:
            ood.append(load_iroyin(found.iroyin, found.iroyin_manifest))
        except ManifestError as exc:
            ood_notes["iroyin"] = str(exc)
            log.warning("ÌròyìnSpeech skipped: %s", exc)
    ood_df = pd.concat(ood, ignore_index=True) if ood else pd.DataFrame(
        columns=["path", "sentence", "speaker", "corpus", "split", "gender", "age"])
    leak_removed = {}
    kept_parts = []
    for corpus, g in ood_df.groupby("corpus") if len(ood_df) else []:
        toneless = float(np.mean([not has_tone_mark(s) for s in g["sentence"]])) if len(g) else 0.0
        needs_check = toneless > float(cfg.data.tone_gate_max_toneless_share)
        force = bool(cfg.data.get(corpus, {}).get("force_include", False))
        gate[corpus] = {"share_toneless": round(toneless, 4), "needs_manual_check": needs_check,
                        "force_include": force, "included": (not needs_check) or force}
        if needs_check:
            log.warning("=" * 70 + f"\n{corpus}: {toneless:.0%} of transcripts have NO tone marks -> "
                        "needs_manual_check. Listen to a 50-utterance sample before using it. "
                        f"{'Included because force_include=true.' if force else 'EXCLUDED (set data.' + corpus + '.force_include: true to include).'}\n" + "=" * 70)
            if not force:
                continue
        g = add_text_columns(g, cfg)
        g, n_removed = remove_forbidden(g, forbidden)
        leak_removed[corpus] = n_removed
        kept_parts.append(g)
    ood_df = pd.concat(kept_parts, ignore_index=True) if kept_parts else ood_df.assign(
        text_tone="", text_notone="", drop_reason=None)
    ood_df = _limit(ood_df, lim.get("ood"))

    tts, tts_info = build_tts_pool(cfg, cv["train"], forbidden, cv_dir)
    if "removed_forbidden" in tts_info:
        leak_removed["tts_pool"] = tts_info["removed_forbidden"]
    return {"cv": cv, "cv_dir": cv_dir, "primary": primary, "primary_note": primary_note, "tts_info": tts_info,
            "extra_counts": extra_counts, "forbidden": forbidden, "ood": ood_df,
            "ood_notes": ood_notes, "tone_gate": gate, "leak_removed": leak_removed, "tts_sentences": tts}


def build_tts_pool(cfg: Config, train: pd.DataFrame, forbidden: set, cv_dir) -> tuple[pd.DataFrame, dict]:
    """TTS text pool: CV validated sentences (or, with OpenSLR 86 as primary, its training sentences),
    never dev/test sentences (tone-stripped match), no digits. Returns (sentences, step counts)."""
    from .vocab import canonical_charset

    if cv_dir is not None:
        pool = load_sentence_pool(cv_dir, [cfg.paths.input_root, str(Path(cfg.paths.tmp_dir) / "extracted")])
        info = {"source": pool.attrs.get("pool_source")}
    else:
        pool = train[["sentence"]].drop_duplicates().reset_index(drop=True)
        info = {"source": "primary training sentences (OpenSLR 86)"}
    info["loaded"] = len(pool)
    empty = pd.DataFrame(columns=["sentence", "text_tone", "text_notone"])
    if not len(pool):
        log.warning("TTS sentence pool is empty (%s)", info["source"])
        return empty, info
    pool = add_text_columns(pool, cfg)
    pool = pool[pool["drop_reason"].isna()]
    info["after_text_filter"] = len(pool)
    pool, info["removed_forbidden"] = remove_forbidden(pool, forbidden)
    # Only sentences written with characters already used by the training text (or the canonical Yoruba
    # set), so odd encodings (e.g. U+031F "plus below" instead of the sub-dot) never enter the vocabulary.
    allowed = canonical_charset() | set("".join(train["text_tone"])) | {" "}
    pool = pool[pool["text_tone"].map(lambda t: set(t) <= allowed)].drop_duplicates("text_tone").copy()
    info["after_charset_filter"] = len(pool)
    train_keys = set(train["sentence"].map(leakage_key))
    pool["recorded_in_train"] = pool["sentence"].map(leakage_key).isin(train_keys)
    pool = pool.sample(frac=1.0, random_state=int(cfg.seed))
    pool = pool.sort_values("recorded_in_train", kind="stable")  # unrecorded sentences first
    cap = int(np.ceil(len(train) * float(cfg.tts.max_per_real_clip) * 1.5))
    tts = pool.head(cap).reset_index(drop=True)
    info["selected"] = len(tts)
    log.info("TTS sentence pool: %s", info)
    return tts, info


def _unlabelled_pool(ctx: Ctx, data: dict, durations_cv: dict) -> pd.DataFrame:
    from .train.datasets import probe_manifest

    cfg = ctx.cfg
    other = data["cv"]["other"][["path", "speaker", "corpus"]].copy()
    other["corpus"] = "common_voice_other"
    parts = [other]
    if data.get("primary") == "slr86":  # primary training audio (never dev/test) is valid unlabelled audio
        parts.append(data["cv"]["train"][["path", "speaker", "corpus"]])
    ood = data["ood"]
    if len(ood):
        parts.append(ood[["path", "speaker", "corpus"]])
    if ctx.found.bibletts is not None:
        files = list_audio(ctx.found.bibletts)
        parts.append(pd.DataFrame({"path": files, "speaker": "bibletts", "corpus": "bibletts"}))
    pool = pd.concat(parts, ignore_index=True).drop_duplicates("path")
    pool = _limit(pool, cfg.data.limit.get("unlabelled"))
    # Durations: CV from clip_durations.tsv (fast) else file headers.
    known = pool["path"].map(lambda p: durations_cv.get(Path(p).name))
    probed = probe_manifest(pool[known.isna()], 0.0, 1e9) if known.isna().any() else pool.iloc[:0].assign(duration=[])
    pool["duration"] = known
    if len(probed):
        pool.loc[known.isna(), "duration"] = probed["duration"].values
    pool = pool[pool["duration"].notna() & (pool["duration"].astype(float) >= float(cfg.data.min_seconds))]
    return pool.reset_index(drop=True)


# --------------------------------------------------------------------------------------
# stages
# --------------------------------------------------------------------------------------
def stage_audit(cfg: Config, **_) -> dict:
    """Corpus audit from transcripts + file headers (no feature extraction)."""
    from .train.datasets import probe_manifest

    ctx = make_ctx(cfg)
    data = load_corpora(ctx)
    rows = []
    for split, df in data["cv"].items():
        if split == "other":
            df = add_text_columns(df, cfg)
        p = probe_manifest(df, cfg.data.min_seconds, cfg.data.max_seconds)
        p["drop_reason"] = df["drop_reason"].where(df["drop_reason"].notna(), p["drop_reason"])
        rows.append(p)
    if len(data["ood"]):
        p = probe_manifest(data["ood"], cfg.data.min_seconds, cfg.data.max_seconds)
        p["drop_reason"] = data["ood"]["drop_reason"].where(data["ood"]["drop_reason"].notna(), p["drop_reason"])
        rows.append(p)
    audit = build_audit(pd.concat(rows, ignore_index=True), extra={
        "cv_other_counts": data["extra_counts"], "ood_tone_gate": data["tone_gate"], "ood_notes": data["ood_notes"],
        "note": "audit stage uses raw (untrimmed) durations; 'prepare' reports trimmed durations"})
    write_audit(audit, ctx.results)
    print(f"Audit written to {ctx.results / 'corpus_audit.md'}")
    return audit


def stage_prepare(cfg: Config, resume: bool = True, **_) -> dict:
    """Load, normalise, filter, leakage-check, build the shared vocabulary and cache Common Voice."""
    from .train.datasets import build_cached_split, load_cached_split, probe_manifest
    from .vocab import build_vocab, make_processor, save_vocab, unknown_chars, vocab_sha256

    ctx = make_ctx(cfg)
    run = RunInfo(ctx.results / "stages" / "prepare_run_info.json", "prepare")
    data = load_corpora(ctx)
    cv, ood, tts = data["cv"], data["ood"], data["tts_sentences"]
    durations_cv = clip_durations(data["cv_dir"]) if data["cv_dir"] else {}

    # Shared vocabulary from CV train + OOD train + TTS sentences (never CV dev/test).
    vocab_texts = (cv["train"].loc[cv["train"]["drop_reason"].isna(), "text_tone"].tolist()
                   + (ood.loc[ood["drop_reason"].isna(), "text_tone"].tolist() if len(ood) else [])
                   + tts["text_tone"].tolist())
    vocab_path = ctx.paths.shared_dir / "vocab.json"
    if resume and vocab_path.exists() and (ctx.results / "vocab_info.json").exists():
        log.info("Reusing existing shared vocabulary %s", vocab_path)
        vinfo = load_json(ctx.results / "vocab_info.json")
    else:
        vinfo = save_vocab(build_vocab(vocab_texts), ctx.paths.shared_dir, ctx.results,
                           sources={"cv_train": int(cv["train"]["drop_reason"].isna().sum()),
                                    "ood_train": int(ood["drop_reason"].isna().sum()) if len(ood) else 0,
                                    "tts_sentences": len(tts)})
    processor = make_processor(vocab_path)
    vocab = processor.tokenizer.get_vocab()
    unk = {s: unknown_chars(cv[s]["text_tone"], vocab) for s in ("dev", "test")}
    vinfo["unknown_chars_in_cv_dev_test"] = unk
    save_json(vinfo, ctx.results / "vocab_info.json")

    # Common Voice audio -> cached HF datasets (input_values, input_length, labels).
    sha = vocab_sha256(ctx.paths.shared_dir)
    audit_rows, manifests = [], {}
    for split in ("train", "dev", "test"):
        df = cv[split]
        kept_text = df[df["drop_reason"].isna()]
        out = ctx.paths.cache_dir / f"cv_{split}"
        cache_info = load_json(out / "cache_info.json") or {}
        if resume and out.exists() and cache_info.get("vocab_sha256") == sha and cache_info.get("rows_in") == len(kept_text):
            ds = load_cached_split(out)
            drop_log = cache_info.get("drop_log", {})
            log.info("cv_%s: reusing cache (%d clips)", split, len(ds))
        else:
            if out.exists():
                shutil.rmtree(out)
            log.info("cv_%s: decoding %d clips ...", split, len(kept_text))
            ds, drop_log = build_cached_split(kept_text, processor, out, ctx.paths.tmp_dir, cfg.data.min_seconds,
                                              cfg.data.max_seconds, cfg.data.trim_top_db)
            save_json({"vocab_sha256": sha, "rows_in": len(kept_text), "rows_out": len(ds), "drop_log": drop_log,
                       "built_at": now_iso()}, out / "cache_info.json")
        meta = ds.select_columns(["path", "duration"]).to_pandas()
        dur_map = dict(zip(meta["path"], meta["duration"]))
        full = df.copy()
        full["duration"] = full["path"].map(dur_map)
        full["drop_reason"] = full["drop_reason"].where(full["drop_reason"].notna(), full["path"].map(drop_log))
        hdr = probe_manifest(full[["path"]], 0, 1e9)
        full["sample_rate"] = hdr["sample_rate"].values
        audit_rows.append(full)
        kept = full[full["drop_reason"].isna()].copy()
        manifests[f"cv_{split}"] = kept
        kept.drop(columns=["drop_reason"]).to_csv(ctx.manifest(f"cv_{split}"), index=False, encoding="utf-8")

    # OOD labelled (B1) - durations from headers; audio is loaded on the fly at training time.
    if len(ood):
        o = probe_manifest(ood, cfg.data.min_seconds, cfg.data.max_seconds)
        o["drop_reason"] = ood["drop_reason"].where(ood["drop_reason"].notna(), o["drop_reason"])
        audit_rows.append(o)
        ood_kept = o[o["drop_reason"].isna()].copy()
    else:
        ood_kept = pd.DataFrame(columns=["path", "sentence", "text_tone", "text_notone", "speaker", "corpus", "duration"])
    ood_kept.drop(columns=["drop_reason"], errors="ignore").to_csv(ctx.manifest("ood_train"), index=False, encoding="utf-8")

    # Unlabelled pool for CPT (CV other + OOD audio + BibleTTS), never CV dev/test.
    unl = _unlabelled_pool(ctx, data, durations_cv)
    unl.to_csv(ctx.manifest("unlabelled"), index=False, encoding="utf-8")
    other = add_text_columns(cv["other"], cfg)
    other["duration"] = other["path"].map(lambda p: durations_cv.get(Path(p).name))
    audit_rows.append(other.assign(drop_reason=other["drop_reason"]))
    tts.to_csv(ctx.manifest("tts_sentences"), index=False, encoding="utf-8")

    # Leakage report.
    pools = {"cv_train": manifests["cv_train"], "ood_train": ood_kept, "unlabelled": unl}
    in_pools = assert_no_eval_audio(pools, cv["dev"], cv["test"])
    tts_leaks = int(tts["sentence"].map(leakage_key).isin(data["forbidden"]).sum()) if len(tts) else 0
    assert tts_leaks == 0, "TTS sentences overlap CV dev/test"
    overlap = speaker_overlap(cv["train"], cv["dev"], cv["test"])
    leak = {"forbidden_sentences": len(data["forbidden"]), "removed_by_corpus": data["leak_removed"],
            "cv_dev_test_audio_in_pools": in_pools, "tts_sentences_in_forbidden": tts_leaks,
            "speaker_overlap": overlap, "speaker_overlap_ok": all(v == 0 for v in overlap.values()),
            "checked_at": now_iso()}
    save_json(leak, ctx.results / "leakage_report.json")
    if not leak["speaker_overlap_ok"]:
        log.warning("CV train/dev/test speakers overlap: %s (recorded in the audit)", overlap)

    # Audit.
    intraword = {"apostrophes": 0, "hyphens": 0}
    for df in cv.values():
        for s in df["sentence"]:
            for k, v in count_intraword(str(s)).items():
                intraword[k] += v
    rows = pd.concat(audit_rows, ignore_index=True)
    audit = build_audit(rows, extra={
        "primary_dataset": data["primary_note"],
        "cv_other_counts": data["extra_counts"], "ood_tone_gate": data["tone_gate"], "ood_notes": data["ood_notes"],
        "intraword_punctuation_removed": intraword, "digit_policy": "utterances containing digits are dropped",
        "speaker_overlap": overlap, "unlabelled_pool_hours": round(float(unl["duration"].sum()) / 3600, 2),
        "unlabelled_pool_clips": len(unl), "tts_sentence_pool": len(tts),
        "unknown_chars_cv_dev_test": unk})
    write_audit(audit, ctx.results)

    summary = {
        "primary_dataset": data["primary_note"],
        "cv_train": len(manifests["cv_train"]), "cv_dev": len(manifests["cv_dev"]), "cv_test": len(manifests["cv_test"]),
        "cv_train_hours": round(float(manifests["cv_train"]["duration"].sum()) / 3600, 3),
        "ood_train": len(ood_kept), "ood_hours": round(float(ood_kept["duration"].astype(float).sum()) / 3600, 2) if len(ood_kept) else 0,
        "unlabelled": len(unl), "unlabelled_hours": round(float(unl["duration"].sum()) / 3600, 2),
        "tts_sentences": len(tts), "tts_pool_steps": data["tts_info"], "vocab_size": vinfo["size"], "vocab_sha256": sha,
        "discovered": ctx.found.summary(),
    }
    save_json(summary, ctx.results / "data_summary.json")
    run.finish(steps=None)
    print("\n== prepare summary ==")
    for k, v in summary.items():
        if k != "discovered":
            print(f"  {k}: {v}")
    return summary


def rebuild_tts_sentences(ctx: Ctx, train: pd.DataFrame) -> pd.DataFrame:
    """Rebuild an empty TTS sentence list (e.g. prepare ran before validated_sentences.tsv was found).

    Uses the same rules as 'prepare': dev/test sentences of the primary corpus are forbidden and only
    characters already in the shared vocabulary's source text are allowed. The vocabulary is unchanged.
    """
    summary = load_json(ctx.results / "data_summary.json", default={}) or {}
    use_cv = str(summary.get("primary_dataset", "Common Voice")).startswith("Common Voice")
    cv_dir = ctx.found.common_voice if (use_cv and ctx.found) else None
    if cv_dir is not None:
        dev, test = load_split(cv_dir, "dev"), load_split(cv_dir, "test")
    else:
        dev, test = _read_manifest(ctx.manifest("cv_dev")), _read_manifest(ctx.manifest("cv_test"))
    tts, info = build_tts_pool(ctx.cfg, train, forbidden_sentences(dev, test), cv_dir)
    tts.to_csv(ctx.manifest("tts_sentences"), index=False, encoding="utf-8")
    save_json(info, ctx.results / "V2" / "tts_pool_rebuilt.json")
    log.info("TTS sentence list rebuilt in the augment stage: %d sentences", len(tts))
    return tts


def stage_augment(cfg: Config, resume: bool = True, **_) -> dict:
    """Offline augmented copies (2 per CV train clip), pitch check and MMS-TTS synthesis."""
    from .augment.build_augmented import build_augmented_set, pitch_check
    from .augment.tts import synthesize_set

    ctx = make_ctx(cfg)
    run = RunInfo(ctx.results / "stages" / "augment_run_info.json", "augment")
    train = _read_manifest(ctx.manifest("cv_train"))
    a, d = cfg.augment, cfg.data
    v2_results = ctx.results / "V2"
    musan = ctx.found.musan if ctx.found else None
    rirs = ctx.found.rirs if ctx.found else None

    pc = pitch_check(train, a, int(cfg.seed), float(d.trim_top_db), int(a.pitch_check_clips))
    save_json(pc, v2_results / "pitch_check.json")
    log.info("pitch check: median relative F0 change %.4f (limit %.2f) -> %s", pc["median_relative_diff"],
             pc["limit"], "PASS" if pc["passed"] else "FAIL")
    if not pc["passed"]:
        raise AssertionError(f"Tempo perturbation changed pitch by {pc['median_relative_diff']:.2%} "
                             f"(limit {pc['limit']:.0%}). See results/V2/pitch_check.json")

    manifest, stats = build_augmented_set(train, ctx.paths.augmented_dir / "main", a, int(cfg.seed),
                                          list(a.transforms), d.min_seconds, d.max_seconds, d.trim_top_db,
                                          musan, rirs, name="main")
    tts_info = {"enabled_effective": False, "reason": "tts.enabled=false"}
    if cfg.tts.enabled:
        sentences = _read_manifest(ctx.manifest("tts_sentences"), required=False)
        if len(sentences) == 0:
            sentences = rebuild_tts_sentences(ctx, train)
        cap = int(len(train) * float(cfg.tts.max_per_real_clip))
        if len(sentences):
            _, tts_info = synthesize_set(sentences, ctx.paths.augmented_dir / "tts", v2_results, cfg.tts.model,
                                         int(cfg.seed), cap, d.min_seconds, d.max_seconds, d.trim_top_db,
                                         int(cfg.tts.listening_sample))
        else:
            tts_info = {"enabled_effective": False, "reason": "no TTS sentence pool (validated_sentences.tsv missing)"}
    save_json(tts_info, v2_results / "tts_status.json")
    info = run.finish(steps=None)
    out = {"augmented": stats, "pitch_check_passed": pc["passed"], "tts": {k: tts_info.get(k) for k in
                                                                           ("enabled_effective", "synthesised", "skipped", "reason", "error")},
           "gpu_hours": info["total_gpu_hours"]}
    save_json(out, v2_results / "augment_summary.json")
    print(f"augmented clips: {stats['written']}, TTS clips: {tts_info.get('synthesised', 0)}, "
          f"TTS effective: {tts_info.get('enabled_effective')}")
    return out


def _v2_train_parts(ctx: Ctx, processor, aug_set: str = "main", use_tts: bool = True):
    """CV train (cached) + augmented copies + TTS clips."""
    from .train.datasets import ConcatASRDataset

    cv_train = _cv_cached(ctx, "train")
    aug_m = ctx.paths.augmented_dir / aug_set / "manifest_augmented.csv"
    if not aug_m.exists():
        raise FileNotFoundError(f"{aug_m} not found - run the 'augment' stage first.")
    aug = _read_manifest(aug_m)
    tts_m = ctx.paths.augmented_dir / "tts" / "manifest_tts.csv"
    tts = _read_manifest(tts_m, required=False) if use_tts else pd.DataFrame()
    ds = ConcatASRDataset([cv_train, _manifest_dataset(ctx, aug, processor), _manifest_dataset(ctx, tts, processor)])
    hours = (sum(cv_train.lengths) / 16000 + float(aug["duration"].sum()) +
             (float(tts["duration"].sum()) if len(tts) else 0.0)) / 3600
    return ds, {"cv_train": len(cv_train), "augmented": len(aug), "tts": len(tts), "train_hours": round(hours, 3)}


def stage_tune_mask(cfg: Config, resume: bool = True, **_) -> dict:
    """Three short V2 runs with mask_time_prob in {0.05, 0.075, 0.10}; choose the lowest dev WER."""
    from .modeling import resolve_model_name
    from .train.finetune import run_finetune

    ctx = make_ctx(cfg, discover=True)
    out_path = ctx.results / "V2" / "mask_tuning.json"
    mt = cfg.mask_tuning
    state = load_json(out_path, default={}) if resume else {}
    if mt.skip:
        state = {"skipped": True, "chosen": float(mt.default_if_skipped),
                 "note": "mask tuning skipped (--skip-mask-tuning); default value used"}
        save_json(state, out_path)
        print(f"mask tuning skipped -> {state['chosen']}")
        return state
    processor = _processor(ctx)
    train_ds, parts = _v2_train_parts(ctx, processor)
    dev = _cv_cached(ctx, "dev")
    results = state.get("candidates", {})
    for p in mt.candidates:
        key = f"{float(p):g}"
        if key in results and results[key].get("status") == "complete":
            continue
        tag = f"mask_{key}"
        s = run_finetune(cfg, tag=tag, train_ds=train_ds, eval_ds=dev, processor=processor,
                         init_from=resolve_model_name(cfg), mask_time_prob=float(p), max_steps=int(mt.max_steps),
                         model_dir=ctx.paths.models_dir / "mask_tuning" / key, results_dir=ctx.results / "V2" / tag,
                         resume=resume, previous_outputs=_previous(ctx))
        results[key] = {"mask_time_prob": float(p), "dev_wer": s.get("best_dev_wer"), "status": s["status"],
                        "global_step": s.get("global_step")}
        save_json({"candidates": results, "max_steps": int(mt.max_steps), "chosen": None}, out_path)
        if s["status"] != "complete":
            print("mask tuning interrupted by the time budget - run 'tune_mask' again.")
            return {"candidates": results, "status": "incomplete"}
        shutil.rmtree(ctx.paths.models_dir / "mask_tuning" / key, ignore_errors=True)
    best = min(results.values(), key=lambda r: (r["dev_wer"] if r["dev_wer"] is not None else 1e9))
    state = {"candidates": results, "max_steps": int(mt.max_steps), "chosen": best["mask_time_prob"],
             "selection": "lowest dev WER", "train_parts": parts, "finished_at": now_iso()}
    save_json(state, out_path)
    print(f"chosen mask_time_prob = {state['chosen']} ({results})")
    return state


def stage_cpt(cfg: Config, resume: bool = True, **_) -> dict:
    from .adapt.cpt import CPTError, run_cpt
    from .modeling import resolve_model_name

    _ensure_block(cfg, "cpt", "cpt.yaml")
    ctx = make_ctx(cfg)
    unl = _read_manifest(ctx.manifest("unlabelled"))
    try:
        summary = run_cpt(cfg, unl, resolve_model_name(cfg), ctx.paths.models_dir / "cpt_encoder",
                          ctx.paths.checkpoints_dir / "cpt", ctx.results / "V3", resume=resume)
    except CPTError as exc:
        q = Path(cfg.paths.work_dir) / "OPEN_QUESTIONS_runtime.md"
        q.write_text(f"# CPT stopped\n\n{exc}\n", encoding="utf-8")
        raise
    print(f"CPT: {summary['completed_steps']} steps, stop reason = {summary['stop_reason']}, "
          f"completed = {summary['completed']}")
    if not summary["completed"]:
        print("CPT is not finished: run STAGE='cpt' again (attach this output) until completed=True.")
    return summary


def write_model_card(model_dir: Path, variant: str, cfg: Config, metrics: dict | None, extra: dict) -> None:
    from .reporting import VARIANT_NAMES

    m = metrics or {}
    lines = [
        f"# Yoruba ASR - {variant}: {VARIANT_NAMES.get(variant, variant)}", "",
        "Fine-tuned from `facebook/wav2vec2-xls-r-300m` (wav2vec 2.0, CTC head, greedy decoding, no LM).", "",
        "## Training", "",
        f"- Training set: {extra.get('training_set')} ({extra.get('train_parts')})",
        f"- Initialisation: {extra.get('init')}; stage B1: {extra.get('stage_b1')}",
        f"- max_steps={cfg.train.max_steps}, lr={cfg.train.learning_rate}, effective batch="
        f"{cfg.train.per_device_train_batch_size * cfg.train.gradient_accumulation_steps}, "
        f"mask_time_prob={extra.get('mask_time_prob')}, seed={cfg.seed}",
        f"- Shared vocabulary SHA-256: {extra.get('vocab_sha256')}", "",
        "## Test results (Common Voice 27.0 Yoruba test, %)", "",
        f"WER {m.get('wer', 'n/a')} | CER {m.get('cer', 'n/a')} | WER (no tone) {m.get('wer_no_tone', 'n/a')} | "
        f"TER {m.get('ter', 'n/a')}", "",
        "## Licences and limitations", "",
        "- Common Voice: CC0 (data may not be re-hosted). OpenSLR 86: CC BY-SA 4.0. ÌròyìnSpeech text: CC BY 4.0.",
        "- Synthetic speech (V2/V4) from `facebook/mms-tts-yor` (CC BY-NC 4.0): research use only.",
        "- Read speech, Standard Yoruba, limited number of speakers; not validated for spontaneous speech.", "",
        "## References", "",
        "Baevski et al. (2020) wav2vec 2.0; Babu et al. (2022) XLS-R; Pratap et al. (2024) MMS; Ardila et al. (2020) "
        "Common Voice; Gutkin et al. (2020) Yoruba corpus; Ògúnrẹ̀mí et al. (2024) ÌròyìnSpeech.",
    ]
    (model_dir / "model_card.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def evaluate_variant(ctx: Ctx, variant: str, model_dir: Path, extra: dict | None = None) -> dict:
    from .evaluate.decode import decode_cached, evaluate_predictions, load_model_and_processor
    from .train.datasets import load_cached_split
    from .vocab import vocab_sha256

    model, processor, device = load_model_and_processor(model_dir)
    test = load_cached_split(ctx.paths.cache_dir / "cv_test")
    preds = decode_cached(model, processor, test, device, int(ctx.cfg.eval.batch_size))
    status = load_json(model_dir / "training_complete.json", default={}) or {}
    info = {"train_clips": status.get("train_clips"), "train_hours": status.get("train_hours"),
            "gpu_hours": status.get("gpu_hours_total"), "mask_time_prob": status.get("mask_time_prob"),
            "checkpoint": str(model_dir), "vocab_sha256": vocab_sha256(ctx.paths.shared_dir), **(extra or {})}
    metrics = evaluate_predictions(preds, variant, ctx.results / variant, info)
    del model
    return metrics


def stage_train(cfg: Config, resume: bool = True, variant: str | None = None, **_) -> dict:
    """Train one variant (V1-V4) and evaluate it on the CV test set."""
    from .adapt.staged import run_stage_b1
    from .modeling import resolve_model_name
    from .train.finetune import run_finetune
    from .vocab import vocab_sha256

    variant = variant or cfg.get("variant")
    if variant not in VARIANTS:
        raise ValueError(f"variant must be one of {VARIANTS}, got {variant}")
    needs_raw = bool(cfg.get("stage_b1"))
    ctx = make_ctx(cfg, discover=True)
    processor = _processor(ctx)
    mask = resolve_mask(cfg, ctx)
    dev = _cv_cached(ctx, "dev")
    if cfg.training_set == "cv":
        train_ds = _cv_cached(ctx, "train")
        parts = {"cv_train": len(train_ds), "train_hours": round(sum(train_ds.lengths) / 16000 / 3600, 3)}
    else:
        train_ds, parts = _v2_train_parts(ctx, processor)

    stage_b1 = "not used"
    if cfg.init == "cpt":
        cpt_dir = ctx.paths.models_dir / "cpt_encoder"
        if not (cpt_dir / "config.json").exists():
            raise FileNotFoundError(f"{cpt_dir} not found - run the 'cpt' stage first.")
        cinfo = load_json(cpt_dir / "cpt_complete.json", default={}) or {}
        if not cinfo.get("completed"):
            log.warning("CPT encoder is NOT marked completed (%s); continuing with the current encoder.",
                        cinfo.get("stop_reason"))
        init = str(cpt_dir)
        if needs_raw:
            ood = _read_manifest(ctx.manifest("ood_train"), required=False)
            b1 = run_stage_b1(cfg, ood_train=_manifest_dataset(ctx, ood, processor), cv_dev=dev, processor=processor,
                              cpt_dir=cpt_dir, models_dir=ctx.paths.models_dir, results_dir=ctx.results,
                              resume=resume, previous_outputs=_previous(ctx))
            stage_b1 = b1["stage_b1"]
            if stage_b1 == "incomplete":
                print("Stage B1 stopped by the time budget - run this stage again.")
                return b1
            init = b1["init_for_b2"]
            save_json({k: v for k, v in b1.items() if k != "newly_initialised"}, ctx.results / variant / "stage_b1.json")
    else:
        init = resolve_model_name(cfg)

    extra = {"variant": variant, "training_set": cfg.training_set, "init": cfg.init, "stage_b1": stage_b1,
             "train_parts": parts, "train_hours": parts.get("train_hours"), "mask_time_prob": mask}
    summary = run_finetune(cfg, tag=variant, train_ds=train_ds, eval_ds=dev, processor=processor, init_from=init,
                           mask_time_prob=mask, max_steps=int(cfg.train.max_steps),
                           model_dir=ctx.paths.models_dir / variant, results_dir=ctx.results / variant,
                           resume=resume, previous_outputs=_previous(ctx), extra_info=extra)
    if summary["status"] != "complete":
        return summary
    model_dir = ctx.paths.models_dir / variant
    metrics = evaluate_variant(ctx, variant, model_dir, {"stage_b1": stage_b1})
    write_model_card(model_dir, variant, cfg, metrics, {**extra, "vocab_sha256": vocab_sha256(ctx.paths.shared_dir)})
    if variant in ("V3", "V4") and cfg.adapt.get("cleanup_b1_after_v4"):
        from .train.finetune import is_complete

        if is_complete(ctx.paths.models_dir / "V3") and is_complete(ctx.paths.models_dir / "V4"):
            shutil.rmtree(ctx.paths.models_dir / "B1", ignore_errors=True)
    return {**summary, "test": metrics}


def _available_variants(ctx: Ctx) -> list[str]:
    from .train.finetune import is_complete

    return [v for v in VARIANTS if is_complete(ctx.paths.models_dir / v)]


def _load_predictions(ctx: Ctx) -> dict[str, pd.DataFrame]:
    out = {}
    for v in VARIANTS:
        p = ctx.results / v / "predictions.csv"
        if p.exists():
            out[v] = pd.read_csv(p, dtype={"speaker": str}, keep_default_na=False)
    return out


def stage_significance(cfg: Config, **_) -> pd.DataFrame:
    from .evaluate.bootstrap import significance_table, system_ci

    ctx = make_ctx(cfg, discover=False)
    preds = _load_predictions(ctx)
    e = cfg.eval
    pairs = [tuple(p) for p in e.pairs]
    table = significance_table(preds, pairs, int(e.bootstrap_resamples), float(e.ci), int(cfg.seed))
    if len(table):
        table.to_csv(ctx.results / "significance.csv", index=False)
    else:
        (ctx.results / "significance.csv").unlink(missing_ok=True)
    cis = {v: {m: system_ci(df, m, int(e.bootstrap_resamples), float(e.ci), int(cfg.seed))
               for m in ("wer", "cer", "ter")} for v, df in preds.items()}
    save_json(cis, ctx.results / "system_cis.json")
    if len(table):
        print(table[["comparison", "metric", "delta", "ci_low", "ci_high", "significant"]].to_string(index=False))
    else:
        print("significance: fewer than two variants evaluated yet")
    return table


def stage_effects(cfg: Config, **_) -> pd.DataFrame | None:
    from .evaluate.effects import effects_table

    ctx = make_ctx(cfg, discover=False)
    metrics = {v: load_json(ctx.results / v / "metrics.json") for v in VARIANTS}
    metrics = {k: v for k, v in metrics.items() if v}
    try:
        table = effects_table(metrics)
    except ValueError as exc:
        print(f"effects: {exc}")
        return None
    table.to_csv(ctx.results / "factorial_effects.csv", index=False)
    print(table.to_string(index=False))
    return table


def stage_evaluate(cfg: Config, variants: list[str] | None = None, force: bool = False, **_) -> pd.DataFrame:
    """(Re)evaluate trained variants on the CV test set; write summary, significance and effects."""
    from .reporting import export_tables

    ctx = make_ctx(cfg, discover=False)
    variants = variants or _available_variants(ctx)
    rows = []
    for v in variants:
        mpath = ctx.results / v / "metrics.json"
        if force or not mpath.exists():
            evaluate_variant(ctx, v, ctx.paths.models_dir / v)
        m = load_json(mpath)
        if m:
            rows.append({k: m.get(k) for k in ("variant", "n_utts", "wer", "cer", "wer_no_tone", "ter",
                                               "ter_matched_words", "train_clips", "train_hours", "gpu_hours",
                                               "mask_time_prob", "checkpoint", "vocab_sha256")})
    summary = pd.DataFrame(rows)
    summary.to_csv(ctx.results / "summary.csv", index=False)
    if len(summary):
        print(summary[["variant", "wer", "cer", "wer_no_tone", "ter"]].to_string(index=False))
    stage_significance(cfg)
    stage_effects(cfg)
    export_tables(ctx.results)
    return summary


def stage_ablation(cfg: Config, resume: bool = True, only: str | None = None, **_) -> pd.DataFrame:
    """Leave-one-out V2 runs (1,500 steps). ``only`` selects a single run for one Kaggle session."""
    from .augment.build_augmented import build_augmented_set
    from .evaluate.ablation import resolve_run, update_table
    from .evaluate.decode import decode_cached, load_model_and_processor
    from .evaluate.metrics import all_metrics
    from .modeling import resolve_model_name
    from .train.datasets import load_cached_split
    from .train.finetune import run_finetune

    _ensure_block(cfg, "ablation", "ablation.yaml")
    ctx = make_ctx(cfg)
    table_path = ctx.results / "ablation.csv"
    tuned = resolve_mask(Config({**cfg, "mask_time_prob": "tuned"}), ctx)
    processor = _processor(ctx)
    dev = _cv_cached(ctx, "dev")
    runs = dict(cfg.ablation.runs)
    names = [only] if only else list(runs)
    unknown = [n for n in names if n not in runs]
    if unknown:
        raise ValueError(f"unknown ablation run(s) {unknown}; choose from {list(runs)}")
    done = pd.read_csv(table_path) if table_path.exists() else pd.DataFrame(columns=["run", "status"])
    table = done
    for name in names:
        if resume and ((done["run"] == name) & (done["status"] == "complete")).any():
            log.info("ablation %s already done", name)
            continue
        spec = resolve_run(name, runs[name], list(cfg.augment.transforms), tuned)
        if spec["aug_set"] != "main":
            train = _read_manifest(ctx.manifest("cv_train"))
            d = cfg.data
            build_augmented_set(train, ctx.paths.augmented_dir / spec["aug_set"], cfg.augment, int(cfg.seed),
                                spec["transforms"], d.min_seconds, d.max_seconds, d.trim_top_db,
                                ctx.found.musan if ctx.found else None, ctx.found.rirs if ctx.found else None,
                                name=spec["aug_set"])
        train_ds, parts = _v2_train_parts(ctx, processor, spec["aug_set"], spec["use_tts"])
        tag = f"abl_{name}"
        mdir = ctx.paths.models_dir / "ablation" / name
        s = run_finetune(cfg, tag=tag, train_ds=train_ds, eval_ds=dev, processor=processor,
                         init_from=resolve_model_name(cfg), mask_time_prob=spec["mask_time_prob"],
                         max_steps=int(cfg.ablation.max_steps), model_dir=mdir, results_dir=ctx.results / "ablation" / name,
                         resume=resume, previous_outputs=_previous(ctx))
        row = {"run": name, "description": spec["description"], "transforms": "+".join(spec["transforms"]),
               "use_tts": spec["use_tts"], "mask_time_prob": spec["mask_time_prob"],
               "max_steps": int(cfg.ablation.max_steps), "train_clips": len(train_ds), "status": s["status"]}
        if s["status"] == "complete":
            model, proc, device = load_model_and_processor(mdir)
            for split in ("dev", "test"):
                preds = decode_cached(model, proc, load_cached_split(ctx.paths.cache_dir / f"cv_{split}"), device,
                                      int(cfg.eval.batch_size))
                m = all_metrics(preds["reference"].tolist(), preds["hypothesis"].tolist())
                row.update({f"{split}_wer": m["wer"], f"{split}_ter": m["ter"]})
                if split == "test":
                    row.update({"test_cer": m["cer"], "test_wer_no_tone": m["wer_no_tone"]})
                    preds.to_csv(ctx.results / "ablation" / name / "test_predictions.csv", index=False)
            del model
            shutil.rmtree(mdir, ignore_errors=True)
        table = update_table(table_path, row)
        if s["status"] != "complete":
            print(f"ablation {name} interrupted by the time budget - run it again.")
            break
    print(table.to_string(index=False) if len(table) else "no ablation results yet")
    return table


def stage_analyse(cfg: Config, **_) -> dict:
    from .evaluate.error_analysis import analyse_variant, load_cis, write_figures
    from .reporting import export_tables

    ctx = make_ctx(cfg, discover=False)
    preds = _load_predictions(ctx)
    out = {}
    for v, df in preds.items():
        out[v] = analyse_variant(df, v, ctx.results / v / "error_analysis", int(cfg.eval.n_examples), int(cfg.seed))
        print(f"{v}: tonal share of WER = {out[v]['tonal_word_error_share']:.2f} points")
    summary = pd.read_csv(ctx.results / "summary.csv") if (ctx.results / "summary.csv").exists() else None
    if summary is not None and not (ctx.results / "system_cis.json").exists() and len(preds):
        stage_significance(cfg)
    figs = write_figures(ctx.results, summary, load_cis(ctx.results))
    tables = export_tables(ctx.results)
    print(f"figures: {figs}\ntables: {tables['md']}")
    return {"variants": list(out), "figures": figs}


def stage_export_tables(cfg: Config, **_) -> dict:
    from .reporting import export_tables

    ctx = make_ctx(cfg, discover=False)
    out = export_tables(ctx.results)
    print(f"written: {', '.join(str(p) for p in out.values())}")
    return {k: str(v) for k, v in out.items()}


# --------------------------------------------------------------------------------------
# dispatcher
# --------------------------------------------------------------------------------------
LIGHT_OVERRIDES = {
    "paths.work_dir": "${paths.work_root}/light", "run_name": "light", "train.max_steps": 300,
    "train.eval_steps": 100, "train.save_steps": 100, "mask_tuning.skip": True, "cpt.max_steps": 300,
    "cpt.max_total_hours": 1.0, "adapt.b1_max_steps": 200, "tts.max_per_real_clip": 0.2,
}

SEQUENCES = {
    "smoke": ["prepare", "augment", "tune_mask", "cpt", "V1", "V2", "V3", "V4", "evaluate", "analyse"],
    "all_light": ["prepare", "V1", "augment", "tune_mask", "V2", "cpt", "V3", "V4", "evaluate", "analyse"],
}

STAGE_FUNCTIONS: dict[str, Callable] = {
    "audit": stage_audit, "prepare": stage_prepare, "augment": stage_augment, "tune_mask": stage_tune_mask,
    "cpt": stage_cpt, "V1": stage_train, "V2": stage_train, "V3": stage_train, "V4": stage_train,
    "evaluate": stage_evaluate, "significance": stage_significance, "effects": stage_effects,
    "ablation": stage_ablation, "analyse": stage_analyse, "export_tables": stage_export_tables,
}
NOTEBOOK_STAGES = ["smoke", "prepare", "V1", "augment", "tune_mask", "V2", "cpt", "V3", "V4", "evaluate",
                   "ablation", "analyse", "all_light"]

PLANS = {
    "prepare": "discover data -> load CV/SLR86/ÌròyìnSpeech -> normalise (NFC) -> drop digits -> leakage filter -> "
               "tone gate -> shared vocab (hash) -> cache CV train/dev/test -> manifests -> audit + leakage report",
    "augment": "pitch check (tempo keeps F0) -> 2 augmented copies per CV train clip (FLAC) -> MMS-TTS coverage "
               "check -> TTS clips (<= 1 per real clip) -> listening sample",
    "tune_mask": "3 x V2 runs (1,000 steps) with mask_time_prob 0.05/0.075/0.10 -> choose lowest dev WER",
    "cpt": "continued wav2vec 2.0 pretraining on unlabelled audio (checkpoint every 500 steps, resumable)",
    "V1": "fine-tune XLS-R on CV train (3,000 steps) -> best dev WER -> test evaluation",
    "V2": "fine-tune XLS-R on CV train + augmented + TTS (tuned mask) -> test evaluation",
    "V3": "CPT encoder -> B1 (OOD labelled) -> B2 (CV train) -> test evaluation",
    "V4": "CPT encoder -> B1 (shared with V3) -> B2 on V2 training set (tuned mask) -> test evaluation",
    "evaluate": "decode test set for every trained variant -> summary.csv -> bootstrap significance -> effects",
    "ablation": "leave-one-out V2 runs (1,500 steps each), one per session with ABLATION_ONLY",
    "analyse": "error analysis per variant, figures (300 dpi), thesis tables",
    "smoke": "tiny model end-to-end: " + " -> ".join(SEQUENCES["smoke"]),
    "all_light": "short real-data sanity run: " + " -> ".join(SEQUENCES["all_light"]),
}


def run_stage(stage: str, overrides: dict | None = None, smoke: bool = False, resume: bool = True,
              only: str | None = None, dry_run: bool = False, extra_files: list | None = None) -> dict:
    """Run one stage, a ``+``-joined list of stages (``augment+tune_mask``) or a sequence (smoke/all_light)."""
    if "+" in stage:
        return {s: run_stage(s, overrides, smoke, resume, only, dry_run, extra_files) for s in stage.split("+")}
    if stage in SEQUENCES:
        ov = dict(overrides or {})
        if stage == "all_light":
            ov = {**LIGHT_OVERRIDES, **ov}
        results = {}
        for s in SEQUENCES[stage]:
            print(f"\n{'=' * 30} {stage}: {s} {'=' * 30}")
            results[s] = run_stage(s, ov, smoke or stage == "smoke", resume, only, dry_run, extra_files)
        return results
    if stage not in STAGE_FUNCTIONS:
        raise ValueError(f"unknown stage {stage!r}; choose from {sorted(set(STAGE_FUNCTIONS) | set(SEQUENCES))}")
    if dry_run:
        print(f"[dry-run] {stage}: {PLANS.get(stage, STAGE_FUNCTIONS[stage].__doc__ or '')}")
        return {"stage": stage, "dry_run": True, "function": STAGE_FUNCTIONS[stage].__name__}
    cfg = stage_config(stage, overrides, smoke, extra_files)
    if stage in ("cpt",):
        _ensure_block(cfg, "cpt", "cpt.yaml")
    kwargs = {"resume": resume, "only": only}
    if stage in VARIANTS:
        kwargs["variant"] = stage
    result = STAGE_FUNCTIONS[stage](cfg, **kwargs)
    return result if isinstance(result, dict) else {"stage": stage, "done": True}


def next_step_message(stage: str) -> str:
    nxt, attach = NEXT_STAGE.get(stage, ("see docs/KAGGLE_GUIDE.md", ""))
    return f"NEXT: set STAGE = \"{nxt}\". {attach}"


def environment_report(cfg: Config | None = None) -> dict:
    from .utils import disk_free_gb

    cfg = cfg or stage_config("prepare")
    info = gpu_info()
    info["disk_free_gb_work"] = disk_free_gb(cfg.paths.work_root)
    info["disk_free_gb_tmp"] = disk_free_gb(cfg.paths.tmp_dir)
    return info


def safe_run(fn: Callable, *a, **k):
    try:
        return fn(*a, **k)
    except Exception:
        traceback.print_exc()
        raise


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/reporting.py
"""Export result tables for the thesis (Markdown, CSV and LaTeX)."""
from __future__ import annotations

from pathlib import Path

import pandas as pd

from .utils import load_json

VARIANT_NAMES = {"V1": "Source-only baseline", "V2": "Augmentation only", "V3": "Domain adaptation only",
                 "V4": "Hybrid (augmentation + adaptation)"}


def _fmt(df: pd.DataFrame, digits: int = 2) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: f"{v:.{digits}f}" if pd.notna(v) else "")
    return out


def _md(df: pd.DataFrame) -> str:
    df = _fmt(df)
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)


def _latex(df: pd.DataFrame, caption: str, label: str) -> str:
    df = _fmt(df)
    cols = list(df.columns)
    esc = lambda s: str(s).replace("_", "\\_").replace("%", "\\%").replace("&", "\\&")  # noqa: E731
    lines = ["\\begin{table}[ht]", "\\centering", f"\\caption{{{esc(caption)}}}", f"\\label{{{label}}}",
             "\\begin{tabular}{" + "l" * len(cols) + "}", "\\hline", " & ".join(esc(c) for c in cols) + " \\\\",
             "\\hline"]
    for _, r in df.iterrows():
        lines.append(" & ".join(esc(r[c]) for c in cols) + " \\\\")
    lines += ["\\hline", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)


def export_tables(results_dir: str | Path) -> dict[str, Path]:
    results_dir = Path(results_dir)
    tables: list[tuple[str, str, pd.DataFrame]] = []
    summary = results_dir / "summary.csv"
    if summary.exists():
        s = pd.read_csv(summary)
        s.insert(1, "description", s["variant"].map(VARIANT_NAMES))
        keep = [c for c in ("variant", "description", "wer", "cer", "wer_no_tone", "ter", "ter_matched_words",
                            "n_utts", "train_clips", "train_hours", "gpu_hours", "mask_time_prob") if c in s]
        tables.append(("Table 4.1: Test-set results of the four variants (%)", "tab:main_results", s[keep]))
    for fname, caption, label in [
        ("significance.csv", "Table 4.2: Speaker-block bootstrap significance (difference A - B, percentage points)",
         "tab:significance"),
        ("factorial_effects.csv", "Table 4.3: Factorial main effects and interaction (percentage points)",
         "tab:effects"),
        ("factorial_effects_ci.csv", "Table 4.3b: Factorial effects with speaker-bootstrap 95% CIs (percentage points)",
         "tab:effects_ci"),
        ("ablation.csv", "Table 4.4: Ablation study (V2, 1,500 steps each)", "tab:ablation"),
    ]:
        p = results_dir / fname
        if p.exists():
            try:
                df = pd.read_csv(p)
            except pd.errors.EmptyDataError:  # e.g. significance with only one trained variant
                continue
            if df.empty:
                continue
            if fname == "significance.csv":
                df = df[[c for c in ("comparison", "metric", "value_a", "value_b", "delta", "ci_low", "ci_high",
                                     "significant", "n_speakers") if c in df]]
            tables.append((caption, label, df))
    audit = load_json(results_dir / "corpus_audit.json")
    if audit:
        rows = []
        for corpus, splits in audit["corpora"].items():
            for split, st in splits.items():
                if not split.startswith("_"):
                    rows.append({"corpus": corpus, "split": split, "clips": st["clips_kept"], "hours": st["hours"],
                                 "speakers": st["speakers"], "tone_marked_%": 100 * (st["share_with_tone_mark"] or 0)})
        tables.insert(0, ("Table 4.0: Corpus audit", "tab:audit", pd.DataFrame(rows)))

    md = ["# Thesis tables", ""]
    tex = []
    csv_parts = []
    for caption, label, df in tables:
        md += [f"## {caption}", "", _md(df), ""]
        tex.append(_latex(df, caption, label))
        csv_parts.append(df.assign(table=label))
    out = {"md": results_dir / "thesis_tables.md", "csv": results_dir / "thesis_tables.csv",
           "tex": results_dir / "thesis_tables.tex"}
    out["md"].write_text("\n".join(md) if tables else "# Thesis tables\n\nNo results yet.\n", encoding="utf-8")
    out["tex"].write_text("\n\n".join(tex), encoding="utf-8")
    (pd.concat(csv_parts, ignore_index=True) if csv_parts else pd.DataFrame()).to_csv(out["csv"], index=False)
    return out


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/text.py
"""Yoruba text normalisation and tone utilities.

Normalisation reproduces the original Colab baseline exactly (NFC, lower-case, remove the
same punctuation set, collapse spaces). Extra helpers support tone-aware evaluation.
"""
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from typing import Iterable

GRAVE, ACUTE, MACRON, SUBDOT = "̀", "́", "̄", "̣"
TONE_MARKS = {GRAVE, ACUTE, MACRON}  # grave (low), acute (high), macron (mid)
# Same character class as the original notebook (hyphens and en/em dashes are removed).
PUNCT = r"[\,\?\.\!\;\:\"\“\”\‘\’\…\(\)\[\]\{\}\–\—\«\»\-]"
_PUNCT_RE = re.compile(PUNCT)
_SPACE_RE = re.compile(r"\s+")
_DIGIT_RE = re.compile(r"\d")

VOWELS = set("aeiou")
NASALS = set("nm")
TONE_LABELS = ("H", "M", "L")
_MARK_TO_LABEL = {ACUTE: "H", GRAVE: "L", MACRON: "M"}


def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)


def strip_tones(s: str) -> str:
    """Remove tone marks (grave/acute/macron) but keep the sub-dot of ẹ, ọ, ṣ."""
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if ch not in TONE_MARKS)
    return unicodedata.normalize("NFC", s)


def normalize_text(s: str, keep_tones: bool = True) -> str:
    s = nfc(s).lower()
    s = _PUNCT_RE.sub("", s)
    s = _SPACE_RE.sub(" ", s).strip()
    return nfc(s if keep_tones else strip_tones(s))


def has_digits(s: str) -> bool:
    return bool(_DIGIT_RE.search(s))


def has_tone_mark(s: str) -> bool:
    return any(ch in TONE_MARKS for ch in unicodedata.normalize("NFD", s))


def has_subdot(s: str) -> bool:
    return SUBDOT in unicodedata.normalize("NFD", s)


def count_intraword(s: str) -> dict[str, int]:
    """Count apostrophes and hyphens/dashes that sit inside words (logged for the thesis)."""
    return {
        "apostrophes": len(re.findall(r"\w['’‘]\w", s)),
        "hyphens": len(re.findall(r"\w[-–—]\w", s)),
    }


def _tbu_units(text: str) -> list[tuple[str, str | None, bool]]:
    """Walk NFD text and return ``(base, label, is_tbu)`` for every vowel/nasal letter.

    ``label`` is H/L/M from the following combining marks (None for a nasal without a mark).
    """
    s = unicodedata.normalize("NFD", text.lower())
    units: list[tuple[str, str | None, bool]] = []
    i, n = 0, len(s)
    while i < n:
        ch = s[i]
        j = i + 1
        marks = []
        while j < n and unicodedata.combining(s[j]):
            marks.append(s[j])
            j += 1
        if ch in VOWELS or ch in NASALS:
            label = None
            for m in marks:
                if m in _MARK_TO_LABEL:
                    label = _MARK_TO_LABEL[m]
            if ch in VOWELS:
                units.append((ch, label or "M", True))
            else:
                units.append((ch, label, label is not None))
        i = j
    return units


def tone_sequence(text: str) -> list[str]:
    """One tone label (H/M/L) per tone-bearing unit (TBU).

    TBUs are the vowels a e i o u (ẹ/ọ count as e/o) plus any n/m carrying a tone mark
    (syllabic nasal). Acute -> H, grave -> L, macron or no mark -> M.
    Examples: ọkọ́ -> [M, H]; ń lọ -> [H, M]; àwọn -> [L, M].
    """
    return [label for _, label, is_tbu in _tbu_units(text) if is_tbu]


def word_tone_pairs(ref_word: str, hyp_word: str) -> list[tuple[str, str]]:
    """Position-wise (ref, hyp) tone labels for two words with equal tone-stripped forms.

    Units are vowels and nasals; a nasal counts if the reference marks it as a TBU.
    A nasal unmarked in the hypothesis is read as M.
    """
    r_units, h_units = _tbu_units(ref_word), _tbu_units(hyp_word)
    pairs = []
    for (rb, rl, r_tbu), (_hb, hl, _h_tbu) in zip(r_units, h_units):
        if r_tbu:
            pairs.append((rl or "M", hl or "M"))
    return pairs


def char_inventory(texts: Iterable[str]) -> dict[str, int]:
    """Character counts (NFC code points), sorted by frequency."""
    c = Counter()
    for t in texts:
        c.update(nfc(t))
    return dict(sorted(c.items(), key=lambda kv: (-kv[1], kv[0])))


def describe_char(ch: str) -> str:
    try:
        return unicodedata.name(ch)
    except ValueError:
        return f"U+{ord(ch):04X}"


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/train/__init__.py
"""CTC fine-tuning: collator, datasets, callbacks and the Trainer wrapper."""


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/train/callbacks.py
"""Trainer callbacks and checkpoint/resume helpers."""
from __future__ import annotations

import re
import shutil
import time
from pathlib import Path

from transformers import TrainerCallback

from ..utils import get_logger

log = get_logger(__name__)


class TimeBudgetCallback(TrainerCallback):
    """Stops training gracefully (and saves a checkpoint) before Kaggle's 12-hour limit."""

    def __init__(self, hours: float):
        self.budget = float(hours) * 3600
        self.start = time.time()
        self.triggered = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.start > self.budget and not self.triggered:
            log.warning("Time budget of %.2f h reached at step %d: saving and stopping.",
                        self.budget / 3600, state.global_step)
            self.triggered = True
            control.should_save = True
            control.should_training_stop = True
        return control


class MetricsLogCallback(TrainerCallback):
    """Prints dev metrics in a compact form."""

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            keys = [k for k in ("eval_wer", "eval_cer", "eval_wer_no_tone", "eval_loss") if k in metrics]
            log.info("step %d | %s", state.global_step, " ".join(f"{k}={metrics[k]:.4f}" for k in keys))


_CKPT = re.compile(r"^checkpoint-(\d+)$")


def latest_checkpoint(*dirs: str | Path) -> Path | None:
    """Most advanced ``checkpoint-N`` folder (with a trainer_state.json) across ``dirs``."""
    best: tuple[int, Path] | None = None
    for d in dirs:
        d = Path(d)
        if not d.exists():
            continue
        for sub in d.iterdir():
            m = _CKPT.match(sub.name)
            if m and sub.is_dir() and (sub / "trainer_state.json").exists():
                step = int(m.group(1))
                if best is None or step > best[0]:
                    best = (step, sub)
    return best[1] if best else None


def find_resume_checkpoint(output_dir: Path, previous_outputs: list[Path], rel: str) -> Path | None:
    """Checkpoint in ``output_dir`` or, failing that, in an attached previous notebook output.

    A checkpoint found in a (read-only) previous output is copied into ``output_dir`` first so the
    Trainer can rotate checkpoints and find the best one at the same absolute path.
    """
    ckpt = latest_checkpoint(output_dir)
    if ckpt is not None:
        return ckpt
    for prev in previous_outputs:
        src_dir = Path(prev) / rel
        ckpt = latest_checkpoint(src_dir)
        if ckpt is not None:
            output_dir.mkdir(parents=True, exist_ok=True)
            for sub in src_dir.iterdir():
                if _CKPT.match(sub.name):
                    shutil.copytree(sub, output_dir / sub.name, dirs_exist_ok=True)
            log.info("Copied checkpoints from previous output %s", src_dir)
            return latest_checkpoint(output_dir)
    return None


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/train/collator.py
"""Data collator for CTC fine-tuning (identical to the original baseline notebook)."""
from __future__ import annotations

from dataclasses import dataclass
from typing import Any


@dataclass
class DataCollatorCTCWithPadding:
    processor: Any  # transformers.Wav2Vec2Processor

    def __call__(self, features):
        inputs = [{"input_values": f["input_values"]} for f in features]
        labels = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, padding=True, return_tensors="pt")
        lab = self.processor.tokenizer.pad(labels, padding=True, return_tensors="pt")
        batch["labels"] = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        return batch


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/train/datasets.py
"""Training datasets.

* Common Voice (small) is preprocessed once into a Hugging Face ``Dataset`` with
  ``input_values``, ``input_length`` and ``labels`` and saved under ``work_dir/cache/``.
* Large or generated corpora (OOD, augmented FLAC, TTS) are read from manifests and loaded
  on the fly, so no float arrays are written to disk for them.

Every dataset exposes ``lengths`` (samples per item) so the Trainer can group by length.
"""
from __future__ import annotations

import uuid
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch

from ..audio import SR, duration_reason, probe, safe_load_trimmed
from ..utils import get_logger

log = get_logger(__name__)

CACHE_META_COLUMNS = ["path", "speaker", "gender", "age", "text_tone", "text_notone", "duration", "corpus"]


# --------------------------------------------------------------------------------------
# Common Voice cache
# --------------------------------------------------------------------------------------
def build_cached_split(rows: pd.DataFrame, processor, out_dir: str | Path, tmp_dir: str | Path,
                       min_s: float, max_s: float, top_db: float, text_column: str = "text_tone"):
    """Decode, trim, duration-filter and feature-extract ``rows``; save a HF dataset to ``out_dir``.

    Returns ``(dataset, drop_log)`` where ``drop_log`` maps path -> drop reason.
    """
    from datasets import Dataset, Features, Sequence, Value

    drop_log: dict[str, str] = {}
    fe, tok = processor.feature_extractor, processor.tokenizer
    records = rows.to_dict("records")

    def gen():
        for r in records:
            audio = safe_load_trimmed(r["path"], SR, top_db)
            if audio is None:
                drop_log[r["path"]] = "corrupt"
                continue
            dur = len(audio) / SR
            reason = duration_reason(dur, min_s, max_s)
            if reason:
                drop_log[r["path"]] = reason
                continue
            iv = fe(audio, sampling_rate=SR).input_values[0].astype(np.float32)
            yield {
                "input_values": iv,
                "input_length": int(len(iv)),
                "labels": tok(r[text_column]).input_ids,
                "path": str(r["path"]), "speaker": str(r.get("speaker", "unknown")),
                "gender": str(r.get("gender", "")), "age": str(r.get("age", "")),
                "text_tone": str(r["text_tone"]), "text_notone": str(r["text_notone"]),
                "duration": float(dur), "corpus": str(r.get("corpus", "")),
            }

    features = Features({
        "input_values": Sequence(Value("float32")), "input_length": Value("int64"),
        "labels": Sequence(Value("int32")), "path": Value("string"), "speaker": Value("string"),
        "gender": Value("string"), "age": Value("string"), "text_tone": Value("string"),
        "text_notone": Value("string"), "duration": Value("float32"), "corpus": Value("string"),
    })
    gen_cache = Path(tmp_dir) / "hf_gen" / uuid.uuid4().hex
    try:
        ds = Dataset.from_generator(gen, features=features, cache_dir=str(gen_cache))
    except ValueError as exc:
        if "corresponds to no data" not in str(exc):
            raise
        reasons = {r: list(drop_log.values()).count(r) for r in set(drop_log.values())}
        raise RuntimeError(f"No usable audio clip in this split ({len(records)} listed, dropped: {reasons}). "
                           "Check that the Common Voice 'clips' folder was uploaded completely.") from exc
    out_dir = Path(out_dir)
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(out_dir))
    return ds, drop_log


def load_cached_split(path: str | Path):
    from datasets import load_from_disk

    return load_from_disk(str(path))


def arrow_row(hf_dataset, i: int, columns=("input_values", "labels")) -> dict:
    """Read one row straight from the Arrow table.

    Deliberately avoids ``with_format("numpy")``: in ``datasets`` that formatter imports
    ``torchvision.io.VideoReader``, which newer torchvision builds (e.g. on Kaggle) no longer have.
    """
    table = hf_dataset.with_format("arrow")[int(i): int(i) + 1]
    out = {}
    if "input_values" in columns:
        out["input_values"] = table.column("input_values")[0].values.to_numpy(zero_copy_only=False).astype(np.float32)
    if "labels" in columns:
        out["labels"] = table.column("labels")[0].values.to_numpy(zero_copy_only=False).astype(np.int64).tolist()
    return out


class CachedASRDataset(torch.utils.data.Dataset):
    """Wraps a cached HF dataset; returns ``input_values`` (np.float32) and ``labels``."""

    def __init__(self, hf_dataset, limit: int | None = None):
        if limit is not None and limit < len(hf_dataset):
            hf_dataset = hf_dataset.select(range(limit))
        self.hf = hf_dataset
        self.lengths = list(hf_dataset["input_length"])

    def __len__(self) -> int:
        return len(self.hf)

    def __getitem__(self, i: int) -> dict:
        return arrow_row(self.hf, int(i))

    def metadata(self) -> pd.DataFrame:
        cols = [c for c in CACHE_META_COLUMNS if c in self.hf.column_names]
        return self.hf.select_columns(cols).to_pandas()


class ManifestASRDataset(torch.utils.data.Dataset):
    """Loads audio on the fly from a manifest with ``path, text_tone, duration`` columns."""

    def __init__(self, manifest: pd.DataFrame, processor, top_db: float = 30, text_column: str = "text_tone",
                 trim: bool = True):
        self.df = manifest.reset_index(drop=True)
        self.processor = processor
        self.top_db = top_db
        self.trim = trim
        self.text_column = text_column
        self.lengths = [int(float(d) * SR) for d in self.df["duration"]]

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, i: int) -> dict:
        r = self.df.iloc[int(i)]
        audio = safe_load_trimmed(r["path"], SR, self.top_db if self.trim else 200)
        if audio is None or len(audio) < int(0.1 * SR):
            audio = np.zeros(int(0.5 * SR), dtype=np.float32)  # corrupt file: silent stand-in, label kept short
            labels = []
        else:
            labels = self.processor.tokenizer(r[self.text_column]).input_ids
        iv = self.processor.feature_extractor(audio, sampling_rate=SR).input_values[0].astype(np.float32)
        return {"input_values": iv, "labels": labels}


class ConcatASRDataset(torch.utils.data.ConcatDataset):
    def __init__(self, datasets: Iterable[torch.utils.data.Dataset]):
        datasets = [d for d in datasets if d is not None and len(d) > 0]
        super().__init__(datasets)
        self.lengths = [n for d in datasets for n in d.lengths]


def probe_manifest(df: pd.DataFrame, min_s: float, max_s: float) -> pd.DataFrame:
    """Add ``duration``, ``sample_rate`` and ``drop_reason`` columns using file headers only."""
    durs, srs, reasons = [], [], []
    for p in df["path"]:
        info = probe(p)
        if info is None:
            durs.append(np.nan); srs.append(np.nan); reasons.append("corrupt")
            continue
        d, sr = info
        durs.append(d); srs.append(sr); reasons.append(duration_reason(d, min_s, max_s))
    out = df.copy()
    out["duration"], out["sample_rate"], out["drop_reason"] = durs, srs, reasons
    return out


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/train/finetune.py
"""CTC fine-tuning with the Hugging Face Trainer (resumable, time-budgeted, GPU-time logged)."""
from __future__ import annotations

import shutil
from pathlib import Path

import pandas as pd
from transformers import Trainer, TrainingArguments
from transformers.trainer_pt_utils import LengthGroupedSampler

from ..config import Config
from ..evaluate.metrics import make_compute_metrics, preprocess_logits
from ..modeling import load_ctc_model
from ..utils import RunInfo, get_logger, has_cuda, load_json, now_iso, save_json, seed_everything
from .callbacks import MetricsLogCallback, TimeBudgetCallback, find_resume_checkpoint
from .collator import DataCollatorCTCWithPadding

log = get_logger(__name__)
COMPLETE_MARKER = "training_complete.json"


class CTCTrainer(Trainer):
    """Trainer that groups by length using the dataset's ``lengths`` attribute (no audio decoding)."""

    def _get_train_sampler(self, *args, **kwargs):
        ds = args[0] if args else kwargs.get("train_dataset")
        ds = ds if ds is not None else self.train_dataset
        if self.args.group_by_length and hasattr(ds, "lengths"):
            return LengthGroupedSampler(self.args.train_batch_size * self.args.gradient_accumulation_steps,
                                        dataset=ds, lengths=list(ds.lengths))
        return super()._get_train_sampler(*args, **kwargs)


def training_arguments(cfg: Config, output_dir: Path, max_steps: int) -> TrainingArguments:
    t = cfg.train
    use_fp16 = bool(t.fp16) and has_cuda()
    eval_steps = min(int(t.eval_steps), int(max_steps))
    save_steps = min(int(t.save_steps), int(max_steps))
    return TrainingArguments(
        output_dir=str(output_dir),
        group_by_length=bool(t.group_by_length), length_column_name="input_length",
        per_device_train_batch_size=int(t.per_device_train_batch_size),
        gradient_accumulation_steps=int(t.gradient_accumulation_steps),
        per_device_eval_batch_size=int(t.per_device_eval_batch_size),
        eval_strategy="steps", eval_steps=eval_steps, save_strategy="steps", save_steps=save_steps,
        logging_steps=int(t.logging_steps), max_steps=int(max_steps),
        learning_rate=float(t.learning_rate), warmup_steps=int(t.warmup_steps), lr_scheduler_type="linear",
        adam_beta1=float(t.adam_beta1), adam_beta2=float(t.adam_beta2), adam_epsilon=float(t.adam_epsilon),
        weight_decay=float(t.weight_decay), optim="adamw_torch",
        fp16=use_fp16, save_total_limit=int(t.save_total_limit),
        load_best_model_at_end=True, metric_for_best_model=t.metric_for_best_model, greater_is_better=False,
        report_to="none", seed=int(cfg.seed), data_seed=int(cfg.seed),
        dataloader_num_workers=int(t.dataloader_num_workers), remove_unused_columns=False,
        save_safetensors=True, disable_tqdm=False, logging_first_step=True,
    )


def is_complete(model_dir: str | Path) -> bool:
    return (Path(model_dir) / COMPLETE_MARKER).exists()


def run_finetune(cfg: Config, *, tag: str, train_ds, eval_ds, processor, init_from: str, mask_time_prob: float,
                 max_steps: int, model_dir: str | Path, results_dir: str | Path, resume: bool = True,
                 previous_outputs: list[Path] | None = None, extra_info: dict | None = None) -> dict:
    """Fine-tune a CTC model and save the best (dev-WER) checkpoint to ``model_dir``.

    Returns ``{"status": "complete"|"incomplete", ...}``. An incomplete run (time budget hit)
    leaves checkpoints in ``checkpoints/<tag>`` so the next session can resume.
    """
    model_dir, results_dir = Path(model_dir), Path(results_dir)
    if resume and is_complete(model_dir):
        log.info("[%s] already complete at %s - skipping training", tag, model_dir)
        return {"status": "complete", "skipped": True, **(load_json(model_dir / COMPLETE_MARKER) or {})}

    seed_everything(int(cfg.seed))
    work_dir = Path(cfg.paths.work_dir)
    ckpt_dir = Path(cfg.paths.checkpoints_dir) / tag
    rel = (Path(cfg.paths.checkpoints_dir) / tag).relative_to(work_dir) if \
        str(Path(cfg.paths.checkpoints_dir)).startswith(str(work_dir)) else Path("checkpoints") / tag
    resume_ckpt = find_resume_checkpoint(ckpt_dir, previous_outputs or [], str(rel)) if resume else None
    if not resume and ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)

    model, load_info = load_ctc_model(init_from, cfg, processor, mask_time_prob)
    args = training_arguments(cfg, ckpt_dir, max_steps)
    budget = TimeBudgetCallback(float(cfg.train.time_budget_hours))
    trainer = CTCTrainer(
        model=model, args=args, data_collator=DataCollatorCTCWithPadding(processor),
        train_dataset=train_ds, eval_dataset=eval_ds, processing_class=processor,
        compute_metrics=make_compute_metrics(processor.tokenizer),
        preprocess_logits_for_metrics=preprocess_logits, callbacks=[budget, MetricsLogCallback()],
    )
    run = RunInfo(results_dir / "run_info.json", stage=tag)
    start_step = 0
    if resume_ckpt is not None:
        log.info("[%s] resuming from %s", tag, resume_ckpt)
        start_step = int((load_json(resume_ckpt / "trainer_state.json") or {}).get("global_step", 0))
    log.info("[%s] training: %d clips, max_steps=%d, mask_time_prob=%.3f, init=%s",
             tag, len(train_ds), max_steps, mask_time_prob, init_from)
    trainer.train(resume_from_checkpoint=str(resume_ckpt) if resume_ckpt else None)

    state = trainer.state
    complete = state.global_step >= max_steps and not budget.triggered
    history = pd.DataFrame(state.log_history)
    results_dir.mkdir(parents=True, exist_ok=True)
    history.to_csv(results_dir / "train_log.csv", index=False)
    info = run.finish(steps=state.global_step - start_step, global_step=state.global_step,
                      train_clips=len(train_ds), max_steps=max_steps, mask_time_prob=mask_time_prob,
                      init_from=str(init_from), status="complete" if complete else "incomplete")
    summary = {
        "tag": tag, "status": "complete" if complete else "incomplete", "global_step": state.global_step,
        "max_steps": max_steps, "best_dev_wer": state.best_metric, "best_checkpoint": state.best_model_checkpoint,
        "mask_time_prob": mask_time_prob, "init_from": str(init_from), "train_clips": len(train_ds),
        "gpu_hours_total": info["total_gpu_hours"], "finished_at": now_iso(),
        "newly_initialised": load_info.get("missing_keys", []), **(extra_info or {}),
    }
    if not complete:
        save_json(summary, results_dir / "run_status.json")
        log.warning("[%s] stopped at step %d/%d (time budget). Run the same stage again with RESUME=True.",
                    tag, state.global_step, max_steps)
        return summary

    model_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(model_dir))  # best checkpoint (load_best_model_at_end=True)
    processor.save_pretrained(str(model_dir))
    save_json(summary, model_dir / COMPLETE_MARKER)
    save_json(summary, results_dir / "run_status.json")
    if not cfg.train.get("keep_checkpoints", False):
        shutil.rmtree(ckpt_dir, ignore_errors=True)
    log.info("[%s] done: best dev WER=%s -> %s", tag, state.best_metric, model_dir)
    del trainer, model
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass
    return summary


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/utils.py
"""Small helpers: seeding, logging, timing, JSON I/O, hashing and GPU information."""
from __future__ import annotations

import hashlib
import json
import logging
import os
import platform
import random
import shutil
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator

import numpy as np


def seed_everything(seed: int = 42) -> None:
    """Seed Python, NumPy and PyTorch (CPU + CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def get_logger(name: str = "yoruba_asr") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("[%(asctime)s %(levelname)s %(name)s] %(message)s", "%H:%M:%S"))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
        logger.propagate = False
    return logger


def _json_default(obj: Any):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (set, tuple)):
        return list(obj)
    return str(obj)


def save_json(obj: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")
    os.replace(tmp, path)
    return path


def load_json(path: str | Path, default: Any = None) -> Any:
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def gpu_info() -> dict:
    """Name/memory of the GPU, or a CPU description."""
    info: dict[str, Any] = {"platform": platform.platform(), "python": platform.python_version()}
    try:
        import torch

        info["torch"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            info["gpu_name"] = props.name
            info["gpu_memory_gb"] = round(props.total_memory / 1024**3, 2)
            info["cuda"] = torch.version.cuda
        else:
            info["gpu_name"] = "cpu"
    except ImportError:
        info["gpu_name"] = "cpu"
    try:
        import transformers

        info["transformers"] = transformers.__version__
    except ImportError:
        pass
    return info


def disk_free_gb(path: str | Path) -> float:
    p = Path(path)
    while not p.exists() and p != p.parent:
        p = p.parent
    return round(shutil.disk_usage(p).free / 1024**3, 2)


def has_cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except ImportError:
        return False


class RunInfo:
    """Records wall-clock time, GPU name, peak GPU memory and steps/sec for one stage.

    Sessions accumulate in ``run_info.json`` so multi-session stages report the total.
    """

    def __init__(self, path: str | Path, stage: str):
        self.path = Path(path)
        self.stage = stage
        self.start = time.time()
        self.extra: dict[str, Any] = {}
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
        except ImportError:
            pass

    def finish(self, steps: int | None = None, **extra: Any) -> dict:
        wall = time.time() - self.start
        info = gpu_info()
        peak = 0.0
        try:
            import torch

            if torch.cuda.is_available():
                peak = torch.cuda.max_memory_allocated() / 1024**3
        except ImportError:
            pass
        session = {
            "stage": self.stage,
            "finished_at": now_iso(),
            "wall_seconds": round(wall, 1),
            "gpu_name": info.get("gpu_name"),
            "peak_gpu_memory_gb": round(peak, 2),
            "gpu_hours": round(wall / 3600, 3) if info.get("cuda_available") else 0.0,
            "steps": steps,
            "steps_per_sec": round(steps / wall, 4) if steps and wall > 0 else None,
            **self.extra,
            **extra,
        }
        previous = load_json(self.path, default={}) or {}
        sessions = previous.get("sessions", []) + [session]
        out = {
            "stage": self.stage,
            "sessions": sessions,
            "total_wall_hours": round(sum(s["wall_seconds"] for s in sessions) / 3600, 3),
            "total_gpu_hours": round(sum(s.get("gpu_hours") or 0 for s in sessions), 3),
            "environment": info,
        }
        save_json(out, self.path)
        return out


@contextmanager
def timed(label: str, logger: logging.Logger | None = None) -> Iterator[None]:
    t0 = time.time()
    yield
    msg = f"{label}: {time.time() - t0:.1f}s"
    (logger or get_logger()).info(msg)


def human_hours(seconds: float) -> str:
    return f"{seconds / 3600:.2f} h"


In [ ]:
%%writefile {CODE_DIR}/src/yoruba_asr/vocab.py
"""Shared character vocabulary and Wav2Vec2 processor.

The vocabulary is built ONCE (before any variant is trained) and every variant loads the same
``shared/vocab.json``; its SHA-256 is stored in ``results/vocab_info.json`` and checked at load.
"""
from __future__ import annotations

import json
import unicodedata
from pathlib import Path
from typing import Iterable

from .text import ACUTE, GRAVE, MACRON, SUBDOT, nfc
from .utils import load_json, save_json, sha256_file

BASE_LETTERS = list("abdefghijklmnoprstuwy") + ["ẹ", "ọ", "ṣ"]
TONE_BASES = ["a", "e", "i", "o", "u", "ẹ", "ọ", "n", "m"]
COMBINING = [GRAVE, ACUTE, MACRON, SUBDOT]
WORD_DELIM, UNK, PAD = "|", "[UNK]", "[PAD]"


class VocabMismatchError(RuntimeError):
    pass


def canonical_charset() -> set[str]:
    """Yoruba letters, every precomposed toned vowel/nasal and the standalone combining marks."""
    chars = set(BASE_LETTERS)
    for base in TONE_BASES:
        for mark in (GRAVE, ACUTE, MACRON):
            s = nfc(base + mark)
            if len(s) == 1:
                chars.add(s)
            else:  # e.g. ẹ́ has no precomposed form: base + combining mark
                chars.update(s)
    chars.update(COMBINING)
    return chars


def build_vocab(texts: Iterable[str]) -> dict[str, int]:
    """Union of characters in ``texts`` (normalised NFC) and the canonical set; space -> ``|``."""
    chars = set(canonical_charset())
    for t in texts:
        chars.update(nfc(t))
    chars.add(" ")
    chars.discard(WORD_DELIM)
    ordered = sorted(chars)
    vocab = {c: i for i, c in enumerate(ordered)}
    vocab[WORD_DELIM] = vocab.pop(" ")
    vocab = {c: i for i, c in enumerate(sorted(vocab, key=lambda c: vocab[c]))}
    vocab[UNK] = len(vocab)
    vocab[PAD] = len(vocab)
    return vocab


def save_vocab(vocab: dict[str, int], shared_dir: str | Path, results_dir: str | Path,
               sources: dict | None = None) -> dict:
    shared_dir, results_dir = Path(shared_dir), Path(results_dir)
    shared_dir.mkdir(parents=True, exist_ok=True)
    path = shared_dir / "vocab.json"
    path.write_text(json.dumps(vocab, ensure_ascii=False, indent=1, sort_keys=False), encoding="utf-8")
    info = {
        "vocab_path": str(path),
        "sha256": sha256_file(path),
        "size": len(vocab),
        "characters": [c for c in vocab if c not in (UNK, PAD)],
        "character_names": {c: (unicodedata.name(c, f"U+{ord(c):04X}") if len(c) == 1 else c) for c in vocab},
        "sources": sources or {},
    }
    save_json(info, results_dir / "vocab_info.json")
    return info


def unknown_chars(texts: Iterable[str], vocab: dict[str, int]) -> dict[str, int]:
    """Characters (other than space) missing from the vocabulary, with counts."""
    out: dict[str, int] = {}
    for t in texts:
        for ch in nfc(t):
            if ch != " " and ch not in vocab:
                out[ch] = out.get(ch, 0) + 1
    return out


def make_feature_extractor():
    from transformers import Wav2Vec2FeatureExtractor

    return Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
                                    do_normalize=True, return_attention_mask=True)


def make_processor(vocab_path: str | Path):
    from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2Processor

    tokenizer = Wav2Vec2CTCTokenizer(str(vocab_path), unk_token=UNK, pad_token=PAD, word_delimiter_token=WORD_DELIM)
    return Wav2Vec2Processor(feature_extractor=make_feature_extractor(), tokenizer=tokenizer)


def load_shared_processor(shared_dir: str | Path, results_dir: str | Path, check_hash: bool = True):
    """Load the processor from the shared vocabulary, failing if the hash differs."""
    path = Path(shared_dir) / "vocab.json"
    if not path.exists():
        raise FileNotFoundError(f"Shared vocabulary not found at {path}. Run the 'prepare' stage first.")
    if check_hash:
        info = load_json(Path(results_dir) / "vocab_info.json")
        if info is None:
            raise VocabMismatchError("results/vocab_info.json missing: cannot verify the shared vocabulary hash.")
        actual = sha256_file(path)
        if actual != info["sha256"]:
            raise VocabMismatchError(f"Shared vocabulary hash mismatch: {actual} != {info['sha256']}. "
                                     "All variants must use the same vocab.json.")
    return make_processor(path)


def vocab_sha256(shared_dir: str | Path) -> str:
    return sha256_file(Path(shared_dir) / "vocab.json")


In [ ]:
# ---- Install the package (editable) and make it importable in this kernel --------------------
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", CODE_DIR], check=True)
sys.path.insert(0, f"{CODE_DIR}/src")
import importlib
for _m in [m for m in list(sys.modules) if m.startswith("yoruba_asr")]:
    del sys.modules[_m]
import yoruba_asr
from yoruba_asr import pipeline
from yoruba_asr.config import CONFIGS_DIR
print("yoruba_asr", yoruba_asr.__version__, "| configs:", CONFIGS_DIR)


## Locate data

In [ ]:
# ---- Locate data ----------------------------------------------------------------------------------
import json
from yoruba_asr.config import discover_paths

BASE_OVERRIDES = {"paths.input_root": INPUT_ROOT, "paths.work_root": WORK_ROOT, "paths.tmp_dir": TMP_DIR,
                  "data.primary": PRIMARY_DATASET, "data.slr86.download": bool(DOWNLOAD_OPENSLR86) and ON_KAGGLE}
_cfg = pipeline.stage_config("prepare", BASE_OVERRIDES)
FOUND = discover_paths(_cfg)   # downloads OpenSLR 86 here if it is not attached and DOWNLOAD_OPENSLR86=True
print(json.dumps(FOUND.summary(), indent=2, ensure_ascii=False))
_needs_data = any(s in STAGE for s in ("smoke", "prepare", "augment", "all_light", "ablation", "cpt", "V3", "V4"))
if FOUND.common_voice is None and not FOUND.slr86 and _needs_data and not DRY_RUN:
    raise FileNotFoundError(
        "No speech data found. Either attach Common Voice 27.0 Yoruba (private dataset, the .tar.gz), or turn "
        "Internet ON and keep DOWNLOAD_OPENSLR86 = True so OpenSLR 86 is downloaded automatically.")
if FOUND.slr86:
    print("OpenSLR 86 (Google Yoruba corpus):", [str(p) for p in FOUND.slr86])
print("Main dataset setting PRIMARY_DATASET =", PRIMARY_DATASET)
if FOUND.common_voice is not None:
    print("\nCommon Voice:", FOUND.common_voice)
    from yoruba_asr.data.common_voice import index_audio_files
    _n_clips = len(index_audio_files([INPUT_ROOT, f"{TMP_DIR}/extracted"], exts=(".mp3",)))
    print(f"Common Voice MP3 clips found in the attached data: {_n_clips} (the full 27.0 Yoruba release has 4,970)")
    if _n_clips < 1000 and STAGE not in ("smoke",) and not DRY_RUN:
        print("*** WARNING: most audio clips are missing. Re-upload the original .tar.gz archive "
              "(see the error message below and docs/KAGGLE_GUIDE.md). ***")
print("SLR86 folders:", len(FOUND.slr86), "| ÌròyìnSpeech:", FOUND.iroyin or FOUND.iroyin_manifest,
      "| MUSAN:", FOUND.musan, "| previous outputs:", len(FOUND.previous_outputs))


## Previous outputs (chaining sessions)

In [ ]:
# ---- Previous outputs: chain sessions (prepare -> V1 -> ...) ----------------------------------------
from yoruba_asr.config import copy_previous_outputs

if FOUND.previous_outputs and not DRY_RUN:
    print("Copying outputs of previous notebook versions into", WORK_ROOT, "(skips files already present) ...")
    for line in copy_previous_outputs(FOUND, __import__("pathlib").Path(WORK_ROOT)):
        print("  copied", line)
else:
    print("No previous output attached." if not FOUND.previous_outputs else "dry run: nothing copied")
print("Work dir contents:", sorted(os.listdir(WORK_ROOT)))


## Run the stage

In [ ]:
# ---- Run the stage --------------------------------------------------------------------------------
import time
SESSION_PLAN = """
| Run  | STAGE                  | Attach as input                               | GPU h (T4, estimate) |
|------|------------------------|-----------------------------------------------|----------------------|
| 0    | smoke                  | Common Voice                                  | < 0.3                |
| 1    | prepare                | Common Voice (+ SLR86, ÌròyìnSpeech, MUSAN)   | ~0.5                 |
| 2    | V1                     | data + output of run 1                        | 4-5                  |
| 3    | augment+tune_mask      | data + previous output                        | 3-4                  |
| 4    | V2                     | data + previous output                        | 4-5                  |
| 5-6  | cpt (twice, resumes)   | data + previous output                        | ~20                  |
| 7    | V3                     | data + previous output                        | 5-6                  |
| 8    | V4                     | data + previous output                        | 4-5                  |
| 9    | evaluate               | previous output                               | ~1                   |
| 10+  | ablation (one per run) | data + previous output                        | ~1.5 each            |
| last | analyse                | previous output                               | < 0.5                |
(estimates; the actual GPU time of every stage is logged in results/**/run_info.json)
"""
print(SESSION_PLAN)

overrides = {**BASE_OVERRIDES, **EXTRA_OVERRIDES}
if "cpt" in STAGE:
    overrides["cpt.time_budget_hours"] = float(CPT_MAX_HOURS)
if STAGE == "smoke" and ON_KAGGLE:
    overrides["tts.model"] = "facebook/mms-tts-yor"   # exercise the real TTS + coverage check

_t0 = time.time()
RESULT = pipeline.run_stage(STAGE, overrides, smoke=(STAGE == "smoke"), resume=RESUME, only=ABLATION_ONLY,
                            dry_run=DRY_RUN)
print(f"\nStage '{STAGE}' finished in {(time.time() - _t0) / 3600:.2f} h")


## Summary

In [ ]:
# ---- Summary: key metrics, files written, figures, optional Hub upload ------------------------------
from pathlib import Path
import pandas as pd

_work = Path(WORK_ROOT) / ("smoke" if STAGE == "smoke" else ("light" if STAGE == "all_light" else ""))
_res = _work / "results"
if (_res / "summary.csv").exists():
    print("== Test-set results so far (%) ==")
    print(pd.read_csv(_res / "summary.csv").to_string(index=False))
for v in ("V1", "V2", "V3", "V4"):
    st = _res / v / "run_status.json"
    if st.exists() and v in STAGE:
        print(f"\n{v} status:", json.loads(st.read_text(encoding="utf-8")))
_gpu = 0.0
for ri in _res.rglob("*run_info.json"):
    try:
        _gpu += json.loads(ri.read_text(encoding="utf-8")).get("total_gpu_hours", 0.0)
    except Exception:
        pass
print(f"\nLogged GPU time across all stages so far: {_gpu:.2f} h")

_recent = sorted((p for p in _res.rglob("*") if p.is_file() and p.stat().st_mtime >= _t0 - 5),
                 key=lambda p: p.stat().st_mtime)
print(f"\nFiles written in this run ({len(_recent)}):")
for p in _recent[-60:]:
    print("  ", p.relative_to(_work))

try:
    from IPython.display import Image, display
    for png in sorted((_res / "figures").glob("*.png")):
        print(png.name); display(Image(filename=str(png), width=600))
except Exception as exc:
    print("figures not shown:", exc)

if PUSH_TO_HUB and STAGE in ("V1", "V2", "V3", "V4") and not DRY_RUN:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    _token = UserSecretsClient().get_secret("HF_TOKEN")
    _api = HfApi(token=_token)
    _repo = f"{HUB_REPO_PREFIX}-{STAGE.lower()}"
    _api.create_repo(_repo, private=True, exist_ok=True)
    _api.upload_folder(folder_path=str(_work / "models" / STAGE), repo_id=_repo)
    print("pushed", STAGE, "to https://huggingface.co/" + _repo, "(private)")
    del _token


In [ ]:
# ---- Next step ------------------------------------------------------------------------------------
from IPython.display import Markdown, display

_msg = pipeline.next_step_message(STAGE.split("+")[-1] if "+" in STAGE and STAGE != "augment+tune_mask" else STAGE)
_incomplete = isinstance(RESULT, dict) and (RESULT.get("status") == "incomplete" or RESULT.get("completed") is False)
if _incomplete:
    _msg = (f"The stage **{STAGE}** stopped at the session time budget and saved a checkpoint. "
            f"Run **the same STAGE again** with RESUME=True and attach this version's output.")
display(Markdown(f"### Next step\n\n{_msg}\n\nIn the next version: *Add Input → Your Work → this notebook "
                 "→ the version you just ran* (keep the data datasets attached), then edit `STAGE` and "
                 "*Save & Run All*."))
print(_msg)
